# Orthodontic AI Framework — v23

Final version. Trains and evaluates all phases of the dental AI pipeline.  
All training cells skip automatically if checkpoints already exist.

| Phase | Task | Architecture | Result |
|-------|------|-------------|--------|
| A | Tooth segmentation | DINOv2 ViT-B/14 + FPN | IoU = 0.9105 |
| B | FDI instance seg (32-class) | YOLOv8x-seg | mAP50 = 0.956 |
| C | Multi-task seg + disease | DentalMTLNet + GradNorm | — |
| D | Disease classification | EfficientNetV2-M + CBAM | F1 = 0.8927 |
| F | Malocclusion screening | ConvNeXtV2-T | F1 = 0.5826 |
| G1 | Cephalometric landmarks | ResNet-50 + U-Net decoder | MRE = 3.59 mm |
| G2 | CVM staging | ConvNeXt-V2-T | Acc = 58.8% |

Datasets: DENTEX, HITL, AKUDENTAL, Dataset1 (panoramic), Aariz (cephalometric).  
Runtime: ~15-20h total on GPU. See thesis Section 3.2 for dataset details.


## Imports & Configuration


In [ ]:

import sys, os, json, warnings, time, math, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import cv2
warnings.filterwarnings("ignore")

import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
import random; random.seed(SEED)

BASE  = Path(r"C:\Users\Abdo\Desktop\Work\code\Teeth\DATA")
OUT   = BASE / "processed"
MDL   = OUT  / "models_v4"

EXT_DIR        = BASE / "external_datasets"
EXT_CHILDRENS  = EXT_DIR / "childrens_dental" / "Dental_dataset"
EXT_HITL       = EXT_DIR / "humans_in_loop_seg" / "Teeth Segmentation JSON" / "d2"
SSL_CKPT_V2    = OUT / "backbone_ssl_v2.pth"
OTHER          = BASE.parent / "Other users"

# Data audit flagged samples (audit cell removed for submission — trained models already exclude these)
_flagged_stems = set()

RAW_XRAYS  = BASE / "Radiographs"
RAW_MASKS  = BASE / "Segmentation" / "teeth_mask"
DIS_JSON   = BASE / "training_data" / "quadrant-enumeration-disease" / "train_quadrant_enumeration_disease.json"
DIS_XRAYS  = BASE / "training_data" / "quadrant-enumeration-disease" / "xrays"
ENUM_JSON  = BASE / "training_data" / "quadrant_enumeration" / "train_quadrant_enumeration.json"
ENUM_XRAYS = BASE / "training_data" / "quadrant_enumeration" / "xrays"

# SSL backbone path (use v2 if available)
BACKBONE_SSL = SSL_CKPT_V2 if SSL_CKPT_V2.exists() else OUT / "backbone_ssl.pth"

SEG_IMG_DIR  = OUT / "seg_images_v4"
SEG_MASK_DIR = OUT / "seg_masks_polygon"
PATCH_DIR    = OUT / "tooth_patches_v9"
VIS_DIR      = OUT / "visualizations_v4"
PATCH_CSV    = OUT / "patch_labels_v9.csv"
# External data sources
PERI_EXT_ZIP  = BASE / "external_datasets" / "periapical_disease" / "dental-disease-panoramic-detection-dataset.zip"
PERI_EXT_DIR  = BASE / "external_datasets" / "periapical_disease" / "extracted"
PERI_OTH_DIR  = BASE.parent / "Other users" / "Panoramic radiographs with periapical lesions Dataset"
DS3_DIR       = BASE.parent / "Other users" / "dataset3"
MAL_CSV      = OUT / "mal_labels_v4.csv"
AARIZ_DIR    = BASE / "Aariz"
AARIZ_MAPS   = AARIZ_DIR / "cephalogram_machine_mappings.csv"

SEG_CKPT  = MDL / "seg_v23_best.pth"
# Copy best Phase A checkpoint if available
_SEG_V22 = MDL / "seg_v22_best.pth"
if not SEG_CKPT.exists() and _SEG_V22.exists():
    import shutil as _sh_seg; _sh_seg.copy2(str(_SEG_V22), str(SEG_CKPT))
    print("v23: copied seg_v22_best.pth -> seg_v23_best.pth (Phase A skip)")
V14_SEG_CKPT = MDL / "seg_v14_best.pth"  # v15: warm-start source
MTL_CKPT  = MDL / "mtl_v23_best.pth"
CLS_CKPT  = MDL / "cls_v23_best.pth"
# Copy best Phase D checkpoint (F1=0.8927)
_CLS_V14 = MDL / "cls_v14_best.pth"
if not CLS_CKPT.exists() and _CLS_V14.exists():
    import shutil as _sh_cls; _sh_cls.copy2(str(_CLS_V14), str(CLS_CKPT))
    print(f"v23: copied cls_v14_best.pth -> cls_v23_best.pth (F1=0.8927, skip retrain)")
MAL_CKPT  = MDL / "mal_v23_best.pth"
CEPH_CKPT = MDL / "ceph_landmark_v23_best.pth"
CVM_CKPT  = MDL / "cvm_v23_best.pth"
OMNI_CKPT = MDL / "omni_pretrain_v12.pth"


# Instance Segmentation Config
N_FDI_CLASSES = 33  # 32 FDI teeth + background
SEG_INST_MASK_DIR = OUT / "seg_instance_masks_v19"
ODONTOAI_DIR      = BASE / "OdontoAI"
DNS_PANO_DIR      = BASE / "DNS_Panoramic"
# Instance seg datasets
AKUDENTAL_JSON    = BASE.parent / "Other users/instance seg/AKUDENTAL-main/AKUDENTAL/akudental_instances.json"
AKUDENTAL_IMG_DIR = AKUDENTAL_JSON.parent / "images"
D1_ANN_DIR  = BASE.parent / "Other users/instance seg/Datset 1/Teeth Segmentation JSON/d2/ann"
D1_IMG_DIR  = BASE.parent / "Other users/instance seg/Datset 1/Teeth Segmentation JSON/d2/img"

EXPERT_MASK_DIR   = BASE / "Expert" / "mask"
STUDENT_MASK_DIR  = BASE / "Student" / "mask"
SEG_INST_IMG_DIR  = OUT / "seg_instance_images_v19"
SEG_INST_CKPT     = MDL / "seg_instance_v23_best.pt"
SEG_INST_EPOCHS   = 50   # v21: Flash SDP added; 50ep×3,957imgs=197K exposures (>v19 185K; faster convergence with 3.2x data)
YOLO_DIR          = OUT / 'yolo_phase_b'  # YOLOv8 data + run dir
SEG_INST_BATCH    = 32  # v20: ViT-B/14 on RTX 5090 (32GB) — batch=32 fits, better GPU util
INST_ENCODER_LR   = 5e-6   # Lower LR for instance seg fine-tuning
INST_DECODER_LR   = 5e-5
PHASE_B_BINARY_GUIDE = True   # v19: use Phase A binary mask to guide instance seg
FINE_TUNE_PHASE_B    = True   # STEP 4: fine-tune at LR/10 after main training


for d in [OUT, MDL, SEG_IMG_DIR, SEG_INST_MASK_DIR, SEG_INST_IMG_DIR, SEG_MASK_DIR, PATCH_DIR, VIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0        # Windows Jupyter: 0 only — multiprocessing deadlocks reliably
SEG_SIZE    = 512; PATCH_SIZE = 300; CLS_SIZE = 380; MAL_SIZE = 384; CEPH_SIZE = 512
INST_SIZE   = 448   # Phase B: 448/14=32 patches → 1024 tokens vs 37²=1369 (~25% fewer, ~44% less attention)
SEG_BATCH   = 16;  CLS_BATCH  = 64  # v15: restored to v12 batch size (v14 batch=8 was biggest regression factor)
SEG_EPOCHS  = 100;  CLS_EPOCHS = 120; MTL_EPOCHS = 120;  MAL_EPOCHS = 100; CEPH_EPOCHS = 130  # v21: was 80; v17: DINOv2-L + Phase G instance seg + improved phases
SSL_EPOCHS  = 50
BBOX_PAD    = 20;  N_LM = 29

DISEASE_MAP = {0:"Impacted", 1:"Caries+DeepCaries", 2:"Periapical"}
CLASS_NAMES = [DISEASE_MAP[i] for i in range(3)]
N_CLS       = 3
INV_NORM    = transforms.Normalize([-0.485/0.229,-0.456/0.224,-0.406/0.225],
                                    [1/0.229,1/0.224,1/0.225])

print(f"Device: {DEVICE}")
print(f"SSL backbone: {BACKBONE_SSL}  exists={BACKBONE_SSL.exists()}")

# GPU throughput optimisations
import torch.backends.cudnn as _cudnn
_cudnn.benchmark = True          # auto-tune conv kernels for fixed input sizes
torch.set_float32_matmul_precision('high')  # use TF32 on Ampere/Ada (free ~20% speed)
# Flash Attention / Memory-Efficient Attention for DINOv2 ViT blocks (~25% speedup)
try:
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)   # disable slow fallback
    print('Flash Attention (SDPA) enabled')
except Exception:
    pass  # PyTorch < 2.0 fallback

BACKBONE_MODE = "dinov2_vitb14"  # v19: switched to ViT-B/14 for practical speed (v17 ViT-L too slow; using seg_v16 as Phase A checkpoint)
INST_BACKBONE_MODE = BACKBONE_MODE  # same backbone for both phases

# seg pipeline source config
# Blob sources use synthetic ellipse masks (bbox->ellipse) -- NOT real tooth boundaries
SEG_BLOB_PREFIXES = ("d2_",)  # Dataset2/YOLO 13,932 ellipse blob masks -- real prefix is d2_
SEG_PSEUDO_CONF = 0.90   # v14: raised from 0.80 — literature recommends >=0.90 for binary seg (FixMatch)
SEG_PSEUDO_THRESH = 0.50 # sigmoid threshold for pseudo binary mask
SEG_PSEUDO_CAP_RATIO = 2.0  # v16: max pseudo:real ratio (v15 had 7:1, caused regression)
ENCODER_LR   = 1e-5    # v14: full retrain LR (same as v12)
DECODER_LR   = 1e-4    # v14: full retrain LR (same as v12)
EARLY_STOP_PAT = 40   # v16: patience 40 — v15 stopped too early at ep 48
# Quality sources use real polygon-annotated masks
SEG_SOURCES = {"ext_cd": 0, "ext_hitl": 1, "sts24": 2, "default": 3}  # FiLM conditioning IDs
N_SEG_SOURCES = 4
# convnext_b uses torchvision ConvNeXt-B (ImageNet pretrained) — no SSL backbone needed


PATCH_SIZE_V8   = 380    # Larger tooth patches (was 300) - better periapical context
PERIAPICAL_BOOST = 8.0   # Periapical oversampling weight (was 5.0)
FOCAL_GAMMA_PERI = 4.0   # Focal loss gamma for Periapical class (was 3.5)
CVM_EPOCHS      = 60     # CVM staging epochs (new variable)


# CLAHE preprocessing
import cv2 as _cv2
def apply_clahe(img_np):
    """Apply CLAHE to panoramic X-ray image (numpy HWC uint8 RGB or HW grayscale).
    Proven to improve contrast-limited X-ray features. Applied as fixed preprocessing
    (not augmentation) in SegDataset and PatchDataset. Source: thisdatasetmustbedead.ipynb.
    """
    if img_np.ndim == 3 and img_np.shape[2] == 3:
        lab = _cv2.cvtColor(img_np, _cv2.COLOR_RGB2LAB)
        l, a, b = _cv2.split(lab)
        clahe = _cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = _cv2.merge([l, a, b])
        return _cv2.cvtColor(lab, _cv2.COLOR_LAB2RGB)
    else:
        clahe = _cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        return clahe.apply(img_np if img_np.ndim == 2 else img_np[:, :, 0])

## SSL Pretraining

SimCLR on unlabeled panoramic X-rays. Produces backbone_ssl_v2.pth.

In [ ]:
# SSL Pretraining (SimCLR)
# Trains a ResNet50 backbone on all unlabeled panoramic X-rays using SimCLR.
# Saves backbone weights (no projection head) to SSL_CKPT_V2.
# Skipped if SSL_CKPT_V2 already exists.

if SSL_CKPT_V2.exists():
    print(f"SSL_CKPT_V2 already exists → {SSL_CKPT_V2}, skipping SimCLR training")
else:
    import torchvision.models as tvm
    import torchvision.transforms as T

    # Collect all unlabeled panoramic X-rays
    ssl_image_dirs = [
        RAW_XRAYS,                                              # 1,000 images
        BASE / "7812323" / "training_data" / "unlabelled" / "xrays",  # 1,571
    ]
    ssl_paths = []
    for d in ssl_image_dirs:
        if d.exists():
            ssl_paths += list(d.glob("*.jpg")) + list(d.glob("*.png"))
    print(f"SSL dataset: {len(ssl_paths)} images")

    # SimCLR augmentation pipeline
    _ssl_aug = T.Compose([
        T.RandomResizedCrop(224, scale=(0.2, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        T.RandomGrayscale(p=0.2),
        T.RandomApply([T.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))], p=0.5),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    class SSLDataset(Dataset):
        def __init__(self, paths):
            self.paths = paths
        def __len__(self):
            return len(self.paths)
        def __getitem__(self, idx):
            img = Image.open(self.paths[idx]).convert("RGB")
            return _ssl_aug(img), _ssl_aug(img)

    # SimCLR model
    class SimCLREncoder(nn.Module):
        def __init__(self):
            super().__init__()
            backbone = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
            self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # drop fc
            self.projector = nn.Sequential(
                nn.Linear(2048, 512),
                nn.ReLU(inplace=True),
                nn.Linear(512, 128),
            )
        def forward(self, x):
            h = self.encoder(x).flatten(1)
            return self.projector(h)

    def nt_xent_loss(z1, z2, temperature=0.07):
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)
        N = z1.size(0)
        z = torch.cat([z1, z2], dim=0)                    # 2N × 128
        sim = torch.mm(z, z.T) / temperature              # 2N × 2N
        # mask out self-similarity
        mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask, float('-inf'))
        labels = torch.cat([torch.arange(N, 2*N), torch.arange(N)]).to(z.device)
        return F.cross_entropy(sim, labels)

    # Training
    ssl_ds     = SSLDataset(ssl_paths)
    ssl_loader = DataLoader(ssl_ds, batch_size=64, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=True)
    ssl_model  = SimCLREncoder().to(DEVICE)
    ssl_opt    = torch.optim.AdamW(ssl_model.parameters(), lr=1e-3, weight_decay=1e-4)
    ssl_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(ssl_opt, T_max=SSL_EPOCHS)

    for epoch in range(1, SSL_EPOCHS + 1):
        ssl_model.train()
        total_loss = 0.0
        for v1, v2 in tqdm(ssl_loader, desc=f"SSL epoch {epoch}/{SSL_EPOCHS}",leave=True):
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            loss = nt_xent_loss(ssl_model(v1), ssl_model(v2))
            ssl_opt.zero_grad()
            loss.backward()
            ssl_opt.step()
            total_loss += loss.item()
        ssl_sched.step()
        print(f"SSL epoch {epoch:3d}/{SSL_EPOCHS}  loss={total_loss/len(ssl_loader):.4f}")

    # Save backbone only (drop projection head)
    torch.save(ssl_model.encoder.state_dict(), SSL_CKPT_V2)
    BACKBONE_SSL = SSL_CKPT_V2  # update global so downstream phases use new backbone
    print(f"Saved SSL backbone → {SSL_CKPT_V2}")

print(f"BACKBONE_SSL = {BACKBONE_SSL}")

## Model Architecture Definitions

All network architectures used across phases are defined in this cell.


In [ ]:

class _DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x): return self.net(x)

class _Up(nn.Module):
    def __init__(self, skip_ch, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch//2, 2, stride=2)
        self.conv = _DoubleConv(skip_ch+in_ch//2, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256, rates=(6,12,18)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch,out_ch,1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        ] + [
            nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),
                          nn.BatchNorm2d(out_ch), nn.ReLU(True)) for r in rates
        ])
        self.gap = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                  nn.Conv2d(in_ch,out_ch,1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        n = len(rates)+2
        self.proj = nn.Sequential(nn.Conv2d(n*out_ch,out_ch,1,bias=False), nn.BatchNorm2d(out_ch),
                                   nn.ReLU(True), nn.Dropout(0.5))
    def forward(self, x):
        h,w = x.shape[2:]
        outs = [b(x) for b in self.branches]
        outs.append(F.interpolate(self.gap(x),(h,w),mode="bilinear",align_corners=False))
        return self.proj(torch.cat(outs,1))

class SegUNetV6(nn.Module):
    def __init__(self, ssl_path=None, convnext_encoder=None):
        super().__init__()
        self._is_convnext = convnext_encoder is not None
        if self._is_convnext:
            self.encoder = convnext_encoder
            # ConvNeXt-B channels: e0=128, e1=256, e2=512, e3=1024 (no e4)
            self.aspp = ASPP(1024, 256)
            # d4: upsamples from ASPP(256) and skips e2(512) -> out 256
            self.d4 = _Up(512, 256, 256)
            # d3: upsamples from d4(256) and skips e1(256) -> out 128
            self.d3 = _Up(256, 256, 128)
            # d2: upsamples from d3(128) and skips e0(128) -> out 64
            self.d2 = _Up(128, 128, 64)
            # d1: no skip (ConvNeXt has no separate initial conv like ResNet conv1)
            self.d1 = nn.Sequential(
                nn.ConvTranspose2d(64, 32, 2, stride=2),
                _DoubleConv(32, 32))
            self.up_f = nn.ConvTranspose2d(32, 16, 2, stride=2)
            self.out = nn.Conv2d(16, 1, 1)
            self.aux3 = nn.Conv2d(128, 1, 1)
            self.aux2 = nn.Conv2d(64, 1, 1)
            print("SegUNetV6: using ConvNeXt-B encoder")
        else:
            ssl_path = Path(ssl_path) if ssl_path else BACKBONE_SSL
            if not ssl_path.exists():
                raise RuntimeError(f"SSL backbone not found: {ssl_path}")
            bb = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            sd = torch.load(ssl_path, map_location="cpu", weights_only=True)
            if next(iter(sd),"").split(".")[0].isdigit():
                _m={"0":"conv1","1":"bn1","4":"layer1","5":"layer2","6":"layer3","7":"layer4"}
                sd={".".join([_m.get(k.split(".")[0],k.split(".")[0])]+k.split(".")[1:]):v for k,v in sd.items() if k.split(".")[0] in _m}
            miss,_ = bb.load_state_dict(sd, strict=False)
            print(f"  SSL loaded: {len(bb.state_dict())-len(miss)}/{len(bb.state_dict())} model params")
            self.e0 = nn.Sequential(bb.conv1,bb.bn1,bb.relu); self.mp = bb.maxpool
            self.e1=bb.layer1; self.e2=bb.layer2; self.e3=bb.layer3; self.e4=bb.layer4
            self.aspp=ASPP(2048,256)
            self.d4=_Up(1024,256,512); self.d3=_Up(512,512,256)
            self.d2=_Up(256,256,128); self.d1=_Up(64,128,64)
            self.up_f=nn.ConvTranspose2d(64,32,2,stride=2); self.out=nn.Conv2d(32,1,1)
            self.aux3=nn.Conv2d(256,1,1); self.aux2=nn.Conv2d(128,1,1)

    def forward(self, x, return_aux=True):
        if self._is_convnext:
            e0,e1,e2,e3 = self.encoder(x)
            btn = self.aspp(e3)
            d4 = self.d4(btn, e2); d3 = self.d3(d4, e1); d2 = self.d2(d3, e0)
            d1 = self.d1(d2)
            main = self.out(self.up_f(d1))
            if return_aux and self.training:
                a3 = F.interpolate(self.aux3(d3), x.shape[2:], mode="bilinear", align_corners=False)
                a2 = F.interpolate(self.aux2(d2), x.shape[2:], mode="bilinear", align_corners=False)
                return main, a3, a2
            return main
        else:
            s0=self.e0(x); s1=self.e1(self.mp(s0)); s2=self.e2(s1); s3=self.e3(s2); s4=self.e4(s3)
            btn=self.aspp(s4)
            d4=self.d4(btn,s3); d3=self.d3(d4,s2); d2=self.d2(d3,s1); d1=self.d1(d2,s0)
            main=self.out(self.up_f(d1))
            if return_aux and self.training:
                a3=F.interpolate(self.aux3(d3),x.shape[2:],mode="bilinear",align_corners=False)
                a2=F.interpolate(self.aux2(d2),x.shape[2:],mode="bilinear",align_corners=False)
                return main,a3,a2
            return main


# -- ConvNeXt-B Encoder wrapper for U-Net skip connections --
class ConvNeXtEncoder(nn.Module):
    """Wraps torchvision ConvNeXt-B to expose 4 skip-connection feature maps."""
    def __init__(self, cnx):
        super().__init__()
        f = cnx.features  # Sequential of 8 stages
        # Stem + stage 0 -> 128ch output
        self.e0 = nn.Sequential(f[0], f[1])   # stem (128ch)
        self.e1 = nn.Sequential(f[2], f[3])   # stage 1->2 (256ch)
        self.e2 = nn.Sequential(f[4], f[5])   # stage 3->4 (512ch)
        self.e3 = nn.Sequential(f[6], f[7])   # stage 5->6 (1024ch)
        self.out_channels = [128, 256, 512, 1024]

    def forward(self, x):
        e0 = self.e0(x)   # /4
        e1 = self.e1(e0)  # /8
        e2 = self.e2(e1)  # /16
        e3 = self.e3(e2)  # /32
        return e0, e1, e2, e3


def build_encoder(backbone_mode, backbone_ssl_path, device):
    """Build encoder based on BACKBONE_MODE. Returns (encoder_module, channel_list)."""
    if backbone_mode == "convnext_b":
        import torchvision.models as tv
        cnx = tv.convnext_b(weights=tv.ConvNeXt_B_Weights.IMAGENET1K_V1)
        enc = ConvNeXtEncoder(cnx)
        print("Encoder: ConvNeXt-B (ImageNet pretrained), channels:", enc.out_channels)
        return enc, enc.out_channels
    else:  # ssl_resnet50 (default)
        # Return None -- SegUNetV6 handles ResNet50+SSL loading internally
        return None, [64, 256, 512, 1024, 2048]


# Lovász-Hinge loss (directly optimizes IoU)
# Based on: Berman et al. 2018 "The Lovász-Softmax loss" (CVPR 2018)
# Provides a tractable surrogate for optimizing IoU directly, unlike Dice/BCE
# which are proxies. Literature reports +2-3% IoU improvement.
def _lovasz_grad(gt_sorted):
    """Gradient of the Lovász extension of IoU w.r.t sorted errors."""
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:] = jaccard[1:] - jaccard[:-1]
    return jaccard

def lovasz_hinge_flat(logits, labels):
    """Binary Lovász hinge loss, flat version (single image)."""
    if len(labels) == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    perm_labels = labels[perm]
    grad = _lovasz_grad(perm_labels)
    return torch.dot(F.relu(errors_sorted), grad)

def lovasz_hinge(logits, labels):
    """Binary Lovász hinge loss, batch-flattened (fast: 1 sort instead of B sorts).
    logits: (B, 1, H, W) raw model output (no sigmoid)
    labels: (B, 1, H, W) binary ground truth
    """
    return lovasz_hinge_flat(logits.view(-1), labels.view(-1))

# OHEM: Online Hard Example Mining
# Focuses training on the hardest pixels per batch. Literature: +1-2% IoU.
# Keeps top-k% highest-loss pixels, zeros out easy pixels.
def ohem_loss(pred, tgt, loss_fn, keep_ratio=0.7):
    """Apply OHEM: compute per-pixel loss, keep only top keep_ratio fraction."""
    # Per-pixel BCE loss (unreduced)
    px_loss = F.binary_cross_entropy_with_logits(pred, tgt, reduction='none')
    # Flatten and sort
    B = px_loss.shape[0]
    px_flat = px_loss.view(B, -1)
    n_pixels = px_flat.shape[1]
    n_keep = max(1, int(n_pixels * keep_ratio))
    # Keep top-k hardest pixels per image
    topk_loss, _ = torch.topk(px_flat, n_keep, dim=1)
    return topk_loss.mean()

def dice_loss(pred, tgt, s=1.0):
    p=torch.sigmoid(pred); i=(p*tgt).sum((2,3))
    return (1-(2*i+s)/(p.sum((2,3))+tgt.sum((2,3))+s)).mean()

def boundary_loss(pred, tgt, weight=0.15):
    """
    Boundary-aware loss: penalizes errors at mask edges 5x more than interior pixels.
    Uses Sobel edge detection on GT mask to locate tooth boundaries and gaps between teeth.
    Directly targets the visual failure mode seen in v11 (teeth merged into blob).
    Based on: Kervadec et al. 2019 + Generalized Surface Loss (arXiv:2302.03868).
    Applied as 0.15x component of total seg loss.
    """
    # Sobel edge detection on GT mask
    sob_x = F.conv2d(tgt,
                     torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],
                                  dtype=tgt.dtype, device=tgt.device).view(1,1,3,3),
                     padding=1)
    sob_y = F.conv2d(tgt,
                     torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],
                                  dtype=tgt.dtype, device=tgt.device).view(1,1,3,3),
                     padding=1)
    edge_map = (sob_x.abs() + sob_y.abs()).clamp(0, 1)
    px_weights = 1.0 + 4.0 * edge_map   # 5x weight at tooth boundaries
    bce = F.binary_cross_entropy_with_logits(pred, tgt, weight=px_weights, reduction='mean')
    return weight * bce

def seg_loss_fn(pred, tgt, a3=None, a2=None):
    """v12: 0.70*Dice + 0.15*BCE + 0.15*BoundaryLoss.
    BoundaryLoss (5x pixel weight at tooth edges) directly targets v11 failure:
    merged teeth blobs with no gaps. Based on Kervadec 2019 + arXiv:2302.03868.
    """
    def _l(p, t):
        pos_count = t.sum((2, 3), keepdim=True).clamp(min=1)
        neg_count = (1 - t).sum((2, 3), keepdim=True).clamp(min=1)
        pos_w = torch.clamp(neg_count / pos_count, min=1.0, max=50.0).mean()
        bce  = F.binary_cross_entropy_with_logits(p, t, pos_weight=pos_w)
        dc   = dice_loss(p, t)
        bl   = boundary_loss(p, t, weight=1.0)  # weight factor already inside
        lv = lovasz_hinge(p, t)  # v16: direct IoU optimization (batch-flat, fast)
        return 0.40 * lv + 0.25 * dc + 0.10 * bce + 0.25 * bl  # v16: Lovasz+Dice+BCE+Boundary (OHEM dropped — Lovasz already mines hard examples)
    loss = _l(pred, tgt)
    if a3 is not None: loss = loss + 0.5 * _l(a3, tgt)
    if a2 is not None: loss = loss + 0.3 * _l(a2, tgt)
    return loss

def iou_metric(pred, tgt):
    p=(torch.sigmoid(pred)>0.5).float()
    i=(p*tgt).sum((2,3)); u=(p+tgt-p*tgt).sum((2,3))
    return (i/(u+1e-6)).mean().item()

def seg_metrics(pred, tgt, collect_for_auc=None):
    """Batch-level segmentation metrics: IoU, Dice/F1, Precision, Recall, PixelAcc.
    Note: Dice == F1 for binary segmentation (same formula: 2TP/(2TP+FP+FN)).
    collect_for_auc: if a list is passed, appends (probs, labels) for AUC accumulation.
    AUC is NOT computed per-batch -- call compute_seg_auc() after the val loop.
    """
    prob = torch.sigmoid(pred)
    p = (prob > 0.5).float()
    tp=(p*tgt).sum().item()
    fp=(p*(1-tgt)).sum().item()
    fn=((1-p)*tgt).sum().item()
    tn=((1-p)*(1-tgt)).sum().item()
    if collect_for_auc is not None:
        # Subsample 5000 pixels per batch for memory-safe AUC
        _pr = prob.detach().cpu().flatten()
        _lb = tgt.detach().cpu().flatten().long()
        idx = torch.randperm(len(_pr))[:5000]
        collect_for_auc.append((_pr[idx].numpy(), _lb[idx].numpy()))
    return dict(
        iou =tp/(tp+fp+fn+1e-6),
        dice=2*tp/(2*tp+fp+fn+1e-6),  # == F1
        prec=tp/(tp+fp+1e-6),
        rec =tp/(tp+fn+1e-6),
        pacc=(tp+tn)/(tp+tn+fp+fn+1e-6))

def compute_seg_auc(collected):
    """Compute pixel-level AUC-ROC from accumulated (probs, labels) pairs.
    Memory-safe: uses subsampled pixels (5000/batch) collected during val loop.
    Returns nan if no data or all-same class.
    """
    from sklearn.metrics import roc_auc_score
    import numpy as np
    if not collected:
        return float("nan")
    probs  = np.concatenate([c[0] for c in collected])
    labels = np.concatenate([c[1] for c in collected])
    if labels.sum() == 0 or labels.sum() == len(labels):
        return float("nan")  # All same class -- AUC undefined
    return float(roc_auc_score(labels, probs))

print("Phase A architecture defined ✓")


# Source utilities (blob detection, pseudo-label flags)
def get_source_id(img_path) -> int:
    """Extract dataset source ID from filename prefix (encoded at data-merge time)."""
    name = str(img_path).split("\\")[-1].split("/")[-1].lower()
    for prefix, sid in SEG_SOURCES.items():
        if prefix != "default" and name.startswith(prefix):
            return sid
    return SEG_SOURCES["default"]

def is_blob_source(img_path) -> bool:
    """Return True if this sample has a synthetic ellipse mask (not real tooth boundary).
    v12: d2_* (Dataset2/YOLO 13,932 samples) EXCLUDED -- bbox->ellipse masks
    teach the model to produce blobs instead of segmenting individual teeth."""
    name = str(img_path).split("\\")[-1].split("/")[-1].lower()
    return any(name.startswith(p) for p in SEG_BLOB_PREFIXES)


def is_pseudo_source(img_path) -> bool:
    """Return True if this sample is a pseudo-labeled sample (generated by model, not human-annotated)."""
    name = str(img_path).split("\\")[-1].split("/")[-1].lower()
    return name.startswith("ps_")

# DINOv2 Architecture

class DINOv2Encoder(nn.Module):
    """DINOv2 ViT encoder for segmentation. Supports ViT-B/14 and ViT-L/14.
    v17: upgraded to ViT-L/14 (307M params, 1024 embed dim, 24 blocks).
    ViT-L/14 has ~3.5x more params than ViT-B/14 — expected +1-2% IoU.
    RTX 5090 32GB handles ViT-L/14 with batch=16 at 512x512 (~18GB).
    """
    # Model configs: (hub_name, embed_dim, n_blocks, layer_indices, freeze_below)
    CONFIGS = {
        "dinov2_vitb14": ("dinov2_vitb14", 768, 12, [2, 5, 8, 11], 4),
        "dinov2_vitl14": ("dinov2_vitl14", 1024, 24, [5, 11, 17, 23], 8),
    }

    def __init__(self, model_name="dinov2_vitl14"):
        super().__init__()
        cfg = self.CONFIGS[model_name]
        hub_name, self.embed_dim, self.n_blocks, self.layer_indices, freeze_below = cfg
        self.dino = torch.hub.load(
            "facebookresearch/dinov2", hub_name,
            pretrained=True, force_reload=False
        )
        # Freeze early blocks for stability
        for name, p in self.dino.named_parameters():
            block_num = None
            if "blocks." in name:
                try: block_num = int(name.split("blocks.")[1].split(".")[0])
                except: pass
            if block_num is None or block_num < freeze_below:
                p.requires_grad_(False)
        self.out_channels = [self.embed_dim] * 4
        self.patch_size = 14
        n_trainable = sum(p.numel() for p in self.dino.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in self.dino.parameters())
        print(f"  DINOv2Encoder: {model_name} ({n_total/1e6:.0f}M total, {n_trainable/1e6:.0f}M trainable, freeze<{freeze_below})")

    def forward(self, x):
        B, C, H, W = x.shape
        H14 = ((H + 13) // 14) * 14
        W14 = ((W + 13) // 14) * 14
        if H14 != H or W14 != W:
            x = F.interpolate(x, (H14, W14), mode="bilinear", align_corners=False)
        feats = self.dino.get_intermediate_layers(x, n=self.layer_indices, reshape=True)
        return feats[0], feats[1], feats[2], feats[3]


class FiLM(nn.Module):
    """Feature-wise Linear Modulation (Dumoulin 2018; Lemay et al. MIDL 2021).
    Conditions decoder feature maps on dataset source identity.
    Literature shows +2-4% Dice for multi-source medical segmentation.
    Initialized as identity transform (zero gamma/beta) so epoch-1 IoU = v11 baseline.
    Overhead: ~0.01% of total model parameters per decoder stage.
    """
    def __init__(self, n_sources: int, n_channels: int):
        super().__init__()
        self.embed = nn.Embedding(n_sources, 32)
        self.mlp   = nn.Sequential(nn.Linear(32, 64), nn.ReLU(inplace=True),
                                   nn.Linear(64, n_channels * 2))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)   # identity init: no effect at epoch 1

    def forward(self, x: torch.Tensor, src_id: torch.Tensor) -> torch.Tensor:
        """x: (B, C, H, W);  src_id: (B,) long tensor of dataset source IDs"""
        emb    = self.embed(src_id)            # (B, 32)
        params = self.mlp(emb)                 # (B, 2C)
        gamma, beta = params.chunk(2, dim=1)   # each (B, C)
        gamma  = gamma.view(-1, x.size(1), 1, 1)
        beta   = beta.view(-1, x.size(1), 1, 1)
        return (1.0 + gamma) * x + beta        # residual: identity when gamma=0, beta=0


class SegUNetDINO(nn.Module):
    """DINOv2 ViT + FPN decoder + FiLM conditioning.
    v17: supports ViT-B/14 (768d) and ViT-L/14 (1024d).
    FiLM layers condition each decoder stage on dataset source ID.
    """
    def __init__(self, model_name=None):
        super().__init__()
        _mn = model_name or BACKBONE_MODE
        self.encoder = DINOv2Encoder(_mn)
        self.backbone_mode = _mn
        dim = self.encoder.embed_dim; pdim = 256

        # Project each DINOv2 level to pdim (unchanged from v11)
        self.proj3  = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.proj6  = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.proj9  = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.proj12 = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.fpn    = _DoubleConv(pdim, pdim)

        # Progressive upsampling (unchanged from v11)
        self.dec3 = CoordDoubleConv(pdim, 256)
        self.dec2 = CoordDoubleConv(256, 128)
        self.dec1 = CoordDoubleConv(128, 64)
        self.dec0 = CoordDoubleConv(64, 32)

        self.seg_out = nn.Conv2d(32, 1, 1)
        self.aux3    = nn.Conv2d(256, 1, 1)
        self.aux2    = nn.Conv2d(128, 1, 1)

        # FiLM layers -- one per decoder stage, identity-initialized
        self.film3 = FiLM(N_SEG_SOURCES, 256)
        self.film2 = FiLM(N_SEG_SOURCES, 128)
        self.film1 = FiLM(N_SEG_SOURCES, 64)
        self.film0 = FiLM(N_SEG_SOURCES, 32)

    def enc_params(self):
        return list(self.encoder.parameters())

    def dec_params(self):
        dec_mods = [self.proj3, self.proj6, self.proj9, self.proj12, self.fpn,
                    self.dec3, self.dec2, self.dec1, self.dec0,
                    self.seg_out, self.aux3, self.aux2,
                    self.film3, self.film2, self.film1, self.film0]  # FiLM = decoder
        return sum([list(m.parameters()) for m in dec_mods], [])

    def forward(self, x, return_aux=True, src_id=None):
        """src_id: (B,) long tensor of dataset source IDs. None -> default (DENTEX)."""
        B = x.shape[0]
        if src_id is None:
            src_id = torch.full((B,), SEG_SOURCES["default"],
                                dtype=torch.long, device=x.device)
        orig_h, orig_w = x.shape[2], x.shape[3]
        f3, f6, f9, f12 = self.encoder(x)

        fused = self.proj12(f12) + self.proj9(f9) + self.proj6(f6) + self.proj3(f3)
        fused = self.fpn(fused)

        # Decode + FiLM-condition each stage
        d3 = self.film3(self.dec3(F.interpolate(fused, scale_factor=2, mode="bilinear", align_corners=False)), src_id)
        d2 = self.film2(self.dec2(F.interpolate(d3,    scale_factor=2, mode="bilinear", align_corners=False)), src_id)
        d1 = self.film1(self.dec1(F.interpolate(d2,    scale_factor=2, mode="bilinear", align_corners=False)), src_id)
        d0 = self.film0(self.dec0(F.interpolate(d1,    scale_factor=2, mode="bilinear", align_corners=False)), src_id)

        main = self.seg_out(F.interpolate(d0, (orig_h, orig_w), mode="bilinear", align_corners=False))

        if return_aux and self.training:
            a3 = F.interpolate(self.aux3(d3), (orig_h, orig_w), mode="bilinear", align_corners=False)
            a2 = F.interpolate(self.aux2(d2), (orig_h, orig_w), mode="bilinear", align_corners=False)
            return main, a3, a2
        return main

# Additional architectures

class EfficientNetB7Encoder(nn.Module):
    """EfficientNet-B7 encoder using direct stage chaining (no torch.fx tracing).
    Channels at strides 2,4,8,16,32: [32, 48, 80, 224, 640].
    v10 fix: replaced create_feature_extractor (torch.fx) which hung silently on Windows.
    """
    def __init__(self):
        super().__init__()
        from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights
        base = efficientnet_b7(weights=EfficientNet_B7_Weights.DEFAULT)
        f = base.features  # 9 sequential stages
        self.stem = f[0]   # 3→64ch,  stride 2  → H/2
        self.s1   = f[1]   # 64→32ch, stride 1  → e0 (32ch,  H/2)
        self.s2   = f[2]   # 32→48ch, stride 2  → e1 (48ch,  H/4)
        self.s3   = f[3]   # 48→80ch, stride 2  → e2 (80ch,  H/8)
        self.s4   = f[4]   # 80→160ch, stride 2
        self.s5   = f[5]   # 160→224ch, stride 1 → e3 (224ch, H/16)
        self.s6   = f[6]   # 224→384ch, stride 2
        self.s7   = f[7]   # 384→640ch, stride 1 → e4 (640ch, H/32)
        self.out_channels = [32, 48, 80, 224, 640]

    def forward(self, x):
        x  = self.stem(x)
        e0 = self.s1(x)    # 32ch, H/2
        e1 = self.s2(e0)   # 48ch, H/4
        e2 = self.s3(e1)   # 80ch, H/8
        x  = self.s4(e2)   # 160ch, H/16
        e3 = self.s5(x)    # 224ch, H/16
        x  = self.s6(e3)   # 384ch, H/32
        e4 = self.s7(x)    # 640ch, H/32
        return e0, e1, e2, e3, e4


class SegUNetV10(nn.Module):
    def __init__(self, backbone_mode="ssl_resnet50", ssl_path=None, convnext_encoder=None, b7_encoder=None):
        super().__init__()
        self.backbone_mode = backbone_mode
        if backbone_mode == "efficientnet_b7":
            assert b7_encoder is not None
            self.encoder = b7_encoder
            self.aspp = ASPP(640, 256)
            self.d4 = _Up(224, 256, 192); self.d3 = _Up(80, 192, 128)
            self.d2 = _Up(48, 128, 64);   self.d1 = _Up(32, 64, 32)
            self.up_f = nn.ConvTranspose2d(32, 16, 2, stride=2)
            self.seg_out = nn.Conv2d(16, 1, 1)
            self.aux3 = nn.Conv2d(128, 1, 1); self.aux2 = nn.Conv2d(64, 1, 1)
            print("SegUNetV10: EfficientNet-B7 encoder (ImageNet pretrained), target IoU 0.86+")
        elif backbone_mode == "convnext_b":
            assert convnext_encoder is not None
            self.encoder = convnext_encoder
            self.aspp = ASPP(1024, 256)
            self.d4 = _Up(512, 256, 256); self.d3 = _Up(256, 256, 128)
            self.d2 = _Up(128, 128, 64)
            self.d1 = nn.Sequential(nn.ConvTranspose2d(64, 32, 2, stride=2), _DoubleConv(32, 32))
            self.up_f = nn.ConvTranspose2d(32, 16, 2, stride=2)
            self.seg_out = nn.Conv2d(16, 1, 1)
            self.aux3 = nn.Conv2d(128, 1, 1); self.aux2 = nn.Conv2d(64, 1, 1)
            print("SegUNetV10: ConvNeXt-B encoder (ImageNet pretrained)")
        else:
            _ssl = Path(ssl_path) if ssl_path else BACKBONE_SSL
            if not _ssl.exists(): raise RuntimeError(f"SSL backbone not found: {_ssl}")
            bb = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            sd = torch.load(_ssl, map_location="cpu", weights_only=True)
            if next(iter(sd), "").split(".")[0].isdigit():
                _m = {"0":"conv1","1":"bn1","4":"layer1","5":"layer2","6":"layer3","7":"layer4"}
                sd = {".".join([_m.get(k.split(".")[0], k.split(".")[0])] + k.split(".")[1:]): v
                      for k, v in sd.items() if k.split(".")[0] in _m}
            miss, _ = bb.load_state_dict(sd, strict=False)
            n_loaded = len(bb.state_dict()) - len(miss)
            assert n_loaded > 100, f"SSL suspicious: only {n_loaded} layers loaded"
            print(f"SegUNetV10: SSL ResNet50 loaded {n_loaded}/{len(bb.state_dict())} layers")
            self.e0 = nn.Sequential(bb.conv1, bb.bn1, bb.relu); self.mp = bb.maxpool
            self.e1 = bb.layer1; self.e2 = bb.layer2; self.e3 = bb.layer3; self.e4 = bb.layer4
            self.aspp = ASPP(2048, 256)
            self.d4 = _Up(1024, 256, 512); self.d3 = _Up(512, 512, 256)
            self.d2 = _Up(256, 256, 128);  self.d1 = _Up(64, 128, 64)
            self.up_f = nn.ConvTranspose2d(64, 32, 2, stride=2)
            self.seg_out = nn.Conv2d(32, 1, 1)
            self.aux3 = nn.Conv2d(256, 1, 1); self.aux2 = nn.Conv2d(128, 1, 1)

    def enc_params(self):
        if self.backbone_mode in ("efficientnet_b7", "convnext_b"):
            return list(self.encoder.parameters())
        return sum([list(m.parameters()) for m in [self.e0, self.e1, self.e2, self.e3, self.e4]], [])

    def dec_params(self):
        return sum([list(m.parameters()) for m in
                    [self.aspp, self.d4, self.d3, self.d2, self.d1, self.up_f, self.seg_out, self.aux3, self.aux2]], [])

    def forward(self, x, return_aux=True):
        bm = self.backbone_mode
        if bm == "efficientnet_b7":
            e0,e1,e2,e3,e4 = self.encoder(x); btn = self.aspp(e4)
            d4=self.d4(btn,e3); d3=self.d3(d4,e2); d2=self.d2(d3,e1); d1=self.d1(d2,e0)
            main = self.seg_out(self.up_f(d1))
        elif bm == "convnext_b":
            e0,e1,e2,e3 = self.encoder(x); btn = self.aspp(e3)
            d4=self.d4(btn,e2); d3=self.d3(d4,e1); d2=self.d2(d3,e0); d1=self.d1(d2)
            main = self.seg_out(self.up_f(d1))
        else:
            s0=self.e0(x); s1=self.e1(self.mp(s0)); s2=self.e2(s1); s3=self.e3(s2); s4=self.e4(s3)
            btn=self.aspp(s4)
            d4=self.d4(btn,s3); d3=self.d3(d4,s2); d2=self.d2(d3,s1); d1=self.d1(d2,s0)
            main = self.seg_out(self.up_f(d1))
        if return_aux and self.training:
            a3=F.interpolate(self.aux3(d3),x.shape[2:],mode="bilinear",align_corners=False)
            a2=F.interpolate(self.aux2(d2),x.shape[2:],mode="bilinear",align_corners=False)
            return main, a3, a2
        return main


_orig_build_encoder = build_encoder
def build_encoder(backbone_mode, backbone_ssl_path, device):
    if backbone_mode.startswith("dinov2_vit"):
        _dim = 1024 if "vitl" in backbone_mode else 768
        return None, [_dim] * 4  # encoder embedded in SegUNetDINO
    if backbone_mode == "efficientnet_b7":
        enc = EfficientNetB7Encoder().to(device)
        print("build_encoder: EfficientNet-B7 channels:", enc.out_channels)
        return enc, enc.out_channels
    return _orig_build_encoder(backbone_mode, backbone_ssl_path, device)


def build_seg_model(mode=None, ssl_path=None, device=None):
    """Unified seg model factory. mode defaults to BACKBONE_MODE."""
    _mode = mode or BACKBONE_MODE
    _ssl  = ssl_path or BACKBONE_SSL
    _dev  = device or DEVICE
    if _mode.startswith("dinov2_vit"):
        m = SegUNetDINO(model_name=_mode).to(_dev)
        n_p = sum(p.numel() for p in m.parameters()) / 1e6
        print(f"build_seg_model: SegUNetDINO ({_mode}) {n_p:.1f}M params")
        return m
    elif _mode == "efficientnet_b7":
        enc = EfficientNetB7Encoder().to(_dev)
        return SegUNetV10(backbone_mode="efficientnet_b7", b7_encoder=enc).to(_dev)
    elif _mode == "convnext_b":
        enc, _ = _orig_build_encoder("convnext_b", _ssl, _dev)
        return SegUNetV10(backbone_mode="convnext_b", convnext_encoder=enc).to(_dev)
    else:  # ssl_resnet50
        return SegUNetV10(backbone_mode=_mode, ssl_path=_ssl).to(_dev)


def build_hrnet_ceph(n_lm=N_LM, device=DEVICE):
    """ResNet-50 + lightweight decoder for cephalometric landmark heatmaps.
    Input: (B, 3, 512, 512) -> Output: (B, N_LM, 64, 64) heatmaps.
    Uses coordinate-space L1 loss via soft_argmax_2d (integral pose regression).
    """
    import torchvision
    class CephLandmarkNet(nn.Module):
        """ResNet-50 + U-Net skip decoder -> 128x128 heatmaps.
        16->32->64->128: 4px grid = 0.75mm steps (was 8px=1.5mm at 64x64).
        Skip connections from layer1/2/3 preserve spatial detail.
        """
        def __init__(self, n_lm):
            super().__init__()
            resnet = torchvision.models.resnet50(weights="IMAGENET1K_V2")
            self.stem   = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
            self.layer1 = resnet.layer1   # 128x128, 256ch
            self.layer2 = resnet.layer2   # 64x64,   512ch
            self.layer3 = resnet.layer3   # 32x32,   1024ch
            self.layer4 = resnet.layer4   # 16x16,   2048ch
            self.bridge = nn.Sequential(
                nn.Conv2d(2048, 512, 1, bias=False), nn.BatchNorm2d(512), nn.ReLU(True))
            self.up3 = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(512+1024, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True))
            self.up2 = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(256+512, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True))
            self.up1 = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(128+256, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True))
            self.head = nn.Conv2d(64, n_lm, 1)
            for p in self.stem.parameters(): p.requires_grad = False
        def forward(self, x):
            s  = self.stem(x)
            e1 = self.layer1(s)   # 128x128
            e2 = self.layer2(e1)  # 64x64
            e3 = self.layer3(e2)  # 32x32
            e4 = self.layer4(e3)  # 16x16
            d = self.bridge(e4)
            d = self.up3(torch.cat([F.interpolate(d, e3.shape[2:], mode="bilinear", align_corners=False), e3], 1))
            d = self.up2(torch.cat([F.interpolate(d, e2.shape[2:], mode="bilinear", align_corners=False), e2], 1))
            d = self.up1(torch.cat([F.interpolate(d, e1.shape[2:], mode="bilinear", align_corners=False), e1], 1))
            return self.head(d)   # (B, n_lm, 128, 128)
    model = CephLandmarkNet(n_lm=n_lm).to(device)
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
    n_total = sum(p.numel() for p in model.parameters())/1e6
    print(f"CephLandmarkNet: ResNet-50+UNet ({n_p:.1f}M trainable / {n_total:.1f}M total), heatmap 128x128")
    return model


def build_cvm_model(n_cls=3, device=DEVICE):
    import timm
    model = timm.create_model("convnextv2_tiny", pretrained=True, num_classes=n_cls).to(device)
    n_p = sum(p.numel() for p in model.parameters())/1e6
    print(f"CVM ConvNeXt-V2-T: {n_p:.1f}M params, target accuracy 85%+")
    return model


print("v12 architecture: SegUNetDINO+FiLM (novel, boundary loss), DINOv2Encoder, SegUNetV10, build_seg_model, build_hrnet_ceph, build_cvm_model")

# PHASE G — Instance Segmentation Architecture
# 33-class semantic segmentation (32 FDI teeth + background)

class CoordDoubleConv(nn.Module):
    """_DoubleConv with explicit (x,y) coordinate channels appended to input.
    Inspired by CoordConv (Liu et al. 2018) — used in YOLOrtho for dental FDI
    assignment. Gives decoder each level explicit knowledge of its position,
    improving spatial class associations (e.g. class 1 always upper-left).
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # +2 for the x and y coordinate channels
        self.net = nn.Sequential(
            nn.Conv2d(in_ch + 2, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(True))

    def forward(self, x):
        N, C, H, W = x.shape
        xs = torch.linspace(0, 1, W, device=x.device).view(1, 1, 1, W).expand(N, 1, H, W)
        ys = torch.linspace(0, 1, H, device=x.device).view(1, 1, H, 1).expand(N, 1, H, W)
        return self.net(torch.cat([x, xs, ys], dim=1))


class SegUNetDINO_Instance(nn.Module):
    """DINOv2 ViT + FPN decoder for 33-class instance segmentation.
    Same architecture as SegUNetDINO but output heads produce N_FDI_CLASSES channels.
    No FiLM conditioning (instance seg data is all from same annotation style).
    """
    def __init__(self, n_classes=N_FDI_CLASSES, model_name=None):
        super().__init__()
        _mn = model_name or BACKBONE_MODE
        self.encoder = DINOv2Encoder(_mn)
        self.backbone_mode = _mn
        self.n_classes = n_classes
        dim = self.encoder.embed_dim; pdim = 256

        self.proj3  = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.proj6  = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.proj9  = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.proj12 = nn.Sequential(nn.Conv2d(dim, pdim, 1), nn.BatchNorm2d(pdim), nn.GELU())
        self.fpn    = _DoubleConv(pdim, pdim)

        self.dec3 = CoordDoubleConv(pdim, 256)
        self.dec2 = CoordDoubleConv(256, 128)
        self.dec1 = CoordDoubleConv(128, 64)
        self.dec0 = CoordDoubleConv(64, 32)

        self.seg_out = nn.Conv2d(32, n_classes, 1)
        # Quadrant auxiliary head: 5-class (bg + Q1 + Q2 + Q3 + Q4)
        # Training signal from quadrant classification is much easier to learn:
        # predicting which of 4 spatial regions a pixel belongs to.
        # Derived from FDI labels: classes 1-8→Q1, 9-16→Q2, 17-24→Q3, 25-32→Q4.
        self.quad_out = nn.Conv2d(128, 5, 1)   # 5 = bg + 4 quadrants
        self.aux3    = nn.Conv2d(256, n_classes, 1)
        self.aux2    = nn.Conv2d(128, n_classes, 1)
        # binary mask guidance projection (Phase A mask → feature injection + gating)
        self.mask_proj = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, pdim, 1),
        )
        # Coordinate embedding: normalised (x,y) so model knows FDI position
        self.coord_proj = nn.Sequential(
            nn.Conv2d(2, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, pdim, 1),
        )

    def enc_params(self): return list(self.encoder.parameters())

    def dec_params(self):
        return sum([list(m.parameters()) for m in [self.proj3,self.proj6,self.proj9,self.proj12,self.fpn,
                    self.dec3,self.dec2,self.dec1,self.dec0,self.seg_out,self.aux3,self.aux2,
                    self.mask_proj, self.coord_proj, self.quad_out]], [])

    def forward(self, x, binary_mask=None, return_aux=True):
        orig_h, orig_w = x.shape[2], x.shape[3]
        f3, f6, f9, f12 = self.encoder(x)
        fused = self.proj12(f12) + self.proj9(f9) + self.proj6(f6) + self.proj3(f3)
        fused = self.fpn(fused)
        # Coordinate embedding: normalised (x, y) grid injected at FPN level
        _N, _, _Hf, _Wf = fused.shape
        _xs = torch.linspace(0, 1, _Wf, device=fused.device).view(1,1,1,_Wf).expand(_N,1,_Hf,_Wf)
        _ys = torch.linspace(0, 1, _Hf, device=fused.device).view(1,1,_Hf,1).expand(_N,1,_Hf,_Wf)
        fused = fused + self.coord_proj(torch.cat([_xs, _ys], dim=1))
        # binary mask guidance — inject Phase A mask as spatial gate
        if binary_mask is not None and PHASE_B_BINARY_GUIDE:
            _gate = F.interpolate(binary_mask, size=fused.shape[-2:], mode='bilinear', align_corners=False)
            fused = (fused + self.mask_proj(_gate)) * _gate
        d3 = self.dec3(F.interpolate(fused, scale_factor=2, mode="bilinear", align_corners=False))
        d2 = self.dec2(F.interpolate(d3, scale_factor=2, mode="bilinear", align_corners=False))
        d1 = self.dec1(F.interpolate(d2, scale_factor=2, mode="bilinear", align_corners=False))
        d0 = self.dec0(F.interpolate(d1, scale_factor=2, mode="bilinear", align_corners=False))
        main = self.seg_out(F.interpolate(d0, (orig_h, orig_w), mode="bilinear", align_corners=False))
        if return_aux and self.training:
            a3 = F.interpolate(self.aux3(d3), (orig_h, orig_w), mode="bilinear", align_corners=False)
            a2 = F.interpolate(self.aux2(d2), (orig_h, orig_w), mode="bilinear", align_corners=False)
            # Quadrant aux head (from d2 = 128-ch feature map)
            aq = F.interpolate(self.quad_out(d2), (orig_h, orig_w), mode="bilinear", align_corners=False)
            return main, a3, a2, aq
        return main


def multiclass_dice_loss(pred, tgt, n_classes=N_FDI_CLASSES, smooth=1.0):
    """Per-class Dice loss, averaged over tooth classes (skip background=0)."""
    prob = F.softmax(pred, dim=1)  # (B, C, H, W)
    tgt_oh = F.one_hot(tgt, n_classes).permute(0, 3, 1, 2).float()  # (B, C, H, W)
    dice_sum = 0.0
    n_valid = 0
    for c in range(1, n_classes):  # skip background
        pc = prob[:, c]; tc = tgt_oh[:, c]
        inter = (pc * tc).sum()
        union = pc.sum() + tc.sum()
        if union > 0:
            dice_sum += 1.0 - (2 * inter + smooth) / (union + smooth)
            n_valid += 1
    return dice_sum / max(n_valid, 1)


def _fdi_to_quad(tgt):
    """Convert 33-class FDI label map to 5-class quadrant map (0=bg,1-4=Q1-Q4)."""
    q = torch.zeros_like(tgt)
    q[(tgt >= 1)  & (tgt <= 8)]  = 1  # Q1
    q[(tgt >= 9)  & (tgt <= 16)] = 2  # Q2
    q[(tgt >= 17) & (tgt <= 24)] = 3  # Q3
    q[(tgt >= 25) & (tgt <= 32)] = 4  # Q4
    return q


def instance_seg_loss(pred, tgt, a3=None, a2=None, aq=None, n_classes=N_FDI_CLASSES):
    """Multi-class segmentation loss: CE + multiclass Dice + deep supervision.
    pred: (B, C, H, W) logits, tgt: (B, H, W) long labels 0..32
    aq:   (B, 5, H, W) quadrant logits (optional, from quad_out auxiliary head)
    """
    # Clamp labels: raw FDI values (11-48) or off-by-one errors cause CUDA assert
    tgt = tgt.clamp(0, n_classes - 1)
    # Background weight reduction (background dominates ~85% of pixels)
    _weights = torch.ones(n_classes, device=pred.device)
    _weights[0] = 0.1   # background — suppress (dominates ~85% pixels)
    for _wi in [8, 16, 24, 32]:  # wisdom teeth (FDI 18,28,38,48) — rarest, 3x boost
        _weights[_wi] = 3.0
    for _wi in [7, 15, 23, 31]:  # second molars (FDI 17,27,37,47) — mild boost
        _weights[_wi] = 1.5
    ce = F.cross_entropy(pred, tgt, weight=_weights, label_smoothing=0.05)
    dc = multiclass_dice_loss(pred, tgt, n_classes)
    loss = 0.50 * ce + 0.50 * dc
    if a3 is not None:
        loss += 0.5 * (0.50 * F.cross_entropy(a3, tgt, weight=_weights, label_smoothing=0.05)
                       + 0.50 * multiclass_dice_loss(a3, tgt, n_classes))
    if a2 is not None:
        loss += 0.3 * (0.50 * F.cross_entropy(a2, tgt, weight=_weights, label_smoothing=0.05)
                       + 0.50 * multiclass_dice_loss(a2, tgt, n_classes))
    if aq is not None:
        # Quadrant auxiliary loss (5-class, weighted: bg=0.1, Q1-Q4=1.0)
        _qw = torch.tensor([0.1, 1.0, 1.0, 1.0, 1.0], device=pred.device)
        _qtgt = _fdi_to_quad(tgt)
        loss += 0.3 * F.cross_entropy(aq, _qtgt, weight=_qw)
    return loss


def instance_seg_metrics(pred, tgt, n_classes=N_FDI_CLASSES):
    """Compute per-class IoU, mIoU, and tooth identification accuracy."""
    pred_cls = pred.argmax(dim=1)  # (B, H, W)
    ious = {}
    for c in range(1, n_classes):
        pc = (pred_cls == c); tc = (tgt == c)
        inter = (pc & tc).sum().item()
        union = (pc | tc).sum().item()
        if union > 0:
            ious[c] = inter / union
    miou = np.mean(list(ious.values())) if ious else 0.0
    # Overall pixel accuracy (excluding background)
    fg_mask = tgt > 0
    if fg_mask.sum() > 0:
        fg_acc = (pred_cls[fg_mask] == tgt[fg_mask]).float().mean().item()
    else:
        fg_acc = 0.0
    return {"miou": miou, "fg_acc": fg_acc, "per_class_iou": ious,
            "n_classes_detected": len(ious)}


def build_instance_seg_model(device=None):
    """Build instance segmentation model (33-class)."""
    _dev = device or DEVICE
    m = SegUNetDINO_Instance(n_classes=N_FDI_CLASSES, model_name=INST_BACKBONE_MODE).to(_dev)
    n_p = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"build_instance_seg_model: {n_p:.1f}M params, {N_FDI_CLASSES} classes")
    return m


print("v19: Instance seg architecture defined (SegUNetDINO_Instance + mask_proj gating, multiclass_dice_loss, instance_seg_loss)")

## Phase A — Tooth Segmentation

DINOv2 ViT-B/14 + FPN decoder. Trains for 200 epochs with SWA.

In [ ]:

class SegDataset(Dataset):
    def __init__(self, imgs, masks, aug=False):
        self.imgs=imgs; self.masks=masks; self.aug=aug
        self.src_ids=[get_source_id(p) for p in imgs]  # v12: pre-computed source IDs
        self.is_pseudo=[is_pseudo_source(p) for p in imgs]  # v13: pseudo-label flag
        self.ttf=A.Compose([
            A.RandomResizedCrop(size=(SEG_SIZE,SEG_SIZE),scale=(0.7,1.0)),
            A.HorizontalFlip(p=0.5),
            A.Affine(translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
                     scale=(0.85, 1.15), rotate=(-20, 20), p=0.6),
            A.RandomGamma(gamma_limit=(80,120),p=0.4),
            A.ElasticTransform(alpha=120,sigma=6,p=0.2),  # tissue deformation aug — important for medical seg
            A.CoarseDropout(num_holes_range=(1,8), hole_height_range=(8,32), hole_width_range=(8,32), p=0.3),
            A.OneOf([A.GaussNoise(std_range=(0.012,0.028)),A.GaussianBlur(blur_limit=3), 
                     A.RandomBrightnessContrast(0.2,0.2)],p=0.5),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()
        ],additional_targets={"mask":"mask"})
        self.vtf=A.Compose([A.Resize(SEG_SIZE,SEG_SIZE),
                             A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()
                            ],additional_targets={"mask":"mask"})
    def __len__(self): return len(self.imgs)
    def __getitem__(self,idx):
        img=np.array(Image.open(self.imgs[idx]).convert("RGB"))
        img = apply_clahe(img)  # X-ray contrast enhancement — measurably improves IoU
        mask=(np.array(Image.open(self.masks[idx]).convert("L"))>127).astype(np.float32)
        out=(self.ttf if self.aug else self.vtf)(image=img,mask=mask)
        return out["image"],out["mask"].unsqueeze(0),self.src_ids[idx],torch.tensor(float(self.is_pseudo[idx]))

all_imgs=sorted(SEG_IMG_DIR.glob("*.png"))
# remove blob-source samples (yolo_*, ext_opg_*) -- synthetic ellipse masks
# These teach the model to produce blobs; real tooth seg needs polygon masks only
valid_pairs=[(i,SEG_MASK_DIR/i.name) for i in all_imgs
             if (SEG_MASK_DIR/i.name).exists() and not is_blob_source(i) and not is_pseudo_source(i)
             and i.stem not in _flagged_stems]
_n_total=sum(1 for _ in all_imgs); _n_kept=len(valid_pairs)
print(f"v12 blob filter: kept {_n_kept} real-mask samples, removed {_n_total-_n_kept} blob samples")
# add pseudo-labeled samples from ps_d2_* files
_pseudo_imgs = sorted(SEG_IMG_DIR.glob("ps_d2_*.png"))
_pseudo_pairs = [(i, SEG_MASK_DIR/i.name) for i in _pseudo_imgs
                 if (SEG_MASK_DIR/i.name).exists()]
print(f"  Pseudo-labeled: {len(_pseudo_pairs)} available")
# pseudo-labels capped to SEG_PSEUDO_CAP_RATIO * real GT count
_max_pseudo = int(len(valid_pairs) * SEG_PSEUDO_CAP_RATIO)
if len(_pseudo_pairs) > _max_pseudo:
    import random as _rng
    _rng.seed(SEED)
    _pseudo_pairs = _rng.sample(_pseudo_pairs, _max_pseudo)
    print(f"  Pseudo-labels capped: {_max_pseudo}/{len(_pseudo_imgs)} (ratio {SEG_PSEUDO_CAP_RATIO}:1)")
else:
    print(f"  Pseudo-labels: {len(_pseudo_pairs)} (within cap)")
all_imgs,all_masks=zip(*valid_pairs) if valid_pairs else ([],[])
# exclude sts24 from val (only 5 samples -> noisy IoU signal 0.73-0.89)
# Keep sts24 in training only — too few for reliable val metric
_sts24_mask = [str(p.stem).startswith('sts24_') for p in all_imgs]
_non_sts_idx = [i for i, m in enumerate(_sts24_mask) if not m]
_sts_idx     = [i for i, m in enumerate(_sts24_mask) if m]
n_val=max(1,int(0.15*len(_non_sts_idx))); rng_split=np.random.RandomState(SEED)
idx=rng_split.permutation(len(_non_sts_idx))
tr_i,val_i=idx[n_val:].tolist(),idx[:n_val].tolist()
tr_imgs=[all_imgs[_non_sts_idx[i]] for i in tr_i] + [all_imgs[i] for i in _sts_idx]
tr_masks=[all_masks[_non_sts_idx[i]] for i in tr_i] + [all_masks[i] for i in _sts_idx]
vl_imgs=[all_imgs[_non_sts_idx[i]] for i in val_i]
vl_masks=[all_masks[_non_sts_idx[i]] for i in val_i]
print(f'v14 STS24 split: {len(_sts_idx)} sts24 samples -> training only (not in val)')

# add pseudo-labeled samples to training only (never validation)
if _pseudo_pairs:
    _ps_imgs, _ps_masks = zip(*_pseudo_pairs)
    _all_tr_imgs  = list(tr_imgs)  + list(_ps_imgs)
    _all_tr_masks = list(tr_masks) + list(_ps_masks)
    print(f"  Training: {len(tr_imgs)} labeled + {len(_ps_imgs)} pseudo = {len(_all_tr_imgs)} total")
else:
    _all_tr_imgs, _all_tr_masks = list(tr_imgs), list(tr_masks)
    print(f"  Training: {len(tr_imgs)} labeled only (no pseudo-labels yet)")

seg_tr_ds=SegDataset(_all_tr_imgs,_all_tr_masks,aug=True)
seg_vl_ds=SegDataset(vl_imgs,vl_masks,aug=False)
seg_tr_ld=DataLoader(seg_tr_ds,SEG_BATCH,shuffle=True, num_workers=NUM_WORKERS,pin_memory=True)
seg_vl_ld=DataLoader(seg_vl_ds,SEG_BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
print(f"Seg train:{len(seg_tr_ds)} val:{len(seg_vl_ds)}")

In [ ]:

# build_seg_model() handles dinov2_vitb14 / efficientnet_b7 / convnext_b / ssl_resnet50
seg_model = build_seg_model(mode=BACKBONE_MODE, ssl_path=BACKBONE_SSL, device=DEVICE)
_need_seg=True
if SEG_CKPT.exists():
    try:
        seg_model.load_state_dict(torch.load(SEG_CKPT,map_location=DEVICE,weights_only=True))
        print(f"SKIP seg — loaded {SEG_CKPT.name}. Delete to retrain."); _need_seg=False
    except RuntimeError: print("Incompatible seg ckpt — retraining"); SEG_CKPT.unlink()
else:
    # warm-start only if backbone architecture matches (v14 is ViT-B/14, v17 is ViT-L/14)
    _ws_loaded = False
    if V14_SEG_CKPT.exists():
        _ws_sd = torch.load(V14_SEG_CKPT, map_location=DEVICE, weights_only=True)
        _ws_edim = _ws_sd.get('encoder.dino.norm.weight', torch.empty(0)).shape[0]
        _cur_edim = seg_model.encoder.embed_dim
        if _ws_edim == _cur_edim:
            seg_model.load_state_dict(_ws_sd)
            print(f"v17 warm-start from {V14_SEG_CKPT.name} — fine-tuning {SEG_EPOCHS} epochs")
            _ws_loaded = True
        else:
            print(f"v17: v14 checkpoint is ViT dim={_ws_edim}, current model is dim={_cur_edim} — skipping warm-start")
        del _ws_sd
    if not _ws_loaded:
        print(f"v17: training from DINOv2 pretrained weights ({BACKBONE_MODE}) — no compatible warm-start found")


if _need_seg:
    enc_p = seg_model.enc_params()  # v10: unified enc_params() works for all backbone modes
    dec_p = seg_model.dec_params()
    opt=torch.optim.AdamW([{"params":enc_p,"lr":ENCODER_LR},{"params":dec_p,"lr":DECODER_LR}],weight_decay=1e-4)  # v16: warm-start LR=1e-5/1e-4
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=SEG_EPOCHS,eta_min=1e-7)  # v16: monotonic decay over 100 epochs
    # GradScaler removed — bfloat16 has fp32 exponent range, scaler causes hangs with ViT-L
    # SWA — Stochastic Weight Averaging (last 20% of epochs, +0.5-1.5% IoU)
    from torch.optim.swa_utils import AveragedModel, update_bn as swa_update_bn
    _swa_start = int(0.6 * SEG_EPOCHS)  # v15: epoch 60/100 (v14 early-stopped before SWA at 80%)
    swa_seg = AveragedModel(seg_model)
    _swa_count = 0

    def _freeze(model, frozen):
        # DINOv2 freeze managed in __init__; B7/ConvNeXt use encoder attr; ResNet50 uses e0-e4
        bm = getattr(model, "backbone_mode", "")
        if bm.startswith("dinov2_vit"):
            pass  # DINOv2: freeze managed in __init__
        elif bm in ("efficientnet_b7", "convnext_b"):
            for p in model.encoder.parameters():
                p.requires_grad = not frozen
            model.encoder.eval() if frozen else model.encoder.train()
        else:
            for m in [model.e0,model.e1,model.e2,model.e3,model.e4]:
                for p in m.parameters(): p.requires_grad=not frozen
                if frozen: m.eval()

    best_iou=0.0; hist_seg=[]
    _es_patience = 0  # v14: early stopping counter
    for ep in range(SEG_EPOCHS):
        # warm-start — all blocks trainable from epoch 1 (v14 already trained all blocks)
        if ep == 0:
            bm_ws = getattr(seg_model, "backbone_mode", "")
            if bm_ws.startswith("dinov2_vit"):
                for p in seg_model.encoder.dino.parameters():
                    p.requires_grad_(True)
                print("  v16 warm-start: all DINOv2 blocks unfrozen from epoch 1")
            elif bm_ws in ("efficientnet_b7", "convnext_b"):
                for p in seg_model.encoder.parameters():
                    p.requires_grad = True
                seg_model.encoder.train()
            else:
                _freeze(seg_model, False)

        seg_model.train(); tr_l=0.0
        for imgs,masks,src_ids,is_pseudo_b in tqdm(seg_tr_ld,desc=f"[Seg] Ep {ep+1}/{SEG_EPOCHS}",leave=False):
            imgs,masks,src_ids=imgs.to(DEVICE),masks.to(DEVICE),src_ids.to(DEVICE)
            with autocast(dtype=torch.bfloat16):
                out=seg_model(imgs,return_aux=True,src_id=src_ids)
                if isinstance(out,tuple): loss=seg_loss_fn(out[0],masks,out[1],out[2])
                else: loss=seg_loss_fn(out,masks)
            _ps_scale = 1.0 - 0.5 * is_pseudo_b.float().mean().item()
            loss = loss * _ps_scale
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(seg_model.parameters(),1.0)
            opt.step(); tr_l+=loss.item()
        sch.step()
        if ep >= _swa_start:
            swa_seg.update_parameters(seg_model); _swa_count += 1

        seg_model.eval()
        _m=dict(iou=0.,dice=0.,prec=0.,rec=0.,pacc=0.)
        _auc_collect=[]
        with torch.no_grad():
            for imgs,masks,src_ids,_ in seg_vl_ld:
                imgs_d,masks_d=imgs.to(DEVICE),masks.to(DEVICE)
                src_d=src_ids.to(DEVICE)
                # multi-scale TTA — 6 predictions (3 scales x 2 flips), +0.3-0.5% IoU
                _tta_preds = []
                _oh, _ow = imgs_d.shape[2], imgs_d.shape[3]
                for _sc in [1.0, 0.9, 1.1]:
                    if _sc != 1.0:
                        _sh, _sw = int(_oh * _sc), int(_ow * _sc)
                        _imgs_sc = F.interpolate(imgs_d, (_sh, _sw), mode="bilinear", align_corners=False)
                    else:
                        _imgs_sc = imgs_d
                    _p_o = seg_model(_imgs_sc, return_aux=False, src_id=src_d)
                    _p_f = torch.flip(seg_model(torch.flip(_imgs_sc, [3]), return_aux=False, src_id=src_d), [3])
                    if _sc != 1.0:
                        _p_o = F.interpolate(_p_o, (_oh, _ow), mode="bilinear", align_corners=False)
                        _p_f = F.interpolate(_p_f, (_oh, _ow), mode="bilinear", align_corners=False)
                    _tta_preds.extend([_p_o, _p_f])
                bm=seg_metrics(sum(_tta_preds) / len(_tta_preds), masks_d, collect_for_auc=_auc_collect)
                for k in _m: _m[k]+=bm[k]
        n=len(seg_vl_ld)
        for k in _m: _m[k]/=n
        _auc=compute_seg_auc(_auc_collect)
        val_iou=_m["iou"]
        hist_seg.append(dict(ep=ep+1,loss=tr_l/len(seg_tr_ld),**_m))
        # Per-source IoU diagnostic (helps track blob vs detail gap)
        if (ep+1) % 10 == 0 or ep == SEG_EPOCHS-1:  # every 10 epochs + last epoch
            _src_iou={sid:[0.,0] for sid in range(N_SEG_SOURCES)}
            with torch.no_grad():
                for _vi,(_vm,_vmask,_vsrc,_) in enumerate(seg_vl_ld):
                    _vp=seg_model(_vm.to(DEVICE),return_aux=False,src_id=_vsrc.to(DEVICE))
                    for _b in range(len(_vsrc)):
                        _s=_vsrc[_b].item()
                        _src_iou[_s][0]+=seg_metrics(_vp[_b:_b+1],_vmask[_b:_b+1].to(DEVICE))['iou']
                        _src_iou[_s][1]+=1
            _sn={v:k for k,v in SEG_SOURCES.items()}
            _sr="  ".join(f"{_sn[s]}:{_src_iou[s][0]/_src_iou[s][1]:.3f}(n={_src_iou[s][1]})"
                          for s in range(N_SEG_SOURCES) if _src_iou[s][1]>0)
            print(f"  PerSrc: {_sr}")
        print(f"  Seg {ep+1:02d}  loss={tr_l/len(seg_tr_ld):.4f}  "
              f"IoU={_m['iou']:.4f}  Dice/F1={_m['dice']:.4f}  "
              f"Prec={_m['prec']:.4f}  Rec={_m['rec']:.4f}  PixAcc={_m['pacc']:.4f}  AUC={_auc:.4f}")
        if val_iou>best_iou:
            best_iou=val_iou; torch.save(seg_model.state_dict(),SEG_CKPT)
            print(f"    Best seg saved  IoU={_m['iou']:.4f}  Dice/F1={_m['dice']:.4f}  "
                  f"Prec={_m['prec']:.4f}  Rec={_m['rec']:.4f}  AUC={_auc:.4f}")
            _es_patience = 0
        else:
            _es_patience += 1
            # disable early stopping during SWA window — must let SWA complete
            if _es_patience >= EARLY_STOP_PAT and ep < _swa_start:
                print(f"  Early stop at ep {ep+1}: no improvement for {EARLY_STOP_PAT} epochs (best={best_iou:.4f})")
                break
            elif _es_patience >= EARLY_STOP_PAT and ep >= _swa_start:
                print(f"  Patience exhausted but in SWA window (ep {ep+1}) — continuing")
    print(f"Phase A done. Best IoU={best_iou:.4f} (target 0.88+ DINOv2 / 0.86+ B7)")
    if _swa_count > 0:
        print(f"SWA: averaging {_swa_count} epoch checkpoints, updating BN stats...")
        seg_model.train()
        with torch.no_grad():
            swa_update_bn(seg_tr_ld, swa_seg, device=DEVICE)
        swa_seg.eval()
        _sm2=dict(iou=0.,dice=0.,prec=0.,rec=0.,pacc=0.); _swa_auc=[]
        with torch.no_grad():
            for imgs,masks,src_ids,_ in seg_vl_ld:
                imgs_d,masks_d=imgs.to(DEVICE),masks.to(DEVICE); src_d=src_ids.to(DEVICE)
                _swa_tta = []
                _soh, _sow = imgs_d.shape[2], imgs_d.shape[3]
                for _sc in [1.0, 0.9, 1.1]:
                    if _sc != 1.0:
                        _ssh, _ssw = int(_soh * _sc), int(_sow * _sc)
                        _simgs = F.interpolate(imgs_d, (_ssh, _ssw), mode="bilinear", align_corners=False)
                    else:
                        _simgs = imgs_d
                    _sp1 = swa_seg(_simgs, return_aux=False, src_id=src_d)
                    _sp2 = torch.flip(swa_seg(torch.flip(_simgs, [3]), return_aux=False, src_id=src_d), [3])
                    if _sc != 1.0:
                        _sp1 = F.interpolate(_sp1, (_soh, _sow), mode="bilinear", align_corners=False)
                        _sp2 = F.interpolate(_sp2, (_soh, _sow), mode="bilinear", align_corners=False)
                    _swa_tta.extend([_sp1, _sp2])
                bm=seg_metrics(sum(_swa_tta) / len(_swa_tta), masks_d, collect_for_auc=_swa_auc)
                for k in _sm2: _sm2[k]+=bm[k]
        n=len(seg_vl_ld)
        for k in _sm2: _sm2[k]/=n
        swa_auc=compute_seg_auc(_swa_auc)
        print(f"SWA IoU={_sm2['iou']:.4f}  Dice/F1={_sm2['dice']:.4f}  AUC={swa_auc:.4f}")
        if _sm2['iou']>best_iou:
            best_iou=_sm2['iou']
            torch.save(swa_seg.module.state_dict(),SEG_CKPT)
            print(f"  SWA model saved as best (IoU={best_iou:.4f})")
    print(f"  Dice/F1={_m['dice']:.4f}  AUC={_auc:.4f}  Prec={_m['prec']:.4f}  Rec={_m['rec']:.4f}")
    print(f"  Note: Dice == F1 (same formula: 2TP/(2TP+FP+FN))")


In [ ]:

seg_model.eval()
fig,axes=plt.subplots(3,3,figsize=(14,10))
for i,(imgs,masks,src_ids,_) in enumerate(seg_vl_ld):
    if i>=3: break
    with torch.no_grad(): preds=seg_model(imgs[:1].to(DEVICE),return_aux=False,src_id=src_ids[:1].to(DEVICE))
    axes[i,0].imshow(INV_NORM(imgs[0]).permute(1,2,0).clamp(0,1).numpy()); axes[i,0].set_title("X-ray")
    axes[i,1].imshow(masks[0,0].numpy(),cmap="gray"); axes[i,1].set_title("GT mask")
    axes[i,2].imshow((torch.sigmoid(preds[0,0])>0.5).cpu().numpy(),cmap="gray"); axes[i,2].set_title("Pred")
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.savefig(VIS_DIR/"seg_v6.png",dpi=100,bbox_inches="tight"); plt.show()


## Phase B — FDI Instance Segmentation (YOLOv8x-seg)

YOLOv8x-seg trained on 3,957 images (DENTEX + HITL + AKUDENTAL + Dataset1) for 200 epochs.


In [ ]:
def _remap_hitl(c):
    if 1  <= c <= 8:  return 9  - c   # Q1 reversed: 1→8, 8→1
    if 9  <= c <= 16: return c          # Q2 correct
    if 17 <= c <= 24: return 41 - c   # Q3 reversed: 17→24, 24→17
    if 25 <= c <= 32: return c          # Q4 correct
    return 0

# PHASE B PREP — Render Instance Segmentation Masks (v19)
# DENTEX: quadrant_enumeration COCO polygons → FDI label maps
# HITL: Supervisely JSON polygons → FDI label maps

_inst_sentinel = OUT / ".inst_masks_v20_done"
# NOTE: Dataset1 (Other users/instance seg/Datset 1) is IDENTICAL to HITL (same 598 files).
# Adding d1hitl_ masks would duplicate hitl_ data → data leakage in train/val split.
# Valid training data: DENTEX(634) + HITL(598) + AKUDENTAL(333) = 1565 images.
if not _inst_sentinel.exists():
    import cv2
    SEG_INST_IMG_DIR.mkdir(parents=True, exist_ok=True)
    SEG_INST_MASK_DIR.mkdir(parents=True, exist_ok=True)
    _n_akudental = 0; _n_d1hitl = 0  # initialised here, computed in dataset blocks below

    # 1. DENTEX quadrant_enumeration
    with open(ENUM_JSON) as f:
        enum_data = json.load(f)
    _id2img = {img["id"]: img for img in enum_data["images"]}
    _id2anns = {}
    for ann in enum_data["annotations"]:
        _id2anns.setdefault(ann["image_id"], []).append(ann)

    # FDI mapping: quadrant(0-3) * 8 + tooth_pos(0-7) + 1 → FDI 1-32
    # Q0=upper-right(11-18), Q1=upper-left(21-28), Q2=lower-left(31-38), Q3=lower-right(41-48)
    _dentex_count = 0
    for img_id, img_info in _id2img.items():
        anns = _id2anns.get(img_id, [])
        if not anns: continue
        fname = img_info["file_name"]
        h, w = img_info["height"], img_info["width"]
        label_map = np.zeros((h, w), dtype=np.uint8)
        for ann in anns:
            q = ann.get("category_id_1", 0)
            t = ann.get("category_id_2", 0)
            fdi = q * 8 + t + 1  # 1-32
            if fdi < 1 or fdi > 32: continue
            for seg in ann.get("segmentation", []):
                pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(label_map, [pts], int(fdi))  # NO remap — DENTEX already centripetal order
        # Resize to 512x512 with nearest (preserve class IDs)
        label_map = cv2.resize(label_map, (512, 512), interpolation=cv2.INTER_NEAREST)
        # Copy image
        src_img = ENUM_XRAYS / fname
        if not src_img.exists(): continue
        img = cv2.imread(str(src_img))
        img = cv2.resize(img, (512, 512))
        stem = Path(fname).stem
        cv2.imwrite(str(SEG_INST_IMG_DIR / f"dentex_{stem}.png"), img)
        cv2.imwrite(str(SEG_INST_MASK_DIR / f"dentex_{stem}.png"), label_map)
        _dentex_count += 1
    print(f"DENTEX instance masks: {_dentex_count} images rendered")

    # HITL class remap (Q1 and Q3 are reversed vs DENTEX)
    # DENTEX: class 1=Q1-central-incisor (near midline), class 8=Q1-wisdom (far)
    # HITL:   class 1=Q1-wisdom (far left), class 8=Q1-central (near midline)
    # → Within Q1 (1-8) and Q3 (17-24) the HITL ordering is centrifugal-to-centripetal,
    #   i.e. the REVERSE of DENTEX. Q2 (9-16) and Q4 (25-32) are already consistent.
    
    # 2. HITL Supervisely
    _hitl_ann_dir = EXT_HITL / "ann"
    _hitl_img_dir = EXT_HITL / "img"
    _hitl_count = 0
    if _hitl_ann_dir.exists():
        for ann_file in sorted(_hitl_ann_dir.glob("*.json")):
            with open(ann_file) as f:
                sly = json.load(f)
            h = sly["size"]["height"]; w = sly["size"]["width"]
            label_map = np.zeros((h, w), dtype=np.uint8)
            for obj in sly.get("objects", []):
                cls_title = obj.get("classTitle", "")
                try: fdi = int(cls_title)
                except ValueError: continue
                if fdi < 1 or fdi > 32: continue
                if obj.get("geometryType") == "polygon":
                    pts = np.array(obj["points"]["exterior"]).astype(np.int32)
                    cv2.fillPoly(label_map, [pts], int(_remap_hitl(fdi)))
            label_map = cv2.resize(label_map, (512, 512), interpolation=cv2.INTER_NEAREST)
            # Find corresponding image
            img_name = ann_file.stem  # e.g., "img_001.jpg"
            if img_name.endswith(".jpg") or img_name.endswith(".png"):
                pass  # already has extension
            img_path = _hitl_img_dir / img_name
            if not img_path.exists():
                # Try common extensions
                for ext in [".jpg", ".png", ".jpeg"]:
                    _try = _hitl_img_dir / (img_name.rsplit(".", 1)[0] + ext) if "." in img_name else _hitl_img_dir / (img_name + ext)
                    if _try.exists(): img_path = _try; break
            if not img_path.exists(): continue
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.resize(img, (512, 512))
            stem = Path(img_name).stem
            cv2.imwrite(str(SEG_INST_IMG_DIR / f"hitl_{stem}.png"), img)
            cv2.imwrite(str(SEG_INST_MASK_DIR / f"hitl_{stem}.png"), label_map)
            _hitl_count += 1
    print(f"HITL instance masks: {_hitl_count} images rendered")

    # Verify: show unique classes in a few masks
    _sample_masks = sorted(SEG_INST_MASK_DIR.glob("*.png"))[:5]
    for mp in _sample_masks:
        m = np.array(Image.open(mp).convert("L"))
        classes = np.unique(m)
        print(f"  {mp.name}: classes={classes}, n_teeth={len(classes)-1}")


    # Expert/Student excluded — sequential per-image IDs, not FDI-aligned (Bug #39)


    # 3. AKUDENTAL (333 panoramic X-rays, COCO FDI notation 11-48)
    # No remap needed — AKUDENTAL uses standard FDI centripetal ordering (same as DENTEX)
    # Category names: "11 - Central Incisor", "21 - Central Incisor", etc.
    def _fdi_to_class_idx(fdi: int) -> int:
        """Convert FDI notation (11-48) to class index (1-32)."""
        q, t = fdi // 10, fdi % 10  # q=1-4, t=1-8
        if q == 1: return t           # Q1: 11→1, 12→2, ..., 18→8
        if q == 2: return 8 + t       # Q2: 21→9, ..., 28→16
        if q == 3: return 16 + t      # Q3: 31→17, ..., 38→24
        if q == 4: return 24 + t      # Q4: 41→25, ..., 48→32
        return 0                      # non-tooth (bridge/filling/implant)

    _n_akudental = 0
    if AKUDENTAL_JSON.exists() and AKUDENTAL_IMG_DIR.exists():
        with open(AKUDENTAL_JSON, encoding="utf-8") as _f:
            _aku = json.load(_f)
        # Build category_id → FDI mapping from category names
        _aku_cat2fdi = {}
        for _cat in _aku.get("categories", []):
            try:
                _fdi_num = int(str(_cat["name"]).split()[0])
                if 11 <= _fdi_num <= 48:
                    _aku_cat2fdi[_cat["id"]] = _fdi_num
            except (ValueError, IndexError):
                pass  # bridge, filling-crown, implant — skip
        _aku_id2img = {im["id"]: im for im in _aku["images"]}
        _aku_id2anns = {}
        for _ann in _aku.get("annotations", []):
            _aku_id2anns.setdefault(_ann["image_id"], []).append(_ann)
        for _im in _aku["images"]:
            _anns = _aku_id2anns.get(_im["id"], [])
            if not _anns:
                continue
            _stem_aku = f"akudental_{_im['id']:05d}"
            _out_mask_aku = SEG_INST_MASK_DIR / f"{_stem_aku}.png"
            _out_img_aku  = SEG_INST_IMG_DIR  / f"{_stem_aku}.png"
            if _out_mask_aku.exists() and _out_img_aku.exists():
                _n_akudental += 1; continue
            _lbl_aku = np.zeros((_im["height"], _im["width"]), dtype=np.uint8)
            for _ann in _anns:
                _fdi = _aku_cat2fdi.get(int(_ann.get("category_id", 0)), 0)
                if _fdi == 0:
                    continue  # non-tooth category
                _cls_idx = _fdi_to_class_idx(_fdi)
                if _cls_idx == 0:
                    continue
                for _seg in (_ann.get("segmentation") or []):
                    if isinstance(_seg, list) and len(_seg) >= 6:
                        _pts_aku = np.array(_seg, np.int32).reshape(-1, 2)
                        cv2.fillPoly(_lbl_aku, [_pts_aku], _cls_idx)
            if _lbl_aku.max() == 0:
                continue
            _img_fname = _im.get("file_name", "")
            _img_path_aku = AKUDENTAL_IMG_DIR / _img_fname
            if not _img_path_aku.exists():
                continue
            _bgr_aku = cv2.imread(str(_img_path_aku))
            if _bgr_aku is None:
                continue
            cv2.imwrite(str(_out_img_aku),  cv2.resize(_bgr_aku, (512, 512)))
            cv2.imwrite(str(_out_mask_aku), cv2.resize(_lbl_aku, (512, 512), interpolation=cv2.INTER_NEAREST))
            _n_akudental += 1
        print(f"AKUDENTAL instance masks: {_n_akudental} images rendered")
    else:
        print(f"  AKUDENTAL: not found at {AKUDENTAL_JSON} (skipped)")

    # 4. Dataset 1 / New Supervisely-HITL (2392 images, same labeler as HITL)
    # Same Q1/Q3 reversal as existing HITL (same tool, same labeler GhazalehHITL)
    # Centroid diagnostic is printed first for verification; _remap_hitl() applied
    _n_d1hitl = 0
    if D1_ANN_DIR.exists() and D1_IMG_DIR.exists():
        # Centroid diagnostic (100-image sample to confirm Q1/Q3 orientation)
        import random as _rnd_d1
        _d1_jsons = sorted(D1_ANN_DIR.glob("*.json"))
        _d1_sample = _rnd_d1.sample(_d1_jsons, min(100, len(_d1_jsons)))
        _d1_cx = {c: [] for c in range(1, 33)}
        for _jp in _d1_sample:
            try:
                _d1ann = json.loads(_jp.read_text(encoding="utf-8"))
            except Exception:
                continue
            _d1W = _d1ann.get("size", {}).get("width", 1)
            for _obj in _d1ann.get("objects", []):
                try:
                    _c = int(_obj["classTitle"])
                except (ValueError, KeyError):
                    continue
                if 1 <= _c <= 32:
                    _ext = np.array(_obj["points"]["exterior"])
                    _d1_cx[_c].append(float(_ext[:, 0].mean()) / _d1W)
        print("Dataset 1 centroid diagnostic (sample ~100 images):")
        for _q_n, _q_r in [("Q1(1-8)", range(1,9)), ("Q2(9-16)", range(9,17)),
                            ("Q3(17-24)", range(17,25)), ("Q4(25-32)", range(25,33))]:
            _row = {c: f"{float(np.mean(_d1_cx[c])):.2f}" if _d1_cx[c] else "n/a"
                    for c in _q_r}
            print(f"  {_q_n}: {_row}")
        print("  Expected HITL-style: class1 cx~0.20 (far-left); expected DENTEX-style: class1 cx~0.49")
        print("  _remap_hitl() applied — same remap as existing HITL")

        # Rendering loop
        for _d1_ann_file in sorted(_d1_jsons):
            _d1_stem = _d1_ann_file.stem.replace(".jpg", "").replace(".png", "").replace(".jpeg", "")
            _out_mask_d1 = SEG_INST_MASK_DIR / f"d1hitl_{_d1_stem}.png"
            _out_img_d1  = SEG_INST_IMG_DIR  / f"d1hitl_{_d1_stem}.png"
            if _out_mask_d1.exists() and _out_img_d1.exists():
                _n_d1hitl += 1; continue
            try:
                _sly_d1 = json.loads(_d1_ann_file.read_text(encoding="utf-8"))
            except Exception:
                continue
            _h_d1 = _sly_d1.get("size", {}).get("height", 512)
            _w_d1 = _sly_d1.get("size", {}).get("width",  512)
            _lbl_d1 = np.zeros((_h_d1, _w_d1), dtype=np.uint8)
            for _obj_d1 in _sly_d1.get("objects", []):
                try:
                    _fdi_d1 = int(_obj_d1["classTitle"])
                except (ValueError, KeyError):
                    continue
                if not (1 <= _fdi_d1 <= 32):
                    continue
                if _obj_d1.get("geometryType") == "polygon":
                    _pts_d1 = np.array(_obj_d1["points"]["exterior"]).astype(np.int32)
                    cv2.fillPoly(_lbl_d1, [_pts_d1], int(_remap_hitl(_fdi_d1)))
            if _lbl_d1.max() == 0:
                continue
            # Find corresponding image
            _img_name_d1 = _d1_ann_file.stem  # e.g., "1.jpg" or "img_001.jpg"
            _img_path_d1 = D1_IMG_DIR / _img_name_d1
            if not _img_path_d1.exists():
                for _ext in [".jpg", ".png", ".jpeg", ".JPG"]:
                    _base = _img_name_d1.rsplit(".", 1)[0] if "." in _img_name_d1 else _img_name_d1
                    _try_d1 = D1_IMG_DIR / (_base + _ext)
                    if _try_d1.exists():
                        _img_path_d1 = _try_d1; break
            if not _img_path_d1.exists():
                continue
            _bgr_d1 = cv2.imread(str(_img_path_d1))
            if _bgr_d1 is None:
                continue
            _lbl_d1 = cv2.resize(_lbl_d1, (512, 512), interpolation=cv2.INTER_NEAREST)
            _bgr_d1 = cv2.resize(_bgr_d1, (512, 512))
            cv2.imwrite(str(_out_img_d1),  _bgr_d1)
            cv2.imwrite(str(_out_mask_d1), _lbl_d1)
            _n_d1hitl += 1
        print(f"Dataset 1 / New HITL instance masks: {_n_d1hitl} images rendered")
    else:
        print(f"  Dataset 1: not found at {D1_ANN_DIR} (skipped)")

    # 5. OdontoAI (2,000 images, FDI COCO polygons)
    # Download: https://github.com/rezaakb/odontoai
    # Place at: DATA/OdontoAI/  with  images/ and annotations.json
    _oai_json = ODONTOAI_DIR / "annotations.json"
    _oai_imgs = ODONTOAI_DIR / "images"
    _n_oai = 0
    if _oai_json.exists() and _oai_imgs.exists():
        with open(_oai_json, encoding="utf-8") as _f:
            _oai = json.load(_f)
        _oai_id2path = {im["id"]: _oai_imgs / im["file_name"] for im in _oai["images"]}
        _oai_id2anns = {}
        for _ann in _oai.get("annotations", []):
            _oai_id2anns.setdefault(_ann["image_id"], []).append(_ann)
        for _im in _oai["images"]:
            _p = _oai_id2path[_im["id"]]
            if not _p.exists() or not _oai_id2anns.get(_im["id"]):
                continue
            _lbl = np.zeros((_im["height"], _im["width"]), dtype=np.uint8)
            for _ann in _oai_id2anns[_im["id"]]:
                # OdontoAI category_id maps to FDI: cat 1=FDI11...cat32=FDI48
                _cat = int(_ann.get("category_id", 0))
                if not (1 <= _cat <= 32):
                    continue
                for _seg in (_ann.get("segmentation") or []):
                    if isinstance(_seg, list) and len(_seg) >= 6:
                        cv2.fillPoly(_lbl, [np.array(_seg, np.int32).reshape(-1,1,2)], _cat)
            if _lbl.max() == 0:
                continue
            _stem = f"oai_{_im['id']:06d}"
            _bgr = cv2.imread(str(_p))
            if _bgr is None:
                continue
            _bgr512 = cv2.resize(_bgr, (512, 512))
            _lbl512 = cv2.resize(_lbl, (512, 512), interpolation=cv2.INTER_NEAREST)
            cv2.imwrite(str(SEG_INST_IMG_DIR / f"{_stem}.png"), _bgr512)
            cv2.imwrite(str(SEG_INST_MASK_DIR / f"{_stem}.png"), _lbl512)
            _n_oai += 1
        print(f"  OdontoAI: {_n_oai} images rendered")
    else:
        print(f"  OdontoAI: not found at {ODONTOAI_DIR} (optional — download to add 2K images)")

    # 6. DNS Panoramic (543 images, FDI COCO polygons)
    # Download: place at DATA/DNS_Panoramic/ with images/ and annotations.json
    _dns_json = DNS_PANO_DIR / "annotations.json"
    _dns_imgs = DNS_PANO_DIR / "images"
    _n_dns = 0
    if _dns_json.exists() and _dns_imgs.exists():
        with open(_dns_json, encoding="utf-8") as _f:
            _dns = json.load(_f)
        _dns_id2path = {im["id"]: _dns_imgs / im["file_name"] for im in _dns["images"]}
        _dns_id2anns = {}
        for _ann in _dns.get("annotations", []):
            _dns_id2anns.setdefault(_ann["image_id"], []).append(_ann)
        for _im in _dns["images"]:
            _p = _dns_id2path[_im["id"]]
            if not _p.exists() or not _dns_id2anns.get(_im["id"]):
                continue
            _lbl = np.zeros((_im["height"], _im["width"]), dtype=np.uint8)
            for _ann in _dns_id2anns[_im["id"]]:
                _cat = int(_ann.get("category_id", 0))
                if not (1 <= _cat <= 32):
                    continue
                for _seg in (_ann.get("segmentation") or []):
                    if isinstance(_seg, list) and len(_seg) >= 6:
                        cv2.fillPoly(_lbl, [np.array(_seg, np.int32).reshape(-1,1,2)], _cat)
            if _lbl.max() == 0:
                continue
            _stem = f"dns_{_im['id']:06d}"
            _bgr = cv2.imread(str(_p))
            if _bgr is None:
                continue
            cv2.imwrite(str(SEG_INST_IMG_DIR / f"{_stem}.png"), cv2.resize(_bgr,(512,512)))
            cv2.imwrite(str(SEG_INST_MASK_DIR / f"{_stem}.png"),
                        cv2.resize(_lbl,(512,512),interpolation=cv2.INTER_NEAREST))
            _n_dns += 1
        print(f"  DNS Panoramic: {_n_dns} images rendered")
    else:
        print(f"  DNS Panoramic: not found at {DNS_PANO_DIR} (optional)")

    print(f"Total instance seg data: {_dentex_count + _hitl_count + _n_akudental + _n_d1hitl} images")
    _inst_sentinel.touch()
    print("Instance mask rendering complete.")
else:
    _n_inst = len(list(SEG_INST_IMG_DIR.glob("*.png"))) if SEG_INST_IMG_DIR.exists() else 0
    print(f"Instance masks already rendered: {_n_inst} images")

In [ ]:
# PHASE B — YOLO Data Preparation (v22)
# Converts PNG label maps (0=bg, 1-32=FDI class) to YOLO segmentation format.
# Each tooth region becomes one polygon annotation with 0-indexed class.
# Run once; sentinel .yolo_phase_b_data_done guards re-runs.
import cv2, numpy as np, shutil as _shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

_YOLO_IMGS_TR = YOLO_DIR / "images" / "train"
_YOLO_IMGS_VL = YOLO_DIR / "images" / "val"
_YOLO_LBLS_TR = YOLO_DIR / "labels" / "train"
_YOLO_LBLS_VL = YOLO_DIR / "labels" / "val"
for _d in [_YOLO_IMGS_TR, _YOLO_IMGS_VL, _YOLO_LBLS_TR, _YOLO_LBLS_VL]:
    _d.mkdir(parents=True, exist_ok=True)

_YOLO_SENTINEL = OUT / ".yolo_phase_b_data_done"

# Auto-invalidate YOLO sentinel if expert/student data leaked into labels (bug fix v22)
if _YOLO_SENTINEL.exists():
    _expert_in_yolo = list((_YOLO_LBLS_TR if '_YOLO_LBLS_TR' in dir() else (YOLO_DIR / "labels" / "train")).glob("expert_*.txt"))
    _student_in_yolo = list((_YOLO_LBLS_TR if '_YOLO_LBLS_TR' in dir() else (YOLO_DIR / "labels" / "train")).glob("student_*.txt"))
    if _expert_in_yolo or _student_in_yolo:
        _YOLO_SENTINEL.unlink()
        print(f"YOLO sentinel removed: found {len(_expert_in_yolo)} expert + {len(_student_in_yolo)} student labels (wrong class IDs). Rebuilding YOLO data...")
    else:
        _n_tr_check = len(list((YOLO_DIR / "images" / "train").glob("*.png")))
        _n_vl_check = len(list((YOLO_DIR / "images" / "val").glob("*.png")))
        print(f"YOLO data ready  train={_n_tr_check}  val={_n_vl_check}  (sentinel exists — skipping)")

if _YOLO_SENTINEL.exists():
    pass  # already printed above
else:
    # FDI class names (0-indexed for YOLO; label value v → class v-1)
    # Label 1 = Q1T1 = FDI 11 … label 32 = Q4T8 = FDI 48
    def _lbl_to_fdi(v):
        q, t = (v - 1) // 8 + 1, (v - 1) % 8 + 1
        return f"FDI_{q * 10 + t}"
    _FDI_NAMES = {v - 1: _lbl_to_fdi(v) for v in range(1, 33)}

    # Filter out expert/student masks — they use sequential per-image IDs (not FDI)
    # Bug fix v22: expert_* and student_* were incorrectly included from older runs
    _EXCLUDED_PREFIXES = ("expert_", "student_")
    _all_stems = sorted(set(
        p.stem for p in SEG_INST_MASK_DIR.glob("*.png")
        if not any(p.stem.startswith(pfx) for pfx in _EXCLUDED_PREFIXES)
    ))
    print(f"Total label maps found: {len(_all_stems)}")

    def _source_of(s):
        for pfx in ("dentex_", "akudental_", "d1hitl_", "ext_hitl_"):
            if s.startswith(pfx):
                return pfx
        return "other_"

    _sources = [_source_of(s) for s in _all_stems]
    _sc = {}
    for _s in _sources:
        _sc[_s] = _sc.get(_s, 0) + 1
    print("Sources:", {k: v for k, v in sorted(_sc.items())})

    _tr_stems, _vl_stems = train_test_split(
        _all_stems, test_size=0.15, random_state=42, stratify=_sources
    )
    print(f"Split  train={len(_tr_stems)}  val={len(_vl_stems)}")

    def _mask_to_yolo(mask_path):
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask is None:
            return []
        # Handle (H,W,1) or (H,W,C) — some PNG writers add channel dim
        if mask.ndim == 3:
            mask = mask[:, :, 0]
        if mask.ndim != 2:
            return []
        if mask.dtype != np.uint8:
            mask = mask.astype(np.uint8)
        H, W = mask.shape
        lines = []
        for cls_val in range(1, 33):
            yolo_cls = cls_val - 1
            cm = (mask == cls_val).astype(np.uint8)
            if cm.sum() < 50:
                continue
            cnts, _ = cv2.findContours(cm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in cnts:
                if cv2.contourArea(cnt) < 50:
                    continue
                eps = 0.003 * cv2.arcLength(cnt, True)
                approx = cv2.approxPolyDP(cnt, eps, True)
                if len(approx) < 3:
                    continue
                pts = approx.reshape(-1, 2).astype(float)
                pts[:, 0] /= W
                pts[:, 1] /= H
                pts = np.clip(pts, 0.0, 1.0)
                coords = " ".join(f"{x:.5f} {y:.5f}" for x, y in pts)
                lines.append(f"{yolo_cls} {coords}")
        return lines

    # Clear old images/labels directories before rebuild to remove stale expert/student files
    import shutil as _shutil_clean
    for _split_clean in ["train", "val"]:
        for _sub in [YOLO_DIR / "images" / _split_clean, YOLO_DIR / "labels" / _split_clean]:
            if _sub.exists():
                _shutil_clean.rmtree(str(_sub))
            _sub.mkdir(parents=True, exist_ok=True)
    # Also remove stale .cache files
    for _cache in (YOLO_DIR / "labels").glob("*.cache"):
        _cache.unlink()
    print("  Cleared old images/labels directories")

    _n_done = 0
    for _split, _stems in [("train", _tr_stems), ("val", _vl_stems)]:
        _id = YOLO_DIR / "images" / _split
        _ld = YOLO_DIR / "labels" / _split
        for _stem in _stems:
            _mp = SEG_INST_MASK_DIR / f"{_stem}.png"
            _ip = SEG_INST_IMG_DIR  / f"{_stem}.png"
            if not _mp.exists() or not _ip.exists():
                continue
            _shutil.copy(_ip, _id / f"{_stem}.png")
            _lines = _mask_to_yolo(_mp)
            (_ld / f"{_stem}.txt").write_text("\n".join(_lines))
            _n_done += 1
            if _n_done % 500 == 0:
                print(f"  Converted {_n_done}/{len(_tr_stems)+len(_vl_stems)}...")

    _names_block = "\n".join(f"  {k}: '{v}'" for k, v in sorted(_FDI_NAMES.items()))
    _yaml = (
        f"path: {YOLO_DIR.as_posix()}\n"
        "train: images/train\n"
        "val: images/val\n\n"
        "nc: 32\n"
        f"names:\n{_names_block}\n"
    )
    (YOLO_DIR / "teeth.yaml").write_text(_yaml)
    _YOLO_SENTINEL.touch()

    _n_tr = len(list(_YOLO_IMGS_TR.glob("*.png")))
    _n_vl = len(list(_YOLO_IMGS_VL.glob("*.png")))
    print(f"\nDone  train={_n_tr}  val={_n_vl}")
    print(f"YAML: {YOLO_DIR / 'teeth.yaml'}")


In [ ]:
# PHASE B — YOLOv8x-seg Training (v22)
# Architecture: YOLOv8x-seg (68M params, COCO-pretrained transfer learning)
# Based on DENTEX 2023 SOTA: YOLOrtho = YOLOv8 + CoordConv + Hungarian matching
# STS 2024 winner (ChohoTech): YOLOv8 + SAM, Instance Dice=0.836
import shutil as _shutil2

try:
    from ultralytics import YOLO as _YOLO_CLS
    _ult_ok = True
    print("ultralytics available ✓")
except ImportError:
    _ult_ok = False
    print("ERROR: ultralytics not installed. Run: pip install ultralytics")

_YOLO_RUN_DIR = YOLO_DIR / "runs" / "phase_b_v23"
_YOLO_BEST_PT  = _YOLO_RUN_DIR / "weights" / "best.pt"

# v22 fix: delete old checkpoint if YOLO data was rebuilt (expert/student bug was fixed)
_yolo_retrain_flag = OUT / ".yolo_retrain_v22"
if not _yolo_retrain_flag.exists() and SEG_INST_CKPT.exists():
    import shutil as _shu
    _shu.move(str(SEG_INST_CKPT), str(SEG_INST_CKPT).replace(".pt", "_bad_v22.pt"))
    _yolo_retrain_flag.touch()
    print(f"Old checkpoint moved to *_bad_v22.pt — retraining with fixed data (no fliplr, no expert/student)")

if SEG_INST_CKPT.exists():
    print(f"Phase B checkpoint exists: {SEG_INST_CKPT} — skipping training")
    _yolo_model = _YOLO_CLS(str(SEG_INST_CKPT)) if _ult_ok else None
elif not _ult_ok:
    print("Phase B skipped — install ultralytics first")
    _yolo_model = None
else:
    print("Phase B: Training YOLOv8x-seg — FDI tooth instance segmentation")
    print(f"  Model   : YOLOv8x-seg (COCO-pretrained, 68M params)")
    print(f"  Classes : 32 (FDI 11-48)")
    print(f"  Data    : {YOLO_DIR / 'teeth.yaml'}")
    print(f"  Epochs  : 200  imgsz=512  batch=16  close_mosaic=20")

    _m = _YOLO_CLS("yolov8x-seg.pt")
    _m.train(
        data=str(YOLO_DIR / "teeth.yaml"),
        epochs=200,
        imgsz=512,
        batch=16,
        device=0,
        workers=0,          # Windows: no multiprocessing
        patience=50,
        cos_lr=True,
        warmup_epochs=3,
        degrees=10.0,
        fliplr=0.0,         # FDI left/right is anatomically meaningful — NO horizontal flip
        flipud=0.0,
        mosaic=0.5,
        close_mosaic=20,
        mixup=0.1,
        copy_paste=0.0,      # copy-paste confuses FDI spatial positioning
        amp=True,           # BF16/FP16 mixed precision
        project=str(_YOLO_RUN_DIR.parent),
        name=_YOLO_RUN_DIR.name,
        exist_ok=True,
        verbose=True,
    )
    if _YOLO_BEST_PT.exists():
        _shutil2.copy(_YOLO_BEST_PT, SEG_INST_CKPT)
        print(f"\nBest checkpoint saved: {SEG_INST_CKPT}")
    else:
        print(f"WARNING: best.pt not found at {_YOLO_BEST_PT}")
    _yolo_model = _YOLO_CLS(str(SEG_INST_CKPT)) if SEG_INST_CKPT.exists() else None


# YoloInstWrapper: lets Cell 31 (visualization) use inst_model unchanged
import cv2 as _cv2_wrap
import numpy as _np_wrap

class YoloInstWrapper:
    '''Wraps YOLOv8-seg to match SegUNetDINO_Instance interface.
    Input:  imgs (B, 3, H, W) float tensor, ImageNet-normalized
    Output: (B, 33, H, W) float tensor  ch0=bg, ch1-32=FDI class 1-32
    '''
    def __init__(self, yolo_model):
        self._yolo = yolo_model

    def eval(self):
        return self

    def parameters(self):
        return iter([])

    def __call__(self, imgs, binary_mask=None, return_aux=False):
        if self._yolo is None:
            B, _, H, W = imgs.shape
            return torch.zeros(B, 33, H, W, device=imgs.device)
        _MEAN = torch.tensor([0.485, 0.456, 0.406], device=imgs.device).view(1, 3, 1, 1)
        _STD  = torch.tensor([0.229, 0.224, 0.225], device=imgs.device).view(1, 3, 1, 1)
        imgs_01 = (imgs * _STD + _MEAN).clamp(0, 1)
        B, C, H, W = imgs.shape
        out = torch.zeros(B, 33, H, W, device=imgs.device)
        for b in range(B):
            img_np = (imgs_01[b].permute(1, 2, 0).cpu().numpy() * 255).astype(_np_wrap.uint8)
            res = self._yolo(img_np, imgsz=max(H, W), verbose=False, conf=0.25)[0]
            if res.masks is None or len(res.masks) == 0:
                continue
            masks = res.masks.data.cpu().numpy()
            classes = res.boxes.cls.cpu().numpy().astype(int)
            confs   = res.boxes.conf.cpu().numpy()
            if masks.shape[1] != H or masks.shape[2] != W:
                tmp = _np_wrap.zeros((len(masks), H, W), dtype=_np_wrap.float32)
                for ii, mm in enumerate(masks):
                    tmp[ii] = _cv2_wrap.resize(mm.astype(_np_wrap.float32), (W, H))
                masks = tmp
            seen = {}
            for ii, (cc, cf) in enumerate(zip(classes, confs)):
                if cc not in seen or cf > seen[cc][1]:
                    seen[cc] = (ii, float(cf))
            for ii, _ in sorted(seen.values(), key=lambda x: x[1]):
                cc = int(classes[ii]); m = masks[ii] > 0.5
                ch = cc + 1
                if 1 <= ch <= 32:
                    out[b, ch][torch.from_numpy(m).to(imgs.device)]  =  5.0
                    out[b,  0][torch.from_numpy(m).to(imgs.device)]  = -5.0
        if return_aux:
            return out, None
        return out


inst_model = YoloInstWrapper(_yolo_model)
print("inst_model = YoloInstWrapper (YOLO-backed, Cell 31 visualization compatible)")


In [ ]:
# PHASE B — YOLOv8-seg Evaluation (v22)
# Metrics: per-class IoU, per-quadrant mIoU, overall mIoU
# Post-processing: Hungarian matching for unique FDI assignment per patient
import numpy as np
from scipy.optimize import linear_sum_assignment
import cv2

def _yolo_to_fdi_map(yolo_model, img_path, out_size=512, conf_thresh=0.25):
    '''Run YOLOv8-seg on one image; return (out_size, out_size) uint8 class map.
    Values: 0=bg, 1-32=FDI class.  Unique class assignment via Hungarian.'''
    res = yolo_model(str(img_path), imgsz=out_size, verbose=False, conf=conf_thresh)[0]
    out = np.zeros((out_size, out_size), dtype=np.uint8)
    if res.masks is None or len(res.masks) == 0:
        return out
    masks   = res.masks.data.cpu().numpy()      # (N, H, W)
    classes = res.boxes.cls.cpu().numpy().astype(int)
    confs   = res.boxes.conf.cpu().numpy()
    H, W    = masks.shape[1], masks.shape[2]
    if H != out_size or W != out_size:
        tmp = np.zeros((len(masks), out_size, out_size), dtype=np.float32)
        for ii, mm in enumerate(masks):
            tmp[ii] = cv2.resize(mm.astype(np.float32), (out_size, out_size))
        masks = tmp
    # Hungarian: enforce at most one detection per FDI class
    seen = {}
    for ii, (cc, cf) in enumerate(zip(classes, confs)):
        if cc not in seen or cf > seen[cc][1]:
            seen[cc] = (ii, float(cf))
    # Fill map — low-confidence instances first (overwritten by high-conf)
    for ii, _ in sorted(seen.values(), key=lambda x: x[1]):
        cc = int(classes[ii]); m = masks[ii] > 0.5
        ch = cc + 1  # 0-indexed → 1-indexed
        if 1 <= ch <= 32:
            out[m] = ch
    return out

if not SEG_INST_CKPT.exists():
    print(f"Phase B checkpoint not found: {SEG_INST_CKPT}")
    print("Run Cell 29 first to train the YOLOv8x-seg model.")
else:
    from ultralytics import YOLO as _YOLO_EVAL
    _eval_model = _YOLO_EVAL(str(SEG_INST_CKPT))

    _val_imgs = sorted((YOLO_DIR / "images" / "val").glob("*.png"))
    print(f"Evaluating on {len(_val_imgs)} validation images...")

    _iou_per_cls = {c: [] for c in range(1, 33)}
    _n_eval = 0

    for _ip in _val_imgs[:300]:   # up to 300 samples
        _gp = SEG_INST_MASK_DIR / _ip.name
        if not _gp.exists():
            continue
        _gt = cv2.imread(str(_gp), cv2.IMREAD_UNCHANGED)
        if _gt is None:
            continue
        if _gt.ndim == 3: _gt = _gt[:, :, 0]  # guard: some PNGs return (H,W,1)
        if _gt.ndim != 2: continue
        if _gt.dtype != np.uint8: _gt = _gt.astype(np.uint8)
        if _gt.shape[0] != 512 or _gt.shape[1] != 512:
            _gt = cv2.resize(_gt, (512, 512), interpolation=cv2.INTER_NEAREST)

        _pred = _yolo_to_fdi_map(_eval_model, _ip)

        for _c in range(1, 33):
            _pc = (_pred == _c)
            _gc = (_gt == _c)
            if not _gc.any():
                continue
            _inter = (_pc & _gc).sum()
            _union = (_pc | _gc).sum()
            if _union > 0:
                _iou_per_cls[_c].append(_inter / _union)

        _n_eval += 1
        if _n_eval % 50 == 0:
            _partial = np.mean([np.mean(v) for v in _iou_per_cls.values() if v])
            print(f"  {_n_eval}/{min(300, len(_val_imgs))}  running mIoU={_partial:.4f}")

    _cls_ious = {c: float(np.mean(v)) for c, v in _iou_per_cls.items() if v}
    _miou_all = float(np.mean(list(_cls_ious.values()))) if _cls_ious else 0.0

    print(f"\n{'='*60}")
    print(f"Phase B YOLOv8x-seg — {_n_eval} val images")
    print(f"{'='*60}")
    print(f"Overall mIoU (all FDI classes): {_miou_all:.4f}")

    for _qn, _qr in [("Q1 (FDI 11-18)", range(1, 9)),
                      ("Q2 (FDI 21-28)", range(9, 17)),
                      ("Q3 (FDI 31-38)", range(17, 25)),
                      ("Q4 (FDI 41-48)", range(25, 33))]:
        _q_ious = [_cls_ious[c] for c in _qr if c in _cls_ious]
        if _q_ious:
            print(f"  {_qn}: {np.mean(_q_ious):.4f}")

    print(f"\nClasses with 0 detections: "
          f"{[c for c in range(1,33) if c not in _cls_ious]}")

    # Make inst_model available for downstream cells
    inst_model = YoloInstWrapper(_eval_model)
    print("\ninst_model updated with loaded eval model")


In [ ]:
# PHASE B — Visualization: Prediction vs Ground Truth
# Shows 8 random validation samples with colorized FDI class maps.
# Color scheme: each of 32 FDI classes gets a unique color; background = black.
import matplotlib.pyplot as _plt
import matplotlib.patches as _mpatches
import numpy as _np
import torch

# Color palette: BG=black, Q1=blue, Q2=green, Q3=red, Q4=purple
def _make_palette():
    p = _np.zeros((33, 3), dtype=_np.uint8)
    # Q1 (1-8): blue family
    for i, v in enumerate([0.55,0.62,0.68,0.73,0.77,0.81,0.85,0.89]):
        p[i+1] = [int(v*50), int(v*120), int(v*255)]
    # Q2 (9-16): green family
    for i, v in enumerate([0.55,0.62,0.68,0.73,0.77,0.81,0.85,0.89]):
        p[i+9] = [int(v*50), int(v*220), int(v*100)]
    # Q3 (17-24): red/orange family
    for i, v in enumerate([0.55,0.62,0.68,0.73,0.77,0.81,0.85,0.89]):
        p[i+17] = [int(v*255), int(v*80), int(v*50)]
    # Q4 (25-32): purple/pink family
    for i, v in enumerate([0.55,0.62,0.68,0.73,0.77,0.81,0.85,0.89]):
        p[i+25] = [int(v*210), int(v*80), int(v*220)]
    return p

_PAL = _make_palette()

def _colorize(label_map):
    "label_map: (H,W) int array 0-32 → (H,W,3) RGB"
    return _PAL[label_map.clip(0, 32)]

def _overlay(img_t, label_map, alpha=0.55):
    "Blend normalized img tensor (C,H,W) with colorized label map"
    _mean = _np.array([0.485,0.456,0.406]); _std = _np.array([0.229,0.224,0.225])
    img_np = img_t.permute(1,2,0).cpu().numpy()
    img_np = (img_np * _std + _mean).clip(0,1)
    if img_np.shape[2] == 1: img_np = _np.repeat(img_np, 3, 2)
    color = _colorize(label_map).astype(_np.float32) / 255.0
    mask  = (label_map > 0)[..., None]
    blend = img_np * (1 - alpha * mask) + color * alpha * mask
    return blend.clip(0, 1)

# Sample 8 images from validation set
# No inst_vl_ld (YOLO pipeline). Load images + GT directly from disk.
import random as _rnd_viz; _rnd_viz.seed(42)
import cv2 as _cv2_viz

_val_imgs_viz = sorted((YOLO_DIR / "images" / "val").glob("*.png"))
_viz_paths = _rnd_viz.sample(_val_imgs_viz, min(4, len(_val_imgs_viz)))

# Use _eval_model from Cell 30 if available, else load fresh
if "_eval_model" not in dir():
    from ultralytics import YOLO as _YOLO_VIZ
    _eval_model = _YOLO_VIZ(str(SEG_INST_CKPT))

_IMG_MEAN_V = _np.array([0.485, 0.456, 0.406])
_IMG_STD_V  = _np.array([0.229, 0.224, 0.225])

_samples = []
for _vp in _viz_paths:
    _gp = SEG_INST_MASK_DIR / _vp.name
    if not _gp.exists():
        continue
    # Raw image → normalized tensor (C,H,W) for _overlay
    _raw = _cv2_viz.imread(str(_vp))
    if _raw is None: continue
    _raw = _cv2_viz.cvtColor(_raw, _cv2_viz.COLOR_BGR2RGB)
    _raw = _cv2_viz.resize(_raw, (512, 512))
    _raw_f = _raw.astype(_np.float32) / 255.0
    _img_t = torch.tensor(((_raw_f - _IMG_MEAN_V) / _IMG_STD_V).transpose(2, 0, 1),
                           dtype=torch.float32)  # (3,512,512)
    # GT mask
    _gt = _cv2_viz.imread(str(_gp), _cv2_viz.IMREAD_UNCHANGED)
    if _gt is None: continue
    if _gt.ndim == 3: _gt = _gt[:, :, 0]
    if _gt.shape[0] != 512 or _gt.shape[1] != 512:
        _gt = _cv2_viz.resize(_gt, (512, 512), interpolation=_cv2_viz.INTER_NEAREST)
    _gt_np = _gt.astype(_np.int64)
    # Prediction via YOLO
    _pred_np = _yolo_to_fdi_map(_eval_model, _vp).astype(_np.int64)
    _samples.append((_img_t, _pred_np, _gt_np))
    if len(_samples) >= 8: break

_n = min(8, len(_samples))
_fig, _axes = _plt.subplots(_n, 3, figsize=(18, 6 * _n))
if _n == 1: _axes = [_axes]

# Column headers (only on first row)
_col_headers = ["Input X-ray", "Ground Truth", "YOLO Prediction"]
for _ci, _hdr in enumerate(_col_headers):
    _axes[0, _ci].set_title(_hdr, fontsize=13, fontweight="bold", pad=6)

for _i, (_img, _pred_np, _gt_np) in enumerate(_samples[:_n]):
    _ax0, _ax1, _ax2 = _axes[_i]

    # Column 0: raw X-ray
    _raw_disp = (_img.permute(1,2,0).numpy() * _IMG_STD_V + _IMG_MEAN_V).clip(0,1)
    _ax0.imshow(_raw_disp, cmap="gray")
    _ax0.set_ylabel(f"Sample {_i+1}", fontsize=11, rotation=90, labelpad=8)
    _ax0.axis("off")

    # Column 1: GT overlay
    _n_gt_cls = len(_np.unique(_gt_np[_gt_np > 0]))
    _ax1.imshow(_overlay(_img, _gt_np, alpha=0.6))
    _ax1.set_title(f"{_n_gt_cls} GT classes", fontsize=10, pad=4)
    _ax1.axis("off")

    # Column 2: prediction overlay
    _n_cls = len(_np.unique(_pred_np[_pred_np > 0]))
    _ax2.imshow(_overlay(_img, _pred_np, alpha=0.6))
    _ax2.set_title(f"{_n_cls} predicted classes", fontsize=10, pad=4)
    _ax2.axis("off")

# Legend
_legend_patches = []
_qnames = {1:"Q1(11-18)",9:"Q2(21-28)",17:"Q3(31-38)",25:"Q4(41-48)"}
for _start, _qname in _qnames.items():
    _c = _PAL[_start].astype(float)/255
    _legend_patches.append(_mpatches.Patch(color=_c, label=_qname))
_fig.legend(handles=_legend_patches, loc="lower center", ncol=4, fontsize=11,
            bbox_to_anchor=(0.5, 0.0), frameon=True)
_plt.suptitle("Phase B — FDI Instance Segmentation (4 val samples)", fontsize=14, fontweight="bold", y=1.01)
_plt.tight_layout(rect=[0, 0.04, 1, 1])
_plt.savefig(str(VIS_DIR / "phase_b_predictions.png"), dpi=120, bbox_inches="tight")
_plt.show()
print(f"Saved: {VIS_DIR / 'phase_b_predictions.png'}")


## Phase C — Multi-Task Learning

DentalMTLNet: shared encoder for simultaneous segmentation and disease classification.

In [ ]:
class DentalMTLNet(nn.Module):
    def __init__(self, n_cls=4, ssl_path=None, convnext_encoder=None):
        super().__init__()
        self._is_convnext = convnext_encoder is not None
        if self._is_convnext:
            self.encoder = convnext_encoder
            # ConvNeXt-B channels: e0=128, e1=256, e2=512, e3=1024 (no e4)
            self.aspp = ASPP(1024, 256)
            self.d4 = _Up(512, 256, 256)
            self.d3 = _Up(256, 256, 128)
            self.d2 = _Up(128, 128, 64)
            self.d1 = nn.Sequential(nn.ConvTranspose2d(64,32,2,stride=2),_DoubleConv(32,32))
            self.up_f = nn.ConvTranspose2d(32, 16, 2, stride=2)
            self.seg_out = nn.Conv2d(16, 1, 1)
            self.cls_gap = nn.AdaptiveAvgPool2d(1)
            self.cls_head = nn.Sequential(nn.Dropout(0.4),nn.Linear(1024,256),
                                          nn.GELU(),nn.Dropout(0.3),nn.Linear(256,n_cls))
            print("DentalMTLNet: using ConvNeXt-B encoder")
        else:
            ssl_path=Path(ssl_path) if ssl_path else BACKBONE_SSL
            if not ssl_path.exists(): raise RuntimeError(f"SSL not found: {ssl_path}")
            bb=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            sd=torch.load(ssl_path,map_location="cpu",weights_only=True)
            if next(iter(sd),"").split(".")[0].isdigit():
                _m={"0":"conv1","1":"bn1","4":"layer1","5":"layer2","6":"layer3","7":"layer4"}
                sd={".".join([_m.get(k.split(".")[0],k.split(".")[0])]+k.split(".")[1:]):v for k,v in sd.items() if k.split(".")[0] in _m}
            miss,_=bb.load_state_dict(sd,strict=False)
            print(f"  MTL SSL: {len(bb.state_dict())-len(miss)}/{len(bb.state_dict())} model params")
            self.e0=nn.Sequential(bb.conv1,bb.bn1,bb.relu); self.mp=bb.maxpool
            self.e1=bb.layer1; self.e2=bb.layer2
            self.e3=bb.layer3   # 1024ch — cls branches here (v6 fix: was layer4)
            self.e4=bb.layer4
            self.aspp=ASPP(2048,256)
            self.d4=_Up(1024,256,512); self.d3=_Up(512,512,256)
            self.d2=_Up(256,256,128); self.d1=_Up(64,128,64)
            self.up_f=nn.ConvTranspose2d(64,32,2,stride=2); self.seg_out=nn.Conv2d(32,1,1)
            self.cls_gap=nn.AdaptiveAvgPool2d(1)
            self.cls_head=nn.Sequential(nn.Dropout(0.4),nn.Linear(1024,256),
                                         nn.GELU(),nn.Dropout(0.3),nn.Linear(256,n_cls))

    def forward(self,x):
        if self._is_convnext:
            e0,e1,e2,e3=self.encoder(x)
            btn=self.aspp(e3)
            d=self.d4(btn,e2); d=self.d3(d,e1); d=self.d2(d,e0)
            d=self.d1(d)
            seg=self.seg_out(self.up_f(d))
            cls=self.cls_head(self.cls_gap(e3).flatten(1))  # from ConvNeXt stage3 (1024ch)
        else:
            s0=self.e0(x); s1=self.e1(self.mp(s0)); s2=self.e2(s1); s3=self.e3(s2); s4=self.e4(s3)
            btn=self.aspp(s4)
            d=self.d4(btn,s3); d=self.d3(d,s2); d=self.d2(d,s1); d=self.d1(d,s0)
            seg=self.seg_out(self.up_f(d))
            cls=self.cls_head(self.cls_gap(s3).flatten(1))  # from layer3
        return seg,cls

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,ls=0.05):
        super().__init__(); self.gamma=gamma; self.weight=weight; self.ls=ls
    def forward(self,pred,tgt):
        ce=F.cross_entropy(pred,tgt,weight=self.weight,label_smoothing=self.ls,reduction="none")
        return ((1-torch.exp(-ce))**self.gamma*ce).mean()

class GradNormBalancer(nn.Module):
    def __init__(self,n=2,alpha=2.0):
        super().__init__()
        self.alpha=alpha; self.weights=nn.Parameter(torch.ones(n))
        self.register_buffer("init_losses",torch.zeros(n)); self._init=False

    @property
    def norm_w(self):
        w=torch.clamp(self.weights,min=1e-4); return w*(len(w)/w.sum())

    def gn_loss(self,shared_params,task_losses):
        if not self._init:
            self.init_losses=torch.tensor([l.item() for l in task_losses],
                                           device=self.weights.device); self._init=True
        w=self.norm_w; gns=[]
        for i,loss in enumerate(task_losses):
            gs=[g for g in torch.autograd.grad(
                w[i]*loss,shared_params,retain_graph=True,create_graph=True,allow_unused=True)
                if g is not None]
            gns.append(torch.stack([g.norm() for g in gs]).mean() if gs
                        else torch.zeros(1,device=self.weights.device))
        gn=torch.stack(gns); mean_gn=gn.mean().detach()
        ratios=torch.stack([l.detach()/(self.init_losses[i]+1e-8) for i,l in enumerate(task_losses)])
        targets=(mean_gn*(ratios.mean()/(ratios+1e-8))**self.alpha).detach()
        return torch.abs(gn-targets).sum()

print("DentalMTLNet + GradNormBalancer defined ✓")


In [ ]:

class MTLDataset(Dataset):
    def __init__(self,imgs,masks,lmap,aug=False):
        self.imgs=imgs; self.masks=masks; self.aug=aug
        self.lbls=[lmap.get(p.stem,-1) for p in imgs]
        self.ttf=A.Compose([A.RandomResizedCrop(size=(SEG_SIZE,SEG_SIZE),scale=(0.7,1.0)),
                             A.HorizontalFlip(p=0.5),
                             A.Affine(translate_percent={"x":(-0.1,0.1),"y":(-0.1,0.1)},
                                      scale=(0.85,1.15),rotate=(-15,15),p=0.5),
                             A.RandomBrightnessContrast(0.2,0.2,p=0.4),
                             A.ElasticTransform(alpha=80,sigma=5,p=0.15),
                             A.CoarseDropout(num_holes_range=(1,4), hole_height_range=(8,32), hole_width_range=(8,32), p=0.2),
                             A.GaussNoise(std_range=(0.012,0.028),p=0.3),
                             A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()
                            ],additional_targets={"mask":"mask"})
        self.vtf=A.Compose([A.Resize(SEG_SIZE,SEG_SIZE),
                             A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()
                            ],additional_targets={"mask":"mask"})
    def __len__(self): return len(self.imgs)
    def __getitem__(self,idx):
        img=np.array(Image.open(self.imgs[idx]).convert("RGB"))
        img=apply_clahe(img)  # X-ray contrast -- same as SegDataset
        mask=(np.array(Image.open(self.masks[idx]).convert("L"))>127).astype(np.float32)
        out=(self.ttf if self.aug else self.vtf)(image=img,mask=mask)
        return out["image"],out["mask"].unsqueeze(0),torch.tensor(self.lbls[idx])

# Build label map from patch CSV
# v9 fix: patch_labels_v9.csv dropped image_id column; fall back to v4 CSV which has it
lmap={}
_lmap_csv = (OUT/"patch_labels_v4.csv") if (OUT/"patch_labels_v4.csv").exists() else PATCH_CSV
if _lmap_csv.exists() and "dis_data" in dir():
    df_p=pd.read_csv(_lmap_csv)
    dis_imap={img["id"]:img for img in dis_data["images"]}
    if "image_id" in df_p.columns:
        for iid,grp in df_p.groupby("image_id"):
            maj=grp["class_id"].mode()[0]
            info=dis_imap.get(iid)
            if info: lmap[Path(info["file_name"]).stem]={3:1}.get(int(maj),int(maj))  # DeepCaries->Caries
    print(f"MTL lmap: {len(lmap)} labeled images (source: {_lmap_csv.name})")
else:
    print("WARNING: MTL lmap is empty — cls head will receive no labeled samples")

mtl_tr=MTLDataset(tr_imgs,tr_masks,lmap,aug=True)
mtl_vl=MTLDataset(vl_imgs,vl_masks,lmap,aug=False)
# WeightedRandomSampler ensures labeled samples appear in every batch
def _ws_mtl(ds):
    """Weight labeled samples (class≥0) 4× over unlabeled (class=-1)."""
    weights = [4.0 if l >= 0 else 1.0 for l in ds.lbls]
    return WeightedRandomSampler(torch.DoubleTensor(weights), len(weights), replacement=True)
mtl_tr_ld=DataLoader(mtl_tr,SEG_BATCH,sampler=_ws_mtl(mtl_tr),
                     num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
mtl_vl_ld=DataLoader(mtl_vl,SEG_BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
labeled_tr = sum(1 for l in mtl_tr.lbls if l >= 0)
labeled_vl = sum(1 for l in mtl_vl.lbls if l >= 0)
print(f"MTL train:{len(mtl_tr)} val:{len(mtl_vl)}")
print(f"  Labeled (has disease class): train={labeled_tr}, val={labeled_vl}")

In [ ]:
# Phase C — MTL training (FP32, fixed task weights, NaN weight recovery)
_mtl_cnx_enc, _ = build_encoder(BACKBONE_MODE, BACKBONE_SSL, DEVICE)
mtl_model = DentalMTLNet(n_cls=N_CLS, ssl_path=BACKBONE_SSL,
                         convnext_encoder=_mtl_cnx_enc if BACKBONE_MODE=="convnext_b" else None).to(DEVICE)
focal_mtl = FocalLoss(gamma=2.0)

W_SEG = 1.0
W_CLS = 4.0  # v21: further boost (2.0 still too weak; MTL F1 < 0.3 with 2.0)

_need_mtl = True
if MTL_CKPT.exists():
    try:
        mtl_model.load_state_dict(torch.load(MTL_CKPT, map_location=DEVICE, weights_only=True))
        print(f"SKIP MTL — loaded {MTL_CKPT.name}"); _need_mtl = False
    except RuntimeError:
        MTL_CKPT.unlink()

if _need_mtl:
    enc_p = sum([list(m.parameters()) for m in [mtl_model.e0, mtl_model.e1, mtl_model.e2,
                                                  mtl_model.e3, mtl_model.e4]], [])
    seg_p = sum([list(m.parameters()) for m in [mtl_model.aspp, mtl_model.d4, mtl_model.d3,
                                                  mtl_model.d2, mtl_model.d1, mtl_model.up_f,
                                                  mtl_model.seg_out]], [])
    cls_p = list(mtl_model.cls_head.parameters())

    o_enc = torch.optim.AdamW(enc_p, lr=1e-5, weight_decay=1e-4)
    o_seg = torch.optim.AdamW(seg_p, lr=1e-4, weight_decay=1e-4)
    o_cls = torch.optim.AdamW(cls_p, lr=2e-4, weight_decay=1e-4)  # v21: was 5e-5 — cls head was LR-starved
    sch_mtl = torch.optim.lr_scheduler.CosineAnnealingLR(o_enc, T_max=MTL_EPOCHS, eta_min=1e-7)
    sch_seg = torch.optim.lr_scheduler.CosineAnnealingLR(o_seg, T_max=MTL_EPOCHS, eta_min=1e-7)
    sch_cls = torch.optim.lr_scheduler.CosineAnnealingLR(o_cls, T_max=MTL_EPOCHS, eta_min=1e-7)

    best_iou_m = 0.0; best_f1_m = 0.0; hist_mtl = []
    _ckpt_backup = copy.deepcopy(mtl_model.state_dict())  # in-memory backup of best weights

    for ep in range(MTL_EPOCHS):
        mtl_model.train(); sl = 0.0; cl = 0.0; n_skip = 0; n_reset = 0
        for imgs, masks, lbls in tqdm(mtl_tr_ld, desc=f"[MTL] Ep {ep+1}/{MTL_EPOCHS}", leave=False):
            imgs, masks, lbls = imgs.to(DEVICE), masks.to(DEVICE), lbls.to(DEVICE)

            with autocast(dtype=torch.bfloat16):  # v21: bfloat16 for faster training
                seg_l, cls_l = mtl_model(imgs)
                L_seg = seg_loss_fn(seg_l, masks)
                has_lbl = lbls >= 0
                L_cls = focal_mtl(cls_l[has_lbl], lbls[has_lbl]) if has_lbl.any() \
                        else torch.zeros(1, device=DEVICE, requires_grad=True)
                total = W_SEG * L_seg + W_CLS * L_cls

            if torch.isnan(total) or torch.isinf(total):
                for o in [o_enc, o_seg, o_cls]: o.zero_grad()
                n_skip += 1; continue

            for o in [o_enc, o_seg, o_cls]: o.zero_grad()
            total.backward()
            nn.utils.clip_grad_norm_(mtl_model.parameters(), 2.0)
            for o in [o_enc, o_seg, o_cls]: o.step()

            # Detect weight corruption and restore from backup
            first_w = next(iter(mtl_model.parameters()))
            if torch.isnan(first_w).any() or torch.isinf(first_w).any():
                mtl_model.load_state_dict(_ckpt_backup)
                for o in [o_enc, o_seg, o_cls]: o.zero_grad()
                n_reset += 1; continue

            sl += L_seg.item(); cl += L_cls.item()
        sch_mtl.step(); sch_seg.step(); sch_cls.step()

        mtl_model.eval(); vi = 0.0; vp, vl_ = [], []
        with torch.no_grad():
            for imgs, masks, lbls in mtl_vl_ld:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                sl_, cl_ = mtl_model(imgs); vi += iou_metric(sl_, masks)
                hl = lbls >= 0
                if hl.any():
                    vp.extend(cl_[hl].argmax(1).cpu().numpy())
                    vl_.extend(lbls[hl].numpy())
        vi /= len(mtl_vl_ld)
        vf = f1_score(vl_, vp, average="macro", zero_division=0) if vl_ else 0.0
        n_ran = max(len(mtl_tr_ld) - n_skip - n_reset, 1)
        avg_sl = sl / n_ran; avg_cl = cl / n_ran
        total_l = W_SEG * avg_sl + W_CLS * avg_cl
        hist_mtl.append(dict(ep=ep+1, vi=vi, vf=vf))
        extra = ""
        if n_skip: extra += f" skip={n_skip}"
        if n_reset: extra += f" reset={n_reset}"
        print(f"  MTL {ep+1:02d}  IoU={vi:.4f} F1={vf:.4f} seg_l={avg_sl:.4f} cls_l={avg_cl:.4f} total={total_l:.4f}{extra}")
        if vi > best_iou_m or vf > best_f1_m:
            best_iou_m = max(best_iou_m, vi); best_f1_m = max(best_f1_m, vf)
            torch.save(mtl_model.state_dict(), MTL_CKPT)
            _ckpt_backup = copy.deepcopy(mtl_model.state_dict())  # update in-memory backup
            print(f"    ✓ Best MTL saved")
    print(f"Phase B done. IoU={best_iou_m:.4f}(≥0.80) F1={best_f1_m:.4f}(≥0.83)")

## Phase D — Disease Classification

EfficientNetV2-M + CBAM attention on 380x380 tooth patches. Three disease classes from DENTEX.

In [ ]:
# PHASE C PREP — Dental Radiography Patch Extraction (v8)
# Extracts ~1,004 additional labeled tooth patches from Dental_Radiography dataset.
# Class mapping: Cavity → class_id=2 (Caries), Impacted Tooth → class_id=1 (Impacted)
# Fillings, Implant, Normal are excluded (no matching DENTEX disease class).
# Output: processed/tooth_patches_v9/ and processed/patch_labels_v9.csv
import shutil, cv2, numpy as np, pandas as pd
from pathlib import Path

PATCH_DIR_V9 = OUT / "tooth_patches_v9"
PATCH_CSV_V9 = OUT / "patch_labels_v9.csv"
DR_BASE = BASE.parent / "other users" / "Dental_Radiography"
PATCH_SZ = 380  # larger patches for periapical context

PATCH_DIR_V9.mkdir(parents=True, exist_ok=True)

# Remove stale sentinel so fixes take effect
_DR_DONE_RESET = (OUT / "tooth_patches_v9" / ".dr_done")
if _DR_DONE_RESET.exists():
    _DR_DONE_RESET.unlink()
    print("[DR] Removed stale .dr_done sentinel — will re-run extraction")

_DR_DONE = PATCH_DIR_V9 / ".dr_done"
if _DR_DONE.exists():
    print(f"[DR] Dental Radiography patches already extracted — skipping")
else:
    records = []

    # Step 1: Copy existing DENTEX v4 patches
    old_patch_dir = OUT / "tooth_patches_v4"
    old_csv = OUT / "patch_labels_v4.csv"
    copied_v4 = 0
    if old_patch_dir.exists() and old_csv.exists():
        old_df = pd.read_csv(old_csv)
        # Detect path column — v4 uses 'patch_file', newer CSVs use 'img_path'
        _path_col = next((c for c in ['img_path', 'patch_file', 'file', 'path'] if c in old_df.columns), None)
        if _path_col is None:
            print(f"WARNING: Cannot find path column in v4 CSV. Columns: {list(old_df.columns)}")
        else:
            for _, row in old_df.iterrows():
                old_path = old_patch_dir / Path(str(row[_path_col])).name
                if not old_path.exists():
                    old_path = Path(str(row[_path_col]))
                if old_path.exists():
                    new_name = f"dentex_{old_path.name}"
                    dst = PATCH_DIR_V9 / new_name
                    if not dst.exists():
                        shutil.copy2(old_path, dst)
                    records.append({"img_path": str(dst), "class_id": int(row["class_id"])})
                    copied_v4 += 1
        print(f"Copied {copied_v4} existing DENTEX patches from tooth_patches_v4/")
    else:
        print("WARNING: tooth_patches_v4/ or patch_labels_v4.csv not found — only DR patches will be used.")

    # Step 2: Extract Dental_Radiography patches
    # Class mapping: Cavity=Caries(2), Impacted Tooth=Impacted(1)
    class_map = {}
    for folder_name, class_id in [("Cavity", 2), ("Impacted Tooth", 1)]:
        # Check train/ subfolder first, then root
        for split in ["train", "Train", ""]:
            src_dir = (DR_BASE / split / folder_name) if split else (DR_BASE / folder_name)
            if src_dir.exists():
                class_map[src_dir] = class_id
                break

    dr_added = 0
    for src_dir, class_id in class_map.items():
        for img_path in list(src_dir.glob("*.png")) + list(src_dir.glob("*.jpg")) + list(src_dir.glob("*.jpeg")):
            # Load and apply CLAHE
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_rgb = apply_clahe(img_rgb)  # fixed CLAHE preprocessing

            # Center crop to PATCH_SZ x PATCH_SZ (DR images are tooth-focused)
            h, w = img_rgb.shape[:2]
            if h < PATCH_SZ or w < PATCH_SZ:
                # Pad if smaller
                pad_h = max(0, PATCH_SZ - h)
                pad_w = max(0, PATCH_SZ - w)
                img_rgb = np.pad(img_rgb, ((pad_h//2, pad_h-pad_h//2), (pad_w//2, pad_w-pad_w//2), (0,0)))
                h, w = img_rgb.shape[:2]
            y1 = (h - PATCH_SZ) // 2; x1 = (w - PATCH_SZ) // 2
            patch = img_rgb[y1:y1+PATCH_SZ, x1:x1+PATCH_SZ]

            # Save
            uid = f"dr_{src_dir.name}_{img_path.stem}"
            dst = PATCH_DIR_V9 / f"{uid}.png"
            if not dst.exists():
                cv2.imwrite(str(dst), cv2.cvtColor(patch, cv2.COLOR_RGB2BGR))
            records.append({"img_path": str(dst), "class_id": class_id})
            dr_added += 1


    # Step 2b: [v10] Other users/Dataset/ — Caries bbox patches (class_id=1)
    _ds_caries_dir = OTHER / "Dataset" / "annotations" / "bboxes_caries"
    _ds_img_dir    = OTHER / "Dataset" / "images"
    _ds_added = 0
    if _ds_caries_dir.exists() and _ds_img_dir.exists():
        for _bbox_file in sorted(_ds_caries_dir.glob("*.txt")):
            _img_path = _ds_img_dir / (_bbox_file.stem + ".png")
            if not _img_path.exists():
                _img_path = _ds_img_dir / (_bbox_file.stem + ".jpg")
            if not _img_path.exists():
                continue
            _img_bgr = cv2.imread(str(_img_path))
            if _img_bgr is None:
                continue
            _img_rgb = cv2.cvtColor(_img_bgr, cv2.COLOR_BGR2RGB)
            _h_ds, _w_ds = _img_rgb.shape[:2]
            with open(_bbox_file) as _bf:
                for _li, _line in enumerate(_bf):
                    _parts = _line.strip().split()
                    if len(_parts) < 4:
                        continue
                    _x1b = float(_parts[0]); _y1b = float(_parts[1])
                    _x2b = float(_parts[2]); _y2b = float(_parts[3])
                    _pw = (_x2b - _x1b) * 0.2; _ph = (_y2b - _y1b) * 0.2
                    _x1c = max(0, int(_x1b - _pw)); _y1c = max(0, int(_y1b - _ph))
                    _x2c = min(_w_ds, int(_x2b + _pw)); _y2c = min(_h_ds, int(_y2b + _ph))
                    if _x2c - _x1c < 10 or _y2c - _y1c < 10:
                        continue
                    _patch = _img_rgb[_y1c:_y2c, _x1c:_x2c]
                    _patch = cv2.resize(_patch, (PATCH_SZ, PATCH_SZ), interpolation=cv2.INTER_LINEAR)
                    _uid = f"ds_caries_{_bbox_file.stem}_{_li:03d}"
                    _dst = PATCH_DIR_V9 / f"{_uid}.png"
                    if not _dst.exists():
                        cv2.imwrite(str(_dst), cv2.cvtColor(_patch, cv2.COLOR_RGB2BGR))
                    records.append({"img_path": str(_dst), "class_id": 1})  # Caries
                    _ds_added += 1
        print(f"[Dataset] +{_ds_added} caries patches (class_id=1) from Other users/Dataset/")
    else:
        print(f"[Dataset] Directory not found at {_ds_caries_dir} — skipping")

    # Step 3: Save merged CSV
    df = pd.DataFrame(records)
    df.to_csv(PATCH_CSV_V9, index=False)
    total = len(df)
    class_counts = df["class_id"].value_counts().sort_index().to_dict()
    print(f"patch_labels_v9.csv: {total} total patches")
    print(f"  DENTEX patches: {copied_v4}")
    print(f"  Dental Radiography: {dr_added} new patches")
    print(f"  Dataset caries:     {_ds_added} new patches (class_id=1)")
    print(f"  Class distribution: {class_counts}")
    print(f"  (0=Periapical, 1=Impacted, 2=Caries, 3=DeepCaries)")
    _DR_DONE.touch()
    print(f"[DR] Done. patch_labels_v9.csv: {total} total patches")

In [ ]:
import shutil  # v10: explicit import for standalone cell execution
# Phase C PREP v9: Periapical Dataset Integration
# Integrates two new periapical data sources into patch_labels_v9.csv:
#   1. external_datasets/periapical_disease/ (COCO format, ~41K files)
#   2. Other users/Panoramic radiographs with periapical lesions Dataset/
# class_id=2 (Periapical/Caries in our 4-class scheme: 0=Impacted,1=Caries,2=Periapical,3=DeepCaries)
# Idempotent: skips entirely if patch_labels_v9.csv already exists.

import json as _json_peri
import zipfile as _zf_peri

_PATCH_DIR_V9 = PATCH_DIR   # set in Cell 2 as OUT/"tooth_patches_v9"
_PATCH_CSV_V9 = PATCH_CSV   # set in Cell 2 as OUT/"patch_labels_v9.csv"

_PERI_DONE_FLAG = _PATCH_DIR_V9 / ".peri_v9_done"

# Force re-run after fixing COCO image path bug
_PERI_DONE_FLAG_RESET = _PATCH_DIR_V9 / ".peri_v9_done"
if _PERI_DONE_FLAG_RESET.exists():
    _PERI_DONE_FLAG_RESET.unlink()
    print("[PERI v9] Removed stale sentinel — will re-run to add COCO patches")

# Validate sentinel: if .peri_v9_done exists but CSV has no periapical rows, remove sentinel
if _PERI_DONE_FLAG.exists() and _PATCH_CSV_V9.exists():
    import pandas as _pd_check
    _check_df = _pd_check.read_csv(str(_PATCH_CSV_V9))
    _peri_count = (_check_df['class_id'] == 2).sum() if 'class_id' in _check_df.columns else 0
    _peri_from_periapical_src = sum(
        1 for p in _check_df.get('img_path', _pd_check.Series(dtype=str))
        if 'peri_' in str(p)
    )
    if _peri_from_periapical_src == 0:
        print(f"[PERI v9] Stale sentinel detected (no peri_ patches in CSV) — removing sentinel to re-run")
        _PERI_DONE_FLAG.unlink()

if _PERI_DONE_FLAG.exists():
    print(f"[PERI v9] Periapical integration already done (sentinel exists) — skipping")
else:
    import pandas as _pd_peri_init
    _PATCH_DIR_V9.mkdir(parents=True, exist_ok=True)
    # Read existing CSV rows (e.g. from Cell 21 DR patches) so we can append to them
    if _PATCH_CSV_V9.exists():
        _rows_peri = _pd_peri_init.read_csv(str(_PATCH_CSV_V9)).to_dict('records')
        print(f"[PERI v9] Loaded {len(_rows_peri)} existing patch rows from {_PATCH_CSV_V9.name}")
    else:
        _rows_peri = []

    # Step 1: Copy existing v8 patches to v9 directory (only if no existing CSV)
    if not _rows_peri:  # only seed from v8 if we haven't loaded existing rows
        _v8_csv = OUT / "patch_labels_v8.csv"
        _v8_dir = OUT / "tooth_patches_v8"
        if _v8_csv.exists() and _v8_dir.exists():
            import pandas as _pd_peri
            _df_v8 = _pd_peri.read_csv(_v8_csv)
            _copied = 0
            for _, _row in _df_v8.iterrows():
                _src = Path(str(_row['img_path']))
                if _src.exists():
                    _dst = _PATCH_DIR_V9 / _src.name
                    if not _dst.exists():
                        shutil.copy2(str(_src), str(_dst))
                        _copied += 1
                    _rows_peri.append({'img_path': str(_dst), 'class_id': int(_row['class_id'])})
            print(f"[PERI v9] Copied {_copied} patches from v8 ({len(_rows_peri)} rows total)")
        else:
            # v8 patches not found — check for v4 fallback
            _v4_csv = OUT / "patch_labels_v4.csv"
            _v4_dir = OUT / "tooth_patches_v4"
            if _v4_csv.exists() and _v4_dir.exists():
                import pandas as _pd_peri
                _df_v4 = _pd_peri.read_csv(_v4_csv)
                for _, _row in _df_v4.iterrows():
                    _src = Path(str(_row['img_path']))
                    if _src.exists():
                        _dst = _PATCH_DIR_V9 / _src.name
                        if not _dst.exists():
                            shutil.copy2(str(_src), str(_dst))
                        _rows_peri.append({'img_path': str(_dst), 'class_id': int(_row['class_id'])})
                print(f"[PERI v9] Fallback: copied {len(_rows_peri)} patches from v4")

    # Step 2: Extract and integrate COCO periapical ZIP
    if PERI_EXT_ZIP.exists():
        if not PERI_EXT_DIR.exists():
            print(f"[PERI v9] Extracting {PERI_EXT_ZIP.name} (~2.5 GB) ...")
            with _zf_peri.ZipFile(str(PERI_EXT_ZIP)) as _z:
                _z.extractall(str(PERI_EXT_DIR))
            print("[PERI v9] Extraction complete")

        # Find COCO annotation files
        _coco_jsons = (list(PERI_EXT_DIR.rglob("train_coco.json")) +
                       list(PERI_EXT_DIR.rglob("train*.json")) +
                       list(PERI_EXT_DIR.rglob("*train*.json")))
        _coco_jsons = [p for p in _coco_jsons if 'annotation' not in str(p).lower() or 'coco' in str(p).lower()]

        for _coco_f in _coco_jsons[:1]:
            print(f"[PERI v9] Loading COCO annotations from {_coco_f.name} ...")
            with open(str(_coco_f)) as _f:
                _coco = _json_peri.load(_f)

            # Discover periapical / disease category IDs
            _all_cats = _coco.get('categories', [])
            print(f"[PERI v9] COCO categories: {[c['name'] for c in _all_cats]}")
            _peri_ids = {c['id'] for c in _all_cats
                         if any(k in c['name'].lower()
                                for k in ['periapical','periapi','lesion','apical','peri','disease','caries','cavity'])}
            if not _peri_ids:
                # No obvious periapical name — use all non-healthy categories
                _peri_ids = {c['id'] for c in _all_cats
                             if not any(k in c['name'].lower() for k in ['normal','healthy','background','none'])}
            print(f"[PERI v9] Using category IDs as periapical: {_peri_ids}")

            # Build id → image path lookup
            # Determine split folder from JSON filename (train_coco.json → "train")
            _split_name = _coco_f.stem.replace('_coco', '').replace('coco_', '')  # "train", "valid", "test"
            _img_root = _coco_f.parent.parent / _split_name  # e.g. extracted/COCO/COCO/train/

            # Build name→path index for fast lookup (handles any subdirectory depth)
            _name2path = {}
            if _img_root.exists():
                for _p in _img_root.rglob("*.jpg"):
                    _name2path[_p.name] = _p
                for _p in _img_root.rglob("*.JPG"):
                    _name2path[_p.name] = _p
                for _p in _img_root.rglob("*.png"):
                    _name2path[_p.name] = _p
            print(f"[PERI v9] Image index built: {len(_name2path)} images in {_img_root}")

            _id2path = {}
            for _im in _coco.get('images', []):
                _fname = Path(_im['file_name']).name  # use just the filename, not any directory prefix
                if _fname in _name2path:
                    _id2path[_im['id']] = _name2path[_fname]
            print(f"[PERI v9] Matched {len(_id2path)}/{len(_coco.get('images', []))} images in id2path")

            _peri_added = 0
            for _ann in _coco.get('annotations', []):
                if _ann['category_id'] not in _peri_ids:
                    continue
                _img_p = _id2path.get(_ann['image_id'])
                if not _img_p:
                    continue
                _img = cv2.imread(str(_img_p))
                if _img is None:
                    continue
                _h, _w = _img.shape[:2]
                _x, _y, _bw, _bh = [int(v) for v in _ann['bbox']]
                # 20% padding for periapical context
                _px, _py = int(_bw * 0.2), int(_bh * 0.2)
                _x1 = max(0, _x - _px); _y1 = max(0, _y - _py)
                _x2 = min(_w, _x + _bw + _px); _y2 = min(_h, _y + _bh + _py)
                if _x2 <= _x1 or _y2 <= _y1:
                    continue
                _crop = _img[_y1:_y2, _x1:_x2]
                if _crop.size == 0:
                    continue
                _crop = cv2.resize(_crop, (PATCH_SIZE_V8, PATCH_SIZE_V8))
                _crop_rgb = cv2.cvtColor(_crop, cv2.COLOR_BGR2RGB)
                _crop_rgb = apply_clahe(_crop_rgb)
                _out_name = f"peri_coco_{_ann['image_id']}_{_ann['id']}.jpg"
                _out_path = _PATCH_DIR_V9 / _out_name
                cv2.imwrite(str(_out_path), cv2.cvtColor(_crop_rgb, cv2.COLOR_RGB2BGR))
                _rows_peri.append({'img_path': str(_out_path), 'class_id': 2})
                _peri_added += 1
            print(f"[PERI v9] COCO: extracted {_peri_added} periapical patches")
    else:
        print(f"[PERI v9] Periapical ZIP not found at {PERI_EXT_ZIP} — skipping COCO source")

    # Step 3: Integrate Other users periapical panoramic dataset
    if PERI_OTH_DIR.exists():
        _oth_images = (list(PERI_OTH_DIR.rglob("*.jpg")) +
                       list(PERI_OTH_DIR.rglob("*.JPG")) +
                       list(PERI_OTH_DIR.rglob("*.png")))
        _oth_labels = list(PERI_OTH_DIR.rglob("*.txt"))
        _oth_coco   = list(PERI_OTH_DIR.rglob("*.json"))
        print(f"[PERI v9] Other users periapical: {len(_oth_images)} images, "
              f"{len(_oth_labels)} YOLO labels, {len(_oth_coco)} JSON files")

        _oth_added = 0

        if _oth_coco:
            # COCO format — same logic as above
            for _coco_f in _oth_coco[:1]:
                try:
                    with open(str(_coco_f)) as _f:
                        _coco2 = _json_peri.load(_f)
                    if 'annotations' not in _coco2:
                        continue
                    _all_cats2 = _coco2.get('categories', [])
                    print(f"[PERI v9] OTH COCO categories: {[c['name'] for c in _all_cats2]}")
                    _peri_ids2 = {c['id'] for c in _all_cats2
                                  if any(k in c['name'].lower()
                                         for k in ['periapical','lesion','apical','peri','disease'])}
                    if not _peri_ids2:
                        _peri_ids2 = {c['id'] for c in _all_cats2
                                      if not any(k in c['name'].lower() for k in ['normal','healthy','background'])}
                    _id2path2 = {}
                    for _im in _coco2.get('images', []):
                        for _cp in [_coco_f.parent / _im['file_name'], PERI_OTH_DIR / _im['file_name']]:
                            if _cp.exists():
                                _id2path2[_im['id']] = _cp; break
                    for _ann in _coco2.get('annotations', []):
                        if _ann['category_id'] not in _peri_ids2: continue
                        _img_p = _id2path2.get(_ann['image_id'])
                        if not _img_p: continue
                        _img = cv2.imread(str(_img_p))
                        if _img is None: continue
                        _h, _w = _img.shape[:2]
                        _x, _y, _bw, _bh = [int(v) for v in _ann['bbox']]
                        _px, _py = int(_bw * 0.2), int(_bh * 0.2)
                        _x1 = max(0, _x - _px); _y1 = max(0, _y - _py)
                        _x2 = min(_w, _x + _bw + _px); _y2 = min(_h, _y + _bh + _py)
                        if _x2 <= _x1 or _y2 <= _y1: continue
                        _crop = cv2.resize(_img[_y1:_y2, _x1:_x2], (PATCH_SIZE_V8, PATCH_SIZE_V8))
                        _crop_rgb = apply_clahe(cv2.cvtColor(_crop, cv2.COLOR_BGR2RGB))
                        _out_name = f"peri_oth_coco_{_ann['image_id']}_{_ann['id']}.jpg"
                        _out_path = _PATCH_DIR_V9 / _out_name
                        cv2.imwrite(str(_out_path), cv2.cvtColor(_crop_rgb, cv2.COLOR_RGB2BGR))
                        _rows_peri.append({'img_path': str(_out_path), 'class_id': 2})
                        _oth_added += 1
                except Exception as _e:
                    print(f"[PERI v9] COCO parse error: {_e}")

        elif _oth_labels:
            # YOLO format
            for _lbl_p in _oth_labels:
                for _ext in ['.jpg', '.JPG', '.jpeg', '.png', '.PNG']:
                    _img_p = _lbl_p.with_suffix(_ext)
                    if not _img_p.exists():
                        continue
                    _img = cv2.imread(str(_img_p))
                    if _img is None:
                        break
                    _h, _w = _img.shape[:2]
                    for _line in _lbl_p.read_text().strip().splitlines():
                        _parts = _line.split()
                        if len(_parts) < 5:
                            continue
                        _cx, _cy, _bw2, _bh2 = float(_parts[1]), float(_parts[2]), float(_parts[3]), float(_parts[4])
                        _x1 = int((_cx - _bw2/2) * _w); _x2 = int((_cx + _bw2/2) * _w)
                        _y1 = int((_cy - _bh2/2) * _h); _y2 = int((_cy + _bh2/2) * _h)
                        _crop = _img[max(0,_y1):_y2, max(0,_x1):_x2]
                        if _crop.size == 0:
                            continue
                        _crop = cv2.resize(_crop, (PATCH_SIZE_V8, PATCH_SIZE_V8))
                        _crop_rgb = apply_clahe(cv2.cvtColor(_crop, cv2.COLOR_BGR2RGB))
                        _out_name = f"peri_oth_{_img_p.stem}_{int(_cx*1000)}.jpg"
                        _out_path = _PATCH_DIR_V9 / _out_name
                        cv2.imwrite(str(_out_path), cv2.cvtColor(_crop_rgb, cv2.COLOR_RGB2BGR))
                        _rows_peri.append({'img_path': str(_out_path), 'class_id': 2})
                        _oth_added += 1
                    break

        else:
            # No labels — images are already periapical crops (use center crop)
            for _img_p in _oth_images[:5000]:
                _img = cv2.imread(str(_img_p))
                if _img is None:
                    continue
                _h, _w = _img.shape[:2]
                _sz = min(_h, _w)
                _cy2, _cx2 = _h // 2, _w // 2
                _crop = _img[max(0, _cy2-_sz//2):_cy2+_sz//2, max(0, _cx2-_sz//2):_cx2+_sz//2]
                _crop = cv2.resize(_crop, (PATCH_SIZE_V8, PATCH_SIZE_V8))
                _crop_rgb = apply_clahe(cv2.cvtColor(_crop, cv2.COLOR_BGR2RGB))
                _out_name = f"peri_oth_{_img_p.stem}.jpg"
                _out_path = _PATCH_DIR_V9 / _out_name
                cv2.imwrite(str(_out_path), cv2.cvtColor(_crop_rgb, cv2.COLOR_RGB2BGR))
                _rows_peri.append({'img_path': str(_out_path), 'class_id': 2})
                _oth_added += 1

        print(f"[PERI v9] Other users periapical: added {_oth_added} patches")
    else:
        print(f"[PERI v9] {PERI_OTH_DIR.name} not found — skipping")

    # Step 4: Save final patch_labels_v9.csv
    import pandas as _pd_peri2
    _df_final = _pd_peri2.DataFrame(_rows_peri)
    _df_final.to_csv(str(_PATCH_CSV_V9), index=False)
    _PERI_DONE_FLAG.touch()
    print(f"\n[PERI v9] patch_labels_v9.csv saved: {len(_df_final)} total patches")
    print(_df_final['class_id'].value_counts().rename({0:'Impacted',1:'Caries',2:'Periapical',3:'DeepCaries'}).to_string())


In [ ]:

def cutmix_v6(imgs, lbls, alpha=1.0):
    """v6 fix: return separate label sets; caller mixes focal losses by lambda."""
    lam=np.random.beta(alpha,alpha); B=imgs.size(0); idx=torch.randperm(B)
    W,H=imgs.size(3),imgs.size(2)
    cw,ch=int(W*np.sqrt(1-lam)),int(H*np.sqrt(1-lam))
    cx,cy=np.random.randint(W),np.random.randint(H)
    x1,x2=max(0,cx-cw//2),min(W,cx+cw//2)
    y1,y2=max(0,cy-ch//2),min(H,cy+ch//2)
    mix=imgs.clone(); mix[:,:,y1:y2,x1:x2]=imgs[idx,:,y1:y2,x1:x2]
    lam_real=1-(x2-x1)*(y2-y1)/(W*H)
    return mix, lbls, lbls[idx], lam_real

def mixup_v7(imgs, lbls, alpha=0.4):
    """Standard Mixup augmentation."""
    lam = float(torch.distributions.Beta(alpha, alpha).sample())
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    return mixed, lbls, lbls[idx], lam

class CBAMBlock(nn.Module):
    """Channel + Spatial Attention Module (Woo et al. 2018).
    Proven to improve dental disease classification AUC by ~5%.
    Source: thisdatasetmustbedead.ipynb community solution.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ch_avg = nn.AdaptiveAvgPool2d(1)
        self.ch_max = nn.AdaptiveMaxPool2d(1)
        self.ch_mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, max(1, channels // reduction), bias=False),
            nn.ReLU(),
            nn.Linear(max(1, channels // reduction), channels, bias=False),
        )
        self.sp_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        # Channel attention
        ca = torch.sigmoid(
            self.ch_mlp(self.ch_avg(x)) + self.ch_mlp(self.ch_max(x))
        ).view(x.size(0), x.size(1), 1, 1)
        x = x * ca
        # Spatial attention
        sp = torch.sigmoid(self.sp_conv(
            torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True).values], dim=1)
        ))
        return x * sp

class DiseaseClassifierV6(nn.Module):
    def __init__(self, n_cls=4, ssl_path=None, use_ssl=True):
        super().__init__()
        from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights
        _cls_backbone = efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1)
        self.eff = _cls_backbone
        in_f = self.eff.classifier[1].in_features  # 1280 for EfficientNetV2-M
        self.cbam = CBAMBlock(channels=1280)  # EfficientNetV2-M penultimate feature dim
        self.use_ssl=use_ssl
        if use_ssl:
            ssl_path=Path(ssl_path) if ssl_path else BACKBONE_SSL
            bb=models.resnet50(weights=None)
            sd=torch.load(ssl_path,map_location="cpu",weights_only=True)
            if next(iter(sd),"").split(".")[0].isdigit():
                _m={"0":"conv1","1":"bn1","4":"layer1","5":"layer2","6":"layer3","7":"layer4"}
                sd={".".join([_m.get(k.split(".")[0],k.split(".")[0])]+k.split(".")[1:]):v for k,v in sd.items() if k.split(".")[0] in _m}
            bb.load_state_dict(sd,strict=False)
            self.ssl_feat=nn.Sequential(bb.conv1,bb.bn1,bb.relu,bb.maxpool,
                                         bb.layer1,bb.layer2,bb.layer3,
                                         nn.AdaptiveAvgPool2d(1),nn.Flatten())
            for p in self.ssl_feat.parameters(): p.requires_grad=False
            ssl_dim=1024
        else: ssl_dim=0
        self.eff.classifier=nn.Identity()
        self.clf=nn.Sequential(nn.Dropout(0.4),nn.Linear(in_f+ssl_dim,512),
                                nn.GELU(),nn.Dropout(0.3),nn.Linear(512,n_cls))

    def forward(self,x):
        feat=self.eff.features(x)
        feat=self.cbam(feat)   # v8: channel + spatial attention
        f=self.eff.avgpool(feat).flatten(1)
        if self.use_ssl:
            xs=F.interpolate(x,(224,224),mode="bilinear",align_corners=False)
            f=torch.cat([f,self.ssl_feat(xs)],1)
        return self.clf(f)

print("DiseaseClassifierV6 + fixed CutMix defined ✓")


In [ ]:
class PatchDataset(Dataset):
    def __init__(self,df,aug=False,minority_aug=False):
        self.df=df.reset_index(drop=True); self.aug=aug; self.min_aug=minority_aug
        # Detect path column: v9 CSV uses 'img_path', legacy uses 'patch_file'
        self._has_img_path = "img_path" in df.columns
        # v9 perf: removed A.CLAHE (patches already CLAHE-processed at extraction time — double CLAHE was wasteful)
        _aug=A.Compose([A.Resize(CLS_SIZE,CLS_SIZE),A.HorizontalFlip(p=0.5),A.VerticalFlip(p=0.2),
                         A.Affine(translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
                         scale=(0.85, 1.15), rotate=(-30, 30), p=0.6),A.RandomBrightnessContrast(0.3,0.3,p=0.5),
                         A.Sharpen(alpha=(0.2, 0.5), lightness=(0.75, 1.5), p=0.3),
                         A.GaussNoise(std_range=(0.012,0.030),p=0.3),
                         A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()])
        _maug=A.Compose([A.Resize(CLS_SIZE,CLS_SIZE),A.HorizontalFlip(p=0.5),A.VerticalFlip(p=0.4),
                          A.Affine(translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)},
                          scale=(0.80, 1.20), rotate=(-45, 45), p=0.8),A.RandomBrightnessContrast(0.4,0.4,p=0.6),
                          A.Sharpen(alpha=(0.2, 0.5), lightness=(0.75, 1.5), p=0.4),
                          A.ElasticTransform(alpha=80,sigma=5,p=0.2),  # v9: reduced from p=0.4 (expensive)
                          A.GaussNoise(std_range=(0.018,0.035),p=0.4),
                          A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()])
        _val=A.Compose([A.Resize(CLS_SIZE,CLS_SIZE),
                         A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()])
        self.atf=_aug; self.mtf=_maug; self.vtf=_val
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; lbl=int(row["class_id"])
        if lbl==3: lbl=1  # DeepCaries merged into Caries
        # Handle path column difference between CSV versions (relative)
        try:
            _p = Path(str(row["img_path"])) if self._has_img_path else PATCH_DIR/row["patch_file"]
            img=np.array(Image.open(_p).convert("RGB"))
        except: img=np.zeros((CLS_SIZE,CLS_SIZE,3),dtype=np.uint8)
        if not self.aug: tf=self.vtf
        elif self.min_aug and lbl==2: tf=self.mtf
        else: tf=self.atf
        out=tf(image=img); return out["image"],torch.tensor(lbl)

df_patches=pd.read_csv(PATCH_CSV)
_cnt4=[len(df_patches[df_patches["class_id"]==i]) for i in range(4)]
cnt=[_cnt4[0],_cnt4[1]+_cnt4[3],_cnt4[2]]  # DeepCaries merged into Caries
# Effective-number class weights (beta=0.9999, Cui et al. 2019)
_beta=0.9999
_eff=[(1-_beta**max(c,1))/(1-_beta) for c in cnt]
_eff_max=max(_eff)
CW=torch.tensor([_eff_max/e for e in _eff],dtype=torch.float32)
print(f"Class counts: {cnt}")
print(f"Class weights (eff-num): {CW.numpy().round(3)}")

def focal_mg(pred, tgt, w=None):
    """Mixed-gamma focal: Periapical(2) gets gamma=FOCAL_GAMMA_PERI (4.0), others gamma=2."""
    fl2=FocalLoss(gamma=2.0,weight=w); fl3=FocalLoss(gamma=FOCAL_GAMMA_PERI,weight=w)
    pm=(tgt==2)
    if pm.any() and (~pm).any():
        l2=fl2(pred[~pm],tgt[~pm]); l3=fl3(pred[pm],tgt[pm])
        return (l2*(~pm).sum()+l3*pm.sum())/len(tgt)
    return fl3(pred,tgt) if pm.any() else fl2(pred,tgt)

sss=StratifiedShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
ti,vi_=next(sss.split(np.arange(len(df_patches)),df_patches["class_id"]))
cls_tr_ds=PatchDataset(df_patches.iloc[ti],aug=True, minority_aug=True)
cls_vl_ds=PatchDataset(df_patches.iloc[vi_],aug=False)

def _ws(df):
    _mc=df["class_id"].replace({3:1})  # DeepCaries->Caries
    cn=_mc.value_counts().to_dict()
    PERIAPICAL_IDX=2  # Periapical is class index 2 in DENTEX label mapping
    w=_mc.map(lambda c:(PERIAPICAL_BOOST/cn[c] if c==PERIAPICAL_IDX else 1.0/cn[c])).values
    return WeightedRandomSampler(torch.DoubleTensor(w),len(w))

cls_tr_ld=DataLoader(cls_tr_ds,CLS_BATCH,sampler=_ws(df_patches.iloc[ti]),
                      num_workers=NUM_WORKERS,pin_memory=True)
cls_vl_ld=DataLoader(cls_vl_ds,CLS_BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
print(f"Cls train:{len(cls_tr_ds)} val:{len(cls_vl_ds)}")

In [ ]:

cls_model=DiseaseClassifierV6(n_cls=N_CLS,ssl_path=BACKBONE_SSL,use_ssl=True).to(DEVICE)
_need_cls=True
if CLS_CKPT.exists():
    try:
        cls_model.load_state_dict(torch.load(CLS_CKPT,map_location=DEVICE,weights_only=True))
        print(f"SKIP cls — loaded {CLS_CKPT.name}"); _need_cls=False
    except RuntimeError: CLS_CKPT.unlink()

if _need_cls:
    opt_c=torch.optim.AdamW([p for p in cls_model.parameters() if p.requires_grad],
                               lr=3e-4,weight_decay=1e-4)
    sch_c=torch.optim.lr_scheduler.OneCycleLR(opt_c,max_lr=3e-4,
              steps_per_epoch=len(cls_tr_ld),epochs=CLS_EPOCHS,pct_start=0.1)
    scl_c=GradScaler(); best_f1=0.0; pc=0; PATIENCE=25; hist_cls=[]  # v21: patience 15→25

    from torch.optim.swa_utils import AveragedModel, update_bn as swa_update_bn
    _swa_cls_start = int(0.75 * CLS_EPOCHS)
    swa_cls = AveragedModel(cls_model); _swa_cls_count = 0
    for ep in range(CLS_EPOCHS):
        cls_model.train(); tl=0.0
        for imgs,lbls in tqdm(cls_tr_ld,desc=f"[Cls] Ep {ep+1}/{CLS_EPOCHS}",leave=False):
            imgs,lbls=imgs.to(DEVICE),lbls.to(DEVICE)
            r=np.random.rand()
            if r < 0.33:
                # CutMix (1/3 of batches)
                imgs_m,la,lb,lam=cutmix_v6(imgs,lbls)
                with autocast(dtype=torch.bfloat16):
                    logits=cls_model(imgs_m)
                    loss=lam*focal_mg(logits,la,CW.to(DEVICE))+(1-lam)*focal_mg(logits,lb,CW.to(DEVICE))
            elif r < 0.66:
                # Mixup (1/3 of batches)
                imgs_m,la,lb,lam=mixup_v7(imgs,lbls)
                with autocast(dtype=torch.bfloat16):
                    logits=cls_model(imgs_m)
                    loss=lam*focal_mg(logits,la,CW.to(DEVICE))+(1-lam)*focal_mg(logits,lb,CW.to(DEVICE))
            else:
                # Standard (1/3 of batches)
                with autocast(dtype=torch.bfloat16): loss=focal_mg(cls_model(imgs),lbls,CW.to(DEVICE))
            opt_c.zero_grad(); scl_c.scale(loss).backward()
            scl_c.unscale_(opt_c); nn.utils.clip_grad_norm_(cls_model.parameters(),3.0)
            scl_c.step(opt_c); scl_c.update(); sch_c.step(); tl+=loss.item()

        cls_model.eval(); ap,al=[],[]
        with torch.no_grad():
            for imgs,lbls in cls_vl_ld:
                p=cls_model(imgs.to(DEVICE)).argmax(1).cpu()
                ap.extend(p.numpy()); al.extend(lbls.numpy())
        acc=np.mean(np.array(ap)==np.array(al))
        f1=f1_score(al,ap,average="macro",zero_division=0)
        pf=f1_score(al,ap,labels=[2],average="macro",zero_division=0)
        hist_cls.append(dict(ep=ep+1,tl=tl/len(cls_tr_ld),acc=acc,f1=f1,pf=pf))
        print(f"  Cls {ep+1:02d}  loss={tl/len(cls_tr_ld):.4f} acc={acc:.4f} F1={f1:.4f} PeriF1={pf:.4f}")
        if ep >= _swa_cls_start: swa_cls.update_parameters(cls_model); _swa_cls_count += 1
        if f1>best_f1: best_f1=f1; pc=0; torch.save(cls_model.state_dict(),CLS_CKPT); print(f"    ✓ Best(F1={f1:.4f})")
        else:
            if ep < _swa_cls_start: pc+=1
            if pc>=PATIENCE: print(f"  Early stop ep{ep+1}"); break
    if _swa_cls_count > 0:
        print(f"\nSWA finalize ({_swa_cls_count} snapshots)...")
        swa_update_bn(cls_tr_ld, swa_cls, device=DEVICE)
        swa_cls.eval(); _sv=[]; _sl=[]
        with torch.no_grad():
            for imgs, lbls in cls_vl_ld:
                _sv.extend(swa_cls(imgs.to(DEVICE)).argmax(1).cpu().numpy())
                _sl.extend(lbls.numpy())
        _swa_f1 = f1_score(_sl, _sv, average="macro", zero_division=0)
        print(f"  SWA F1={_swa_f1:.4f} (best so far={best_f1:.4f})")
        if _swa_f1 > best_f1:
            best_f1 = _swa_f1; torch.save(swa_cls.module.state_dict(), CLS_CKPT)
            print("  SWA improved!")

    print(f"Phase C done. F1={best_f1:.4f}(≥0.86)")

def tta_predict(model, imgs):
    """v10 Multi-scale TTA: 3 scales x hflip = 6-view ensemble."""
    model.eval(); preds = []
    with torch.no_grad():
        for scale in [0.8, 1.0, 1.2]:
            sz = int(CLS_SIZE * scale)
            x_s = F.interpolate(imgs, size=(sz,sz), mode="bilinear", align_corners=False)
            x_s = F.interpolate(x_s, size=(CLS_SIZE,CLS_SIZE), mode="bilinear", align_corners=False)
            preds.append(torch.softmax(model(x_s), dim=1))
            preds.append(torch.softmax(model(torch.flip(x_s,[3])), dim=1))
    return torch.stack(preds).mean(0)


In [ ]:

from sklearn.metrics import roc_auc_score
cls_model.eval(); ap,al,all_probs=[],[],[]
for imgs,lbls in cls_vl_ld:
    prob=tta_predict(cls_model,imgs.to(DEVICE))
    ap.extend(prob.argmax(1).cpu().numpy())
    al.extend(lbls.numpy())
    all_probs.append(prob.cpu())
all_probs=torch.cat(all_probs,0).numpy()
print(classification_report(al,ap,labels=list(range(N_CLS)),target_names=CLASS_NAMES,zero_division=0))

# ROC-AUC (one-vs-rest) -- standard in dental disease classification papers
# Literature: AUC 0.82-0.95 for dental disease detection
try:
    auc_macro=roc_auc_score(al,all_probs,multi_class="ovr",average="macro")
    auc_per=[roc_auc_score((np.array(al)==c).astype(int),all_probs[:,c]) for c in range(N_CLS)]
    print(f"\nROC-AUC (OvR macro): {auc_macro:.4f}")
    for c,name in enumerate(CLASS_NAMES):
        print(f"  AUC {name}: {auc_per[c]:.4f}")
    print("  (>0.90 excellent | >0.80 good | literature range: 0.82-0.95)")
except Exception as _ae:
    print(f"AUC skipped: {_ae}")

cm=confusion_matrix(al,ap); fig,ax=plt.subplots(figsize=(10,8))
ConfusionMatrixDisplay(cm,display_labels=CLASS_NAMES).plot(ax=ax,cmap="Blues")
ax.set_title("Phase C - Disease (EfficientNetV2-M + TTA)"); plt.tight_layout()
plt.savefig(VIS_DIR/"cls_v6_cm.png",dpi=100,bbox_inches="tight"); plt.show()

## Phase E — Explainability & Ceph Models

GradCAM visualization + cephalometric model definitions used in later phases.

In [ ]:

class GradCAM:
    def __init__(self,model,layer):
        self.model=model; self._f=None; self._g=None
        self._h=[layer.register_forward_hook(lambda m,i,o:setattr(self,"_f",o)),
                 layer.register_full_backward_hook(lambda m,gi,go:setattr(self,"_g",go[0]))]
    def __call__(self,x,cls=None):
        self.model.eval(); x=x.requires_grad_(True)
        out=self.model(x)
        if cls is None: cls=out.argmax(1).item()
        self.model.zero_grad(); out[0,cls].backward()
        w=self._g.mean(dim=(2,3),keepdim=True)
        cam=F.relu((w*self._f).sum(1,keepdim=True))
        cam=F.interpolate(cam,x.shape[2:],mode="bilinear",align_corners=False)
        cam=cam-cam.min(); cam=cam/(cam.max()+1e-8)
        return cam[0,0].detach().cpu().numpy(),cls
    def remove(self):
        for h in self._h: h.remove()

try:
    gradcam=GradCAM(cls_model,cls_model.eff.features[-1]); print("GradCAM ready ✓")
except Exception as e: print(f"GradCAM failed: {e}"); gradcam=None

COLORS={"Impacted":(255,180,0),"Caries":(255,80,80),"Periapical":(255,0,200),"DeepCaries":(100,100,255)}

with open(DIS_JSON) as f: dis_full=json.load(f)
dis_imgs_full={img["id"]:img for img in dis_full["images"]}
fig,axes=plt.subplots(2,3,figsize=(20,12))
for ax,iid in zip(axes.flat,list(dis_imgs_full.keys())[:6]):
    info=dis_imgs_full[iid]; src=DIS_XRAYS/info["file_name"]
    if not src.exists(): ax.axis("off"); continue
    img=Image.open(src).convert("RGB"); draw=ImageDraw.Draw(img,"RGBA")
    anns=[a for a in dis_full["annotations"] if a["image_id"]==iid]
    for ann in anns:
        cname=DISEASE_MAP.get(int(ann.get("category_id_3", ann.get("category_id",1)-1)),"?")
        col=COLORS.get(cname,(128,128,128))
        x,y,w,h=ann["bbox"]
        x2,y2=x+w,y+h
        # Clamp to image bounds
        x,y=max(0,x),max(0,y)
        x2,y2=min(img.width,x2),min(img.height,y2)
        draw.rectangle([x,y,x2,y2],outline=col+(220,),width=2)
    ax.imshow(img); ax.set_title(f"ID {iid} — {len(anns)} annotations", fontsize=11); ax.axis("off")
ptch=[mpatches.Patch(color=np.array(c)/255,label=n) for n,c in COLORS.items()]
fig.legend(handles=ptch,loc="lower center",ncol=4)
plt.suptitle("Phase D — FDI Overlays"); plt.tight_layout()
plt.savefig(VIS_DIR/"fdi_v6.png",dpi=100,bbox_inches="tight"); plt.show()


In [ ]:
# Cephalometric model definitions (CephNet, soft_argmax, wing_loss)

class CephNet(nn.Module):
    def __init__(self,n_lm=N_LM,ssl_path=None,convnext_encoder=None):
        super().__init__()
        self._is_convnext = convnext_encoder is not None
        if self._is_convnext:
            self.encoder = convnext_encoder
            # ConvNeXt-B channels: e0=128, e1=256, e2=512, e3=1024
            self.d3=nn.Sequential(nn.ConvTranspose2d(1024,512,2,stride=2),nn.BatchNorm2d(512),nn.ReLU(True),_DoubleConv(512,512))
            self.d2=nn.Sequential(nn.ConvTranspose2d(512,256,2,stride=2), nn.BatchNorm2d(256),nn.ReLU(True),_DoubleConv(256,256))
            self.d1=nn.Sequential(nn.ConvTranspose2d(256,64,2,stride=2),  nn.BatchNorm2d(64), nn.ReLU(True),_DoubleConv(64,64))
        else:
            ssl_path=Path(ssl_path) if ssl_path else BACKBONE_SSL
            bb=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            if ssl_path.exists():
                sd=torch.load(ssl_path,map_location="cpu",weights_only=True)
                if next(iter(sd),"").split(".")[0].isdigit():
                    _m={"0":"conv1","1":"bn1","4":"layer1","5":"layer2","6":"layer3","7":"layer4"}
                    sd={".".join([_m.get(k.split(".")[0],k.split(".")[0])]+k.split(".")[1:]):v for k,v in sd.items() if k.split(".")[0] in _m}
                miss,_=bb.load_state_dict(sd,strict=False)
                print(f"  CephNet SSL: {len(bb.state_dict())-len(miss)}/{len(bb.state_dict())} model params")
            self.e0=nn.Sequential(bb.conv1,bb.bn1,bb.relu); self.mp=bb.maxpool
            self.e1=bb.layer1; self.e2=bb.layer2; self.e3=bb.layer3
            self.d3=nn.Sequential(nn.ConvTranspose2d(1024,512,2,stride=2),nn.BatchNorm2d(512),nn.ReLU(True),_DoubleConv(512,512))
            self.d2=nn.Sequential(nn.ConvTranspose2d(512,256,2,stride=2), nn.BatchNorm2d(256),nn.ReLU(True),_DoubleConv(256,256))
            self.d1=nn.Sequential(nn.ConvTranspose2d(256,64,2,stride=2),  nn.BatchNorm2d(64), nn.ReLU(True),_DoubleConv(64,64))
        self.head=nn.Conv2d(64,n_lm,1)
    def forward(self,x):
        if self._is_convnext:
            e0,e1,e2,e3=self.encoder(x)
            s3=e3
        else:
            s0=self.e0(x); s1=self.e1(self.mp(s0)); s2=self.e2(s1); s3=self.e3(s2)
        d=self.d3(s3); d=self.d2(d); d=self.d1(d)
        return self.head(F.adaptive_avg_pool2d(d,256))

import math

def wing_loss(pred, gt, wts, w=10.0, eps=2.0):
    """Wing loss for heatmap-based landmark detection (Feng et al. 2018).
    Better than MSE for small displacement errors — more sensitive to precise localization.
    pred, gt: (B, N_LM, H, W) — heatmaps
    wts: (B, N_LM) or (N_LM,) — per-landmark consistency weights
    w: wing width threshold (pixels in normalized heatmap space)
    eps: curvature at zero
    """
    # Element-wise difference in heatmap space
    diff = (pred - gt).abs()  # (B, N_LM, H, W)
    C = w * (1.0 - math.log(1.0 + w / eps))
    loss_map = torch.where(
        diff < w,
        w * torch.log(1.0 + diff / eps),
        diff - C
    )  # (B, N_LM, H, W)
    # Average over spatial dims, weight by landmark consistency
    per_lm = loss_map.mean(dim=(2, 3))  # (B, N_LM)
    if wts.dim() == 1:
        wts = wts.unsqueeze(0)  # broadcast over batch
    weighted = (per_lm * wts).sum(dim=1) / (wts.sum(dim=1) + 1e-8)  # (B,)
    return weighted.mean()

def decode_hm(hm):
    B,N,H,W=hm.shape; flat=torch.softmax(hm.view(B,N,-1),-1)
    xs=torch.arange(W,dtype=torch.float32,device=hm.device)
    ys=torch.arange(H,dtype=torch.float32,device=hm.device)
    gx=xs.view(1,W).expand(H,W).reshape(-1); gy=ys.view(H,1).expand(H,W).reshape(-1)
    return torch.stack([(flat*gx).sum(-1),(flat*gy).sum(-1)],-1)

def soft_argmax_2d(heatmap):
    """Differentiable sub-pixel landmark localization.
    Args:
        heatmap: (B, N, H, W) tensor
    Returns:
        coords: (B, N, 2) tensor with (x, y) in [0, 1] range
    """
    B, N, H, W = heatmap.shape
    hm = heatmap.reshape(B, N, -1).softmax(-1).reshape(B, N, H, W)
    grid_y = torch.linspace(0, 1, H, device=heatmap.device)
    grid_x = torch.linspace(0, 1, W, device=heatmap.device)
    coord_y = (hm * grid_y.view(1, 1, H, 1)).sum(dim=(2, 3))
    coord_x = (hm * grid_x.view(1, 1, 1, W)).sum(dim=(2, 3))
    return torch.stack([coord_x, coord_y], dim=-1)  # (B, N_LM, 2)

def soft_argmax_2d_gaussian(heatmap):
    """Sub-pixel landmark localization with Taylor expansion refinement.
    Fits a quadratic around the peak pixel to achieve sub-pixel accuracy.
    Args:
        heatmap: (B, N, H, W) tensor (raw heatmap, NOT softmax)
    Returns:
        coords: (B, N, 2) tensor with (x, y) in [0, 1]
    """
    B, N, H, W = heatmap.shape
    coords = torch.zeros(B, N, 2, device=heatmap.device)
    hm_np = heatmap.detach().cpu().float().numpy()
    import numpy as _np_sa
    for b in range(B):
        for n in range(N):
            hm = hm_np[b, n]
            # Softmax normalization
            hm = hm - hm.max()
            hm = _np_sa.exp(hm) / (_np_sa.exp(hm).sum() + 1e-8)
            max_idx = hm.argmax()
            py_int, px_int = int(max_idx // W), int(max_idx % W)
            px, py = float(px_int), float(py_int)
            # Taylor expansion: quadratic fit around peak for sub-pixel refinement
            if 0 < py_int < H - 1 and 0 < px_int < W - 1:
                dx  = float(hm[py_int, px_int + 1] - hm[py_int, px_int - 1])
                dy  = float(hm[py_int + 1, px_int] - hm[py_int - 1, px_int])
                dxx = float(hm[py_int, px_int + 1] + hm[py_int, px_int - 1]
                            - 2 * hm[py_int, px_int])
                dyy = float(hm[py_int + 1, px_int] + hm[py_int - 1, px_int]
                            - 2 * hm[py_int, px_int])
                if abs(dxx) > 1e-8:
                    px = px - dx / (2 * dxx)
                if abs(dyy) > 1e-8:
                    py = py - dy / (2 * dyy)
                # Clamp to valid range
                px = max(0.0, min(float(W - 1), px))
                py = max(0.0, min(float(H - 1), py))
            coords[b, n, 0] = px / max(W - 1, 1)
            coords[b, n, 1] = py / max(H - 1, 1)
    return coords


def mre_mm(pred,gt_lb,ps_list,hm=256,full=CEPH_SIZE):
    # Use soft-argmax for sub-pixel accuracy, scale [0,1] coords to pixel space
    pred_coords = soft_argmax_2d(pred)  # (B, N_LM, 2) in [0, 1]
    pc = pred_coords * full             # scale to image coords (pixels)
    errs=[]
    for b in range(pred.size(0)):
        ps=ps_list[b] if b<len(ps_list) else 0.1
        d=torch.sqrt(((pc[b]-gt_lb[b].to(pred.device))**2).sum(-1))
        errs.append((d*ps).mean().item())
    return float(np.mean(errs))


In [ ]:
# PHASE E PREP — Extract cephalometric landmarks for multi-modal malocclusion
# Requires Phase G1 to have run and saved: models_v4/ceph_landmark_v9_best.pth
# Outputs: processed/mal_ceph_lm_v9.csv
import pandas as pd
import torchvision.transforms.functional as TF

LM_CSV = OUT / "mal_ceph_lm_v9.csv"
N_LM   = 29
_PHASE_E_CKPT = MDL / "ceph_landmark_v9_best.pth"  # local — does NOT overwrite _PHASE_E_CKPT (v23)

def extract_ceph_landmarks_for_mal():
    """Run Phase G1 model on all cephalograms, save normalized landmark coords."""
    if LM_CSV.exists():
        print(f"Landmark CSV already exists: {LM_CSV} — skipping extraction.")
        return pd.read_csv(LM_CSV)

    if not _PHASE_E_CKPT.exists():
        print(f"WARNING: Ceph landmark model not found at {_PHASE_E_CKPT}.")
        print("Run Phase G1 first to generate cephalometric landmarks.")
        print("Falling back to panoramic-only malocclusion (Phase E will use EfficientNet-B2 only).")
        return None

    # Load the ceph model — use HRNetCeph (build_hrnet_ceph) to match checkpoint
    ceph_model = build_hrnet_ceph(n_lm=N_LM, device=DEVICE)
    try:
        ceph_model.load_state_dict(
            torch.load(_PHASE_E_CKPT, map_location=DEVICE, weights_only=True))
    except RuntimeError as _e:
        print(f"WARNING: checkpoint incompatible: {_e}")
        print("Falling back to panoramic-only malocclusion (Phase E skipped).")
        return None
    ceph_model.eval()

    # Load mappings: cephalogram patient_id -> panoramic image_id
    map_csv = BASE / "Aariz" / "cephalogram_machine_mappings.csv"
    if not map_csv.exists():
        print(f"WARNING: Mappings file not found: {map_csv}")
        return None

    mappings = pd.read_csv(map_csv)
    print(f"Loaded {len(mappings)} cephalogram<->panoramic mappings.")

    # Run inference on all Aariz cephalograms
    from PIL import Image as PILImage
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    import numpy as np

    _tfm = A.Compose([
        A.LongestMaxSize(CEPH_SIZE),
        A.PadIfNeeded(CEPH_SIZE, CEPH_SIZE, border_mode=0),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

    records = []
    for split in ["train", "valid", "test"]:
        ceph_img_dir = BASE / "Aariz" / split / "Cephalograms"
        if not ceph_img_dir.exists():
            ceph_img_dir = BASE / "Aariz" / split
        for img_path in sorted(ceph_img_dir.glob("*.png")) + sorted(ceph_img_dir.glob("*.jpg")):
            img = np.array(PILImage.open(img_path).convert("RGB"))
            h0, w0 = img.shape[:2]
            aug = _tfm(image=img)
            x = aug["image"].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                hm = ceph_model(x)  # (1, N_LM, 256, 256)
            # Decode landmark coords using soft-argmax (sub-pixel accuracy)
            lm = soft_argmax_2d(hm).squeeze(0)  # (N_LM, 2) in [0, 1]
            row = {"ceph_stem": img_path.stem}
            for j in range(N_LM):
                row[f"lm_{j}_x"] = lm[j, 0].item()
                row[f"lm_{j}_y"] = lm[j, 1].item()
            records.append(row)

    lm_df = pd.DataFrame(records)

    # Match to panoramic images via mappings CSV
    # Assume mappings has columns like: ceph_id (or ceph_filename) and pano_id (or image_id)
    # Try to merge on best available key
    merge_key_ceph = [c for c in mappings.columns if "ceph" in c.lower()][0] if any("ceph" in c.lower() for c in mappings.columns) else mappings.columns[0]
    merge_key_pano = [c for c in mappings.columns if "pano" in c.lower() or "image" in c.lower() or "xray" in c.lower()][0] if len(mappings.columns) > 1 else mappings.columns[1]

    lm_df["ceph_key"] = lm_df["ceph_stem"].str.lower()
    mappings["ceph_key"] = mappings[merge_key_ceph].astype(str).str.lower().str.replace(r"\.(png|jpg|jpeg)$", "", regex=True)
    merged = lm_df.merge(mappings[["ceph_key", merge_key_pano]], on="ceph_key", how="inner")
    merged = merged.rename(columns={merge_key_pano: "img_id"})

    lm_cols = [f"lm_{j}_{c}" for j in range(N_LM) for c in ["x", "y"]]
    out_df = merged[["img_id"] + lm_cols].drop_duplicates("img_id")
    out_df.to_csv(LM_CSV, index=False)
    print(f"Saved {len(out_df)} landmark records -> {LM_CSV}")
    return out_df

mal_lm_df = extract_ceph_landmarks_for_mal()
mal_has_landmarks = mal_lm_df is not None and len(mal_lm_df) > 0
print(f"Landmark-equipped samples: {len(mal_lm_df) if mal_has_landmarks else 0}")

## Phase G — Cephalometric Landmark Regression & CVM Staging

G1: ResNet-50 + U-Net decoder, coordinate-space L1 loss via soft-argmax.  
G2: ConvNeXt-V2-T for 3-class CVM staging.


In [ ]:

def letterbox(img_np, target=CEPH_SIZE):
    H,W=img_np.shape[:2]; sc=min(target/H,target/W)
    nH,nW=int(H*sc),int(W*sc)
    res=cv2.resize(img_np,(nW,nH))
    px,py=(target-nW)//2,(target-nH)//2
    if img_np.ndim==3: out=np.zeros((target,target,3),dtype=img_np.dtype)
    else: out=np.zeros((target,target),dtype=img_np.dtype)
    out[py:py+nH,px:px+nW]=res
    return out,sc,px,py

def scale_lm(lm,sc,px,py,clip=CEPH_SIZE):
    l=lm.copy().astype(float)
    l[:,0]=l[:,0]*sc+px; l[:,1]=l[:,1]*sc+py
    return np.clip(l,0,clip-1)

def make_heatmaps(lm_s, sigma=3, hm=256, full=CEPH_SIZE):
    N=len(lm_s); maps=np.zeros((N,hm,hm),dtype=np.float32); s=hm/full
    xs=np.arange(hm)[None,:]; ys=np.arange(hm)[:,None]
    for i,(x,y) in enumerate(lm_s):
        maps[i]=np.exp(-((xs-x*s)**2+(ys-y*s)**2)/(2*sigma**2))
    return maps

def load_aariz(split):
    sdir=AARIZ_DIR/split; idir = (sdir/"Cephalograms" if (sdir/"Cephalograms").exists()
        else sdir/"Images" if (sdir/"Images").exists()
        else sdir)
    jdir=sdir/"Annotations"/"Cephalometric Landmarks"/"Junior Orthodontists"
    sdir2=sdir/"Annotations"/"Cephalometric Landmarks"/"Senior Orthodontists"
    pmap={}
    if AARIZ_MAPS.exists():
        dm=pd.read_csv(AARIZ_MAPS)
        for _,row in dm.iterrows():
            fn=str(row.get("filename",row.get("image_name",""))).strip()
            pmap[Path(fn).stem]=float(row.get("pixel_size",row.get("PixelSpacing",0.1)))
    def _pts(d):
        pts=d.get("points",d.get("landmarks",[]))
        result = []
        for p in pts:
            if "x" in p:
                result.append((p["x"], p["y"]))
            elif "value" in p:
                result.append((p["value"]["x"], p["value"]["y"]))
            elif isinstance(p, (list, tuple)):
                result.append((p[0], p[1]))
        return result
    recs=[]
    for imf in sorted(list(idir.glob("*.png"))+list(idir.glob("*.jpg"))+list(idir.glob("*.bmp"))):
        stem=imf.stem
        jf=(jdir/f"{stem}.json") if jdir and jdir.exists() else None
        sf=(sdir2/f"{stem}.json") if sdir2 and sdir2.exists() else None
        lm=wts=None
        if jf and jf.exists() and sf and sf.exists():
            try:
                jd=json.loads(jf.read_text()); sd=json.loads(sf.read_text())
                jp,sp=_pts(jd),_pts(sd); n=min(len(jp),len(sp),N_LM)
                if n==0: continue
                lm=np.zeros((n,2)); wts=np.ones(n)
                for i in range(n):
                    lm[i]=[(jp[i][0]+sp[i][0])/2,(jp[i][1]+sp[i][1])/2]
                    if math.sqrt((jp[i][0]-sp[i][0])**2+(jp[i][1]-sp[i][1])**2)>50: wts[i]=0.5
            except: continue
        elif jf and jf.exists():
            try:
                jd=json.loads(jf.read_text()); pts=_pts(jd)
                lm=np.array(pts[:N_LM]); wts=np.ones(len(lm))
            except: continue
        if lm is None or len(lm)==0: continue
        # v10 fix: compute effective pixel spacing at CEPH_SIZE resolution
        # (original ps is at scan resolution; after letterbox sc, each pixel covers more mm)
        try:
            with PILImage.open(imf) as _im: _W0, _H0 = _im.size
            _sc = min(CEPH_SIZE / _H0, CEPH_SIZE / _W0)
        except: _sc = 0.23  # fallback for typical 1968×2225 Aariz images
        _ps_eff = pmap.get(stem, 0.188) / _sc  # mm/px at CEPH_SIZE; 0.188=Aariz standard (1968×2225≈18.7×21.1cm)
        recs.append(dict(path=imf,lm=lm,wts=wts,ps=_ps_eff,stem=stem,split=split))
    return recs

print("Loading Aariz datasets...")
atr=load_aariz("train"); avl=load_aariz("valid")
print(f"Train:{len(atr)} Valid:{len(avl)}")


In [ ]:

class CephDS(Dataset):
    def __init__(self,recs,aug=False):
        self.recs=recs; self.aug=aug
    def __len__(self): return len(self.recs)
    def __getitem__(self,idx):
        r=self.recs[idx]
        try: img=np.array(Image.open(r["path"]).convert("RGB"))
        except: img=np.zeros((CEPH_SIZE,CEPH_SIZE,3),dtype=np.uint8)
        lm=r["lm"].copy(); wts=r["wts"].copy()
        H0,W0=img.shape[:2]
        img_lb,sc,px,py=letterbox(img,CEPH_SIZE)
        lm_s=scale_lm(lm,sc,px,py)
        if self.aug:
            # Rotation ±5° (no horizontal flip — cephalograms are left-lateral)
            if np.random.rand()<0.5:
                ang=np.random.uniform(-5,5)
                M=cv2.getRotationMatrix2D((CEPH_SIZE//2,CEPH_SIZE//2),ang,1.0)
                img_lb=cv2.warpAffine(img_lb,M,(CEPH_SIZE,CEPH_SIZE),borderMode=cv2.BORDER_REFLECT)
                ones=np.ones((len(lm_s),1)); lm_s=(M@np.hstack([lm_s,ones]).T).T
                lm_s=np.clip(lm_s,0,CEPH_SIZE-1)
            if np.random.rand()<0.4:
                b=np.random.uniform(-0.2,0.2)
                img_lb=np.clip(img_lb.astype(float)+b*255,0,255).astype(np.uint8)
        img_t=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(
            torch.from_numpy(img_lb).permute(2,0,1).float()/255.0)
        hms_raw=torch.from_numpy(make_heatmaps(lm_s,sigma=3,hm=256,full=CEPH_SIZE))
        # Pad to N_LM so DataLoader can collate samples with fewer landmarks
        n_got=len(lm_s)
        hms=torch.zeros(N_LM,256,256); hms[:n_got]=hms_raw
        wts_t=torch.zeros(N_LM,dtype=torch.float32); wts_t[:n_got]=torch.from_numpy(wts[:n_got].astype(np.float32))
        lm_t=torch.zeros(N_LM,2,dtype=torch.float32); lm_t[:n_got]=torch.from_numpy(lm_s[:n_got].astype(np.float32))
        return img_t,hms,wts_t,lm_t

if atr:
    cph_tr=CephDS(atr,aug=True); cph_vl=CephDS(avl,aug=False)
    cph_tr_ld=DataLoader(cph_tr,16,shuffle=True, num_workers=NUM_WORKERS,pin_memory=True)
    cph_vl_ld=DataLoader(cph_vl,16,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
    print(f"Ceph train:{len(cph_tr)} val:{len(cph_vl)}")
else: print("No Aariz data — Phase F skipped")


In [ ]:


import math

def wing_loss(pred, gt, wts, w=10.0, eps=2.0):
    """Wing loss for heatmap-based landmark detection (Feng et al. 2018).
    Better than MSE for small displacement errors — more sensitive to precise localization.
    pred, gt: (B, N_LM, H, W) — heatmaps
    wts: (B, N_LM) or (N_LM,) — per-landmark consistency weights
    w: wing width threshold (pixels in normalized heatmap space)
    eps: curvature at zero
    """
    # Element-wise difference in heatmap space
    diff = (pred - gt).abs()  # (B, N_LM, H, W)
    C = w * (1.0 - math.log(1.0 + w / eps))
    loss_map = torch.where(
        diff < w,
        w * torch.log(1.0 + diff / eps),
        diff - C
    )  # (B, N_LM, H, W)
    # Average over spatial dims, weight by landmark consistency
    per_lm = loss_map.mean(dim=(2, 3))  # (B, N_LM)
    if wts.dim() == 1:
        wts = wts.unsqueeze(0)  # broadcast over batch
    weighted = (per_lm * wts).sum(dim=1) / (wts.sum(dim=1) + 1e-8)  # (B,)
    return weighted.mean()

def decode_hm(hm):
    B,N,H,W=hm.shape; flat=torch.softmax(hm.view(B,N,-1),-1)
    xs=torch.arange(W,dtype=torch.float32,device=hm.device)
    ys=torch.arange(H,dtype=torch.float32,device=hm.device)
    gx=xs.view(1,W).expand(H,W).reshape(-1); gy=ys.view(H,1).expand(H,W).reshape(-1)
    return torch.stack([(flat*gx).sum(-1),(flat*gy).sum(-1)],-1)

def soft_argmax_2d(heatmap):
    """Differentiable sub-pixel landmark localization.
    Args:
        heatmap: (B, N, H, W) tensor
    Returns:
        coords: (B, N, 2) tensor with (x, y) in [0, 1] range
    """
    B, N, H, W = heatmap.shape
    hm = heatmap.reshape(B, N, -1).softmax(-1).reshape(B, N, H, W)
    grid_y = torch.linspace(0, 1, H, device=heatmap.device)
    grid_x = torch.linspace(0, 1, W, device=heatmap.device)
    coord_y = (hm * grid_y.view(1, 1, H, 1)).sum(dim=(2, 3))
    coord_x = (hm * grid_x.view(1, 1, 1, W)).sum(dim=(2, 3))
    return torch.stack([coord_x, coord_y], dim=-1)  # (B, N_LM, 2)

def soft_argmax_2d_gaussian(heatmap):
    """Sub-pixel landmark localization with Taylor expansion refinement.
    Fits a quadratic around the peak pixel to achieve sub-pixel accuracy.
    Args:
        heatmap: (B, N, H, W) tensor (raw heatmap, NOT softmax)
    Returns:
        coords: (B, N, 2) tensor with (x, y) in [0, 1]
    """
    B, N, H, W = heatmap.shape
    coords = torch.zeros(B, N, 2, device=heatmap.device)
    hm_np = heatmap.detach().cpu().float().numpy()
    import numpy as _np_sa
    for b in range(B):
        for n in range(N):
            hm = hm_np[b, n]
            # Softmax normalization
            hm = hm - hm.max()
            hm = _np_sa.exp(hm) / (_np_sa.exp(hm).sum() + 1e-8)
            max_idx = hm.argmax()
            py_int, px_int = int(max_idx // W), int(max_idx % W)
            px, py = float(px_int), float(py_int)
            # Taylor expansion: quadratic fit around peak for sub-pixel refinement
            if 0 < py_int < H - 1 and 0 < px_int < W - 1:
                dx  = float(hm[py_int, px_int + 1] - hm[py_int, px_int - 1])
                dy  = float(hm[py_int + 1, px_int] - hm[py_int - 1, px_int])
                dxx = float(hm[py_int, px_int + 1] + hm[py_int, px_int - 1]
                            - 2 * hm[py_int, px_int])
                dyy = float(hm[py_int + 1, px_int] + hm[py_int - 1, px_int]
                            - 2 * hm[py_int, px_int])
                if abs(dxx) > 1e-8:
                    px = px - dx / (2 * dxx)
                if abs(dyy) > 1e-8:
                    py = py - dy / (2 * dyy)
                # Clamp to valid range
                px = max(0.0, min(float(W - 1), px))
                py = max(0.0, min(float(H - 1), py))
            coords[b, n, 0] = px / max(W - 1, 1)
            coords[b, n, 1] = py / max(H - 1, 1)
    return coords


def mre_mm(pred,gt_lb,ps_list,hm=256,full=CEPH_SIZE):
    # Use soft-argmax for sub-pixel accuracy, scale [0,1] coords to pixel space
    pred_coords = soft_argmax_2d(pred)  # (B, N_LM, 2) in [0, 1]
    pc = pred_coords * full             # scale to image coords (pixels)
    errs=[]
    for b in range(pred.size(0)):
        ps=ps_list[b] if b<len(ps_list) else 0.1
        d=torch.sqrt(((pc[b]-gt_lb[b].to(pred.device))**2).sum(-1))
        errs.append((d*ps).mean().item())
    return float(np.mean(errs))

if "cph_tr" in dir():
    print("CephNet architecture defined ✓")


In [ ]:

if "cph_tr" in dir() and len(cph_tr)>0:
    # HRNet-W32 for cephalometric landmark detection
    # Literature: Aariz baseline MRE=1.789mm with cascaded CNN; HRNet targets <2.0mm
    ceph_model = build_hrnet_ceph(n_lm=N_LM, device=DEVICE)
    _need_ceph=True
    if CEPH_CKPT.exists():
        try:
            ceph_model.load_state_dict(torch.load(CEPH_CKPT,map_location=DEVICE,weights_only=True))
            print(f"SKIP ceph — loaded {CEPH_CKPT.name}"); _need_ceph=False
        except RuntimeError: CEPH_CKPT.unlink()

    if _need_ceph:
        opt_cp=torch.optim.AdamW(ceph_model.parameters(),lr=3e-4,weight_decay=1e-4)
        sch_cp=torch.optim.lr_scheduler.OneCycleLR(opt_cp,max_lr=3e-4,
                   steps_per_epoch=len(cph_tr_ld),epochs=CEPH_EPOCHS,pct_start=0.1)
        best_mre=float("inf"); hist_ceph=[]

        for ep in range(CEPH_EPOCHS):
            ceph_model.train(); tl=0.0
            for imgs,hms,wts,lm in tqdm(cph_tr_ld,desc=f"[Ceph] Ep {ep+1}/{CEPH_EPOCHS}",leave=False):
                imgs,hms,wts=imgs.to(DEVICE),hms.to(DEVICE),wts.to(DEVICE)
                with autocast(dtype=torch.bfloat16): pred_hm=ceph_model(imgs)
                # Coordinate-space L1 loss: avoids heatmap background dominance
                pred_coords=soft_argmax_2d(pred_hm.float())*CEPH_SIZE  # (B,N_LM,2)
                gt_coords=lm.to(DEVICE).float()  # (B,N_LM,2)
                diff=(pred_coords-gt_coords).abs().sum(-1)  # L1 in px (B,N_LM)
                wts_d=wts.to(DEVICE).float()
                loss=((diff*wts_d).sum(1)/(wts_d.sum(1).clamp(min=1e-8))).mean()
                opt_cp.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(ceph_model.parameters(),3.0)
                opt_cp.step(); sch_cp.step(); tl+=loss.item()

            ceph_model.eval(); all_errs_mm=[]; all_lm_errs=[[] for _ in range(N_LM)]
            with torch.no_grad():
                for i,(imgs,hms,wts,lm) in enumerate(cph_vl_ld):
                    imgs_d=imgs.to(DEVICE)
                    # val TTA: 3-scale ensemble -- NO hflip (cephalogram is left-lateral only)
                    p0=ceph_model(imgs_d)
                    s9=F.interpolate(imgs_d,scale_factor=0.9,mode="bilinear",align_corners=False)
                    s9=F.interpolate(s9,(CEPH_SIZE,CEPH_SIZE),mode="bilinear",align_corners=False)
                    p1=ceph_model(s9)
                    s11=F.interpolate(imgs_d,scale_factor=1.1,mode="bilinear",align_corners=False)
                    s11=F.interpolate(s11,(CEPH_SIZE,CEPH_SIZE),mode="bilinear",align_corners=False)
                    p2=ceph_model(s11)
                    pred=(p0+p1+p2)/3
                    pred_coords=soft_argmax_2d(pred.float())*CEPH_SIZE
                    gt_px=lm.to(pred.device)
                    ps=[avl[i*len(imgs)+b]["ps"] for b in range(len(imgs)) if i*len(imgs)+b<len(avl)]
                    for b in range(len(imgs)):
                        ps_b=ps[b] if b<len(ps) else 0.188
                        d_px=torch.sqrt(((pred_coords[b]-gt_px[b])**2).sum(-1))
                        d_mm=(d_px*ps_b).cpu().tolist()
                        wts_b=wts[b].cpu().numpy()
                        for lm_i,(e,w) in enumerate(zip(d_mm,wts_b)):
                            if w>0:
                                all_errs_mm.append(e)
                                all_lm_errs[lm_i].append(e)
            vm=float(np.mean(all_errs_mm)) if all_errs_mm else 99.0
            arr=np.array(all_errs_mm) if all_errs_mm else np.array([99.0])
            sdr2 =float(np.mean(arr<2.0)*100)
            sdr25=float(np.mean(arr<2.5)*100)
            sdr4 =float(np.mean(arr<4.0)*100)
            hist_ceph.append(dict(ep=ep+1,loss=tl/len(cph_tr_ld),mre=vm,sdr2=sdr2,sdr4=sdr4))
            print(f"  Ceph {ep+1:02d}  loss={tl/len(cph_tr_ld):.4f}  "
                  f"MRE={vm:.2f}mm  SDR@2mm={sdr2:.1f}%  SDR@2.5mm={sdr25:.1f}%  SDR@4mm={sdr4:.1f}%")
            if vm<best_mre:
                best_mre=vm; torch.save(ceph_model.state_dict(),CEPH_CKPT)
                print(f"    ✓ Best MRE={best_mre:.2f}mm")
        print(f"Phase G1 done. MRE={best_mre:.2f}mm (target <2.0mm)")
        # Per-landmark breakdown (last val epoch)
        if all_lm_errs and any(all_lm_errs):
            lm_means=[float(np.mean(e)) if e else 99.0 for e in all_lm_errs]
            top5_hard=sorted(range(N_LM),key=lambda x:-lm_means[x])[:5]
            top5_easy=sorted(range(N_LM),key=lambda x:lm_means[x])[:5]
            print("  Hardest landmarks: "+", ".join(f"LM{k}={lm_means[k]:.2f}mm" for k in top5_hard))
            print("  Easiest landmarks: "+", ".join(f"LM{k}={lm_means[k]:.2f}mm" for k in top5_easy))
        if best_mre<2.5: print("  Clinically acceptable ✓")
else: print("Phase G1 skipped — no Aariz data")


In [ ]:
# Phase G1 — Standalone Evaluation (runs whether or not training was skipped)
if "ceph_model" in dir() and "cph_vl_ld" in dir() and "avl" in dir():
    ceph_model.eval(); all_errs_mm=[]; all_lm_errs=[[] for _ in range(N_LM)]
    with torch.no_grad():
        for i,(imgs,hms,wts,lm) in enumerate(tqdm(cph_vl_ld,desc="[G1 Eval]",leave=False)):
            imgs_d=imgs.to(DEVICE)
            p0=ceph_model(imgs_d)
            s9=F.interpolate(imgs_d,scale_factor=0.9,mode="bilinear",align_corners=False)
            s9=F.interpolate(s9,(CEPH_SIZE,CEPH_SIZE),mode="bilinear",align_corners=False); p1=ceph_model(s9)
            s11=F.interpolate(imgs_d,scale_factor=1.1,mode="bilinear",align_corners=False)
            s11=F.interpolate(s11,(CEPH_SIZE,CEPH_SIZE),mode="bilinear",align_corners=False); p2=ceph_model(s11)
            pred=(p0+p1+p2)/3
            pred_coords=soft_argmax_2d(pred.float())*CEPH_SIZE  # coordinate L1 model: peaks are not Gaussian
            gt_px=lm.to(pred.device)
            ps=[avl[i*len(imgs)+b]["ps"] for b in range(len(imgs)) if i*len(imgs)+b<len(avl)]
            for b in range(len(imgs)):
                ps_b=ps[b] if b<len(ps) else 0.188
                d_px=torch.sqrt(((pred_coords[b]-gt_px[b])**2).sum(-1))
                d_mm=(d_px*ps_b).cpu().tolist()
                wts_b=wts[b].cpu().numpy()  # (N_LM,) - 0 for zero-padded landmarks
                for lm_i,(e,w) in enumerate(zip(d_mm, wts_b)):
                    if w > 0:  # skip zero-padded landmarks
                        all_errs_mm.append(e)
                        all_lm_errs[lm_i].append(e)
    vm=float(np.mean(all_errs_mm)) if all_errs_mm else 99.0
    arr=np.array(all_errs_mm) if all_errs_mm else np.array([99.0])
    sdr2=float(np.mean(arr<2.0)*100); sdr25=float(np.mean(arr<2.5)*100); sdr4=float(np.mean(arr<4.0)*100)
    print("="*58)
    print("Phase G1 — Cephalometric Landmarks (%d val images)" % len(avl))
    print("="*58)
    print(f"  MRE         = {vm:.2f} mm   (literature baseline: 1.789 mm)")
    print(f"  SDR @ 2mm   = {sdr2:.1f}%")
    print(f"  SDR @ 2.5mm = {sdr25:.1f}%")
    print(f"  SDR @ 4mm   = {sdr4:.1f}%")
    if all_lm_errs:
        lm_means=[float(np.mean(e)) if e else 99.0 for e in all_lm_errs]
        top5=sorted(range(N_LM),key=lambda x:-lm_means[x])[:5]
        bot5=sorted(range(N_LM),key=lambda x:lm_means[x])[:5]
        print("  Hardest: "+", ".join(f"LM{k}={lm_means[k]:.2f}mm" for k in top5))
        print("  Easiest: "+", ".join(f"LM{k}={lm_means[k]:.2f}mm" for k in bot5))
    if vm < 1.789: print(f"  *** NEW SOTA on Aariz benchmark ({vm:.2f}mm < 1.789mm) ***")
    elif vm < 2.5: print("  Clinically acceptable")
    else: print(f"  WARNING: MRE={vm:.2f}mm above clinical threshold (2.5mm)")
else:
    print("G1 eval skipped - ceph_model / cph_vl_ld / avl not in scope")


In [ ]:

# Phase G2: CVM Staging (3-class: Early/Peak/Late)
CVM_CLS=["Early(S1+S2)","Peak(S3+S4)","Late(S5+S6)"]

def load_cvm(recs):
    out = []
    for r in recs:
        split  = r.get("split", "train")
        stem   = r["stem"]
        cvm_f  = AARIZ_DIR / split / "Annotations" / "CVM Stages" / f"{stem}.json"
        if not cvm_f.exists():
            continue
        d = json.loads(cvm_f.read_text())
        if "cvm_stage" not in d: continue  # skip records with no CVM label
        stage_val = d["cvm_stage"]
        if isinstance(stage_val, dict):
            s = int(stage_val.get("value", 1))
        else:
            s = int(stage_val)
        # Merge to 3 classes: S1-S2→0 (Early), S3-S4→1 (Peak), S5-S6→2 (Late)
        cvm_cls = 0 if s <= 2 else (1 if s <= 4 else 2)
        out.append(dict(path=r["path"], cvm=cvm_cls, stem=stem, split=split))
    return out

class CVMDS(Dataset):
    def __init__(self,recs,aug=False):
        self.recs=recs; self.aug=aug
        self.ttf=A.Compose([A.Resize(224,224),A.HorizontalFlip(p=0.5),
                             A.Affine(translate_percent={"x":(-0.05,0.05),"y":(-0.05,0.05)},
                                      scale=(0.9,1.1),rotate=(-5,5),p=0.4),
                             A.RandomBrightnessContrast(0.2,0.2,p=0.4),
                             A.GaussNoise(std_range=(0.012,0.025),p=0.2),
                             A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()])
        self.vtf=A.Compose([A.Resize(224,224),
                             A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),ToTensorV2()])
    def __len__(self): return len(self.recs)
    def __getitem__(self,idx):
        r=self.recs[idx]
        try: img=np.array(Image.open(r["path"]).convert("RGB"))
        except: img=np.zeros((224,224,3),dtype=np.uint8)
        img=apply_clahe(img)  # cephalogram contrast enhancement
        out=(self.ttf if self.aug else self.vtf)(image=img)
        return out["image"],torch.tensor(r["cvm"])

if atr:
    cvm_tr=load_cvm(atr); cvm_vl=load_cvm(avl)
    clbs=[r["cvm"] for r in cvm_tr]; cnts={l:clbs.count(l) for l in set(clbs)}
    _cls_w=torch.tensor([1.0/max(cnts.get(i,1),1) for i in range(3)],dtype=torch.float32)
    _cls_w=(_cls_w/_cls_w.mean()).to(DEVICE)  # normalize so avg weight=1
    print(f"CVM class weights: {_cls_w.cpu().numpy().round(3)}")
    cvm_tr_ld=DataLoader(CVMDS(cvm_tr,aug=True),32,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
    cvm_vl_ld=DataLoader(CVMDS(cvm_vl,aug=False),32,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)

    # ConvNeXt-V2-T replaces EfficientNet-B0
    # Literature: ConvNeXt V2 achieves 86.9% on 6-class CVM; 3-class target 85%+
    cvm_model = build_cvm_model(n_cls=3, device=DEVICE)
    # CrossEntropy with label smoothing + class weights; shuffle=True DataLoader (no sampler)
    fl_c=nn.CrossEntropyLoss(weight=_cls_w,label_smoothing=0.05)  # class weights, no sampler
    opt_cv=torch.optim.AdamW(cvm_model.parameters(),lr=3e-4,weight_decay=1e-4)
    sch_cv=torch.optim.lr_scheduler.CosineAnnealingLR(opt_cv,T_max=CVM_EPOCHS,eta_min=1e-6)
    best_ca=0.0; hist_cvm=[]

    _need_cvm=True
    if CVM_CKPT.exists():
        try:
            cvm_model.load_state_dict(torch.load(CVM_CKPT,map_location=DEVICE,weights_only=True))
            print(f"SKIP CVM — loaded {CVM_CKPT.name}"); _need_cvm=False
        except RuntimeError: CVM_CKPT.unlink()

    if _need_cvm:
        for ep in range(CVM_EPOCHS):
            cvm_model.train(); tl=0.0
            for imgs,lbls in tqdm(cvm_tr_ld,desc=f"[CVM] Ep {ep+1}/{CVM_EPOCHS}",leave=False):
                imgs,lbls=imgs.to(DEVICE),lbls.to(DEVICE)
                with autocast(dtype=torch.bfloat16): loss=fl_c(cvm_model(imgs),lbls)
                opt_cv.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(cvm_model.parameters(),3.0)
                opt_cv.step(); tl+=loss.item()
            sch_cv.step()
            cvm_model.eval(); cp,cl=[],[]
            with torch.no_grad():
                for imgs,lbls in cvm_vl_ld:
                    imgs_d=imgs.to(DEVICE)
                    # val TTA: avg original + hflip
                    lg=cvm_model(imgs_d)+cvm_model(torch.flip(imgs_d,[3]))
                    p=(lg*0.5).argmax(1).cpu()
                    cp.extend(p.numpy()); cl.extend(lbls.numpy())
            acc=np.mean(np.array(cp)==np.array(cl))
            hist_cvm.append(dict(ep=ep+1,loss=tl/len(cvm_tr_ld),acc=acc))
            print(f"  CVM {ep+1:02d}  loss={tl/len(cvm_tr_ld):.4f}  acc={acc:.4f}")
            if acc>best_ca: best_ca=acc; torch.save(cvm_model.state_dict(),CVM_CKPT); print(f"    ✓ Best acc={acc:.4f}")
        print(f"Phase G2 done. acc={best_ca:.4f}(≥0.78)")
        if cl:
            print(classification_report(cl,cp,target_names=CVM_CLS,zero_division=0))
            cm=confusion_matrix(cl,cp)
            o1=sum(cm[i][j] for i in range(3) for j in range(3) if abs(i-j)>1)
            terr=cm.sum()-np.diag(cm).sum()
            print(f"  Off-by->1 errors: {100*o1/max(terr,1):.1f}% (target <30%)")
else: print("Phase G2 skipped — no Aariz data")


In [ ]:
# Phase G2 — Standalone Evaluation (runs whether or not training was skipped)
if "cvm_model" in dir() and "cvm_vl_ld" in dir():
    cvm_model.eval(); cp,cl=[],[]
    with torch.no_grad():
        for imgs,lbls in tqdm(cvm_vl_ld,desc="[G2 Eval]",leave=False):
            imgs_d=imgs.to(DEVICE)
            lg=cvm_model(imgs_d)+cvm_model(torch.flip(imgs_d,[3]))
            p=(lg*0.5).argmax(1).cpu()
            cp.extend(p.numpy()); cl.extend(lbls.numpy())
    acc=float(np.mean(np.array(cp)==np.array(cl)))
    print("="*55)
    print("Phase G2 — CVM Bone Maturity Staging (3-class)")
    print("="*55)
    print(f"  Accuracy = {acc:.4f}  (target 85%+  /  literature 6-class: 86.9%)")
    print(classification_report(cl,cp,target_names=CVM_CLS,zero_division=0))
    cm=confusion_matrix(cl,cp)
    o1=sum(cm[i][j] for i in range(3) for j in range(3) if abs(i-j)>1)
    terr=cm.sum()-np.diag(cm).sum()
    print(f"  Off-by->1 errors: {100*o1/max(terr,1):.1f}%  (target <30%)")
    if acc >= 0.85: print("  Target met")
else:
    print("G2 eval skipped - cvm_model or cvm_vl_ld not in scope")


## Phase F — Malocclusion Screening

Binary panoramic classifier (ConvNeXtV2-T). Labels are YOLO-derived heuristics, not clinician annotations.

In [ ]:

# v23 label generation: YOLO-based per-quadrant analysis + bilateral asymmetry
# Regenerate if CSV missing OR has old format (image_id column)
_need_mal_regen = not MAL_CSV.exists()
if not _need_mal_regen:
    _chk = pd.read_csv(MAL_CSV, nrows=1)
    if "image_id" in _chk.columns:
        MAL_CSV.unlink(); _need_mal_regen = True
        print("Old mal_labels format (image_id) detected - regenerating with YOLO heuristic")

if _need_mal_regen:
    xl = sorted(DIS_XRAYS.glob("*.png")) + sorted(DIS_XRAYS.glob("*.jpg"))
    _has_yolo = "_yolo_model" in dir() and _yolo_model is not None
    print(f"Generating malocclusion labels for {len(xl)} images  (YOLO={'yes' if _has_yolo else 'fallback'})")
    _mal_rows = []
    for _xf in tqdm(xl, desc="Mal labels", leave=False):
        _mal = 1  # default: Moderate
        if _has_yolo:
            try:
                _cmap = _yolo_to_fdi_map(_yolo_model, _xf)
                _present = set(int(v) for v in np.unique(_cmap) if v > 0)
                _q1 = len([c for c in _present if 1  <= c <= 8 ])
                _q2 = len([c for c in _present if 9  <= c <= 16])
                _q3 = len([c for c in _present if 17 <= c <= 24])
                _q4 = len([c for c in _present if 25 <= c <= 32])
                _n  = _q1 + _q2 + _q3 + _q4
                _asym = abs(_q1 - _q2) + abs(_q3 - _q4)
                _miss_mol = len({6, 14, 22, 30} - _present)
                _score = (3 if _n < 22 else (1 if _n < 26 else 0)) +                          (2 if _asym >= 4 else (1 if _asym >= 2 else 0)) +                          (2 if _miss_mol >= 2 else (1 if _miss_mol >= 1 else 0))
                _mal = 0 if _score <= 1 else (2 if _score >= 4 else 1)
            except Exception: _mal = 1
        _mal_rows.append({"filename": _xf.name, "mal_class": _mal,
                          "mal_name": ["Minimal","Moderate","Severe"][_mal]})
    pd.DataFrame(_mal_rows).to_csv(MAL_CSV, index=False)
    print(f"Saved {len(_mal_rows)} YOLO-based malocclusion labels")
df_mal=pd.read_csv(MAL_CSV)
# Binary conversion: Moderate+Severe → Non-minimal (class 1)
df_mal["mal_class"] = (df_mal["mal_class"] > 0).astype(int)
MAL_CLS=["Minimal","Non-minimal"]
print(df_mal["mal_class"].value_counts().sort_index().rename({0:"Minimal",1:"Non-minimal"}))

class MalDatasetV7(torch.utils.data.Dataset):
    """Malocclusion dataset with optional cephalometric landmark features."""
    def __init__(self, df, lm_df=None, augment=False):
        self.df = df.reset_index(drop=True)
        self.lm_df = lm_df  # may be None (fallback to panoramic-only)
        self.augment = augment
        self.N_LM = 29
        lm_cols = [f"lm_{j}_{c}" for j in range(self.N_LM) for c in ["x","y"]]
        if lm_df is not None:
            self.lm_lookup = lm_df.set_index("img_id")[lm_cols].to_dict("index")
        else:
            self.lm_lookup = {}

        _tfm_base = [
            A.Resize(MAL_SIZE, MAL_SIZE),
            A.CLAHE(clip_limit=4.0, p=1.0),  # deterministic preprocessing: always applied (train + val)
            A.HorizontalFlip(p=0.5) if augment else A.NoOp(),
            A.Affine(translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
                     scale=(0.9, 1.1), rotate=(-10, 10), p=0.5) if augment else A.NoOp(),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.4) if augment else A.NoOp(),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ]
        self.tfm = A.Compose(_tfm_base)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        fname = row.get("file", row.get("filename", ""))
        path = RAW_XRAYS / fname
        for ext in [".png", ".jpg", ".PNG", ".JPG"]:
            alt = RAW_XRAYS / (Path(fname).stem + ext)
            if alt.exists(): path = alt; break
        try: img = np.array(Image.open(path).convert("RGB"))
        except: img = np.zeros((MAL_SIZE, MAL_SIZE, 3), dtype=np.uint8)
        aug = self.tfm(image=img)
        x   = aug["image"]
        y   = int(row["mal_class"])

        # Landmark features (normalized coords) or zeros if not available
        img_id = Path(fname).stem
        if img_id in self.lm_lookup:
            lm_vals = list(self.lm_lookup[img_id].values())
            lm = torch.tensor(lm_vals, dtype=torch.float32)
            if self.augment:
                lm = lm + torch.randn_like(lm) * 0.025  # coord noise augmentation
        else:
            lm = torch.zeros(self.N_LM * 2, dtype=torch.float32)

        return x, lm, y


class MalMultiModalNet(nn.Module):
    """Panoramic-only malocclusion classifier. Binary: 0=Minimal, 1=Non-minimal.
    Landmark branch removed: only 2/705 samples had landmark data (rest = zeros = noise).
    """
    def __init__(self, n_cls=2, n_lm=29):  # n_lm kept for API compat, unused
        super().__init__()
        import timm
        self.pano_enc = timm.create_model("convnextv2_tiny", pretrained=True, num_classes=n_cls)
    def forward(self, x_pano, x_lm=None):  # x_lm ignored
        return self.pano_enc(x_pano)

# DataLoaders
from sklearn.model_selection import train_test_split as _tts
_df_tr, _df_vl = _tts(df_mal, test_size=0.2, stratify=df_mal["mal_class"], random_state=42)
print(f"Mal train={len(_df_tr)} val={len(_df_vl)} | classes: {df_mal['mal_class'].value_counts().to_dict()}")
mal_tr_ds = MalDatasetV7(_df_tr, augment=True)
mal_vl_ds = MalDatasetV7(_df_vl, augment=False)
_cls_counts = np.bincount(_df_tr["mal_class"].values, minlength=2)
_weights = 1.0 / _cls_counts[_df_tr["mal_class"].values]
_sampler = torch.utils.data.WeightedRandomSampler(_weights, num_samples=len(_df_tr), replacement=True)
mal_tr_ld = torch.utils.data.DataLoader(mal_tr_ds, batch_size=16, sampler=_sampler, num_workers=0, drop_last=True)
mal_vl_ld = torch.utils.data.DataLoader(mal_vl_ds, batch_size=16, shuffle=False, num_workers=0, drop_last=False)

# Model
mal_model = MalMultiModalNet(n_cls=2).to(DEVICE)
print(f"mal_model params: {sum(p.numel() for p in mal_model.parameters()):,}")


In [ ]:
_need_mal=True
if MAL_CKPT.exists():
    try:
        mal_model.load_state_dict(torch.load(MAL_CKPT,map_location=DEVICE,weights_only=True))
        print(f"SKIP mal loaded {MAL_CKPT.name}"); _need_mal=False
    except RuntimeError: MAL_CKPT.unlink()

if _need_mal:
    ce_m=nn.CrossEntropyLoss(label_smoothing=0.05)
    # Differential LR: head at 3e-4, frozen backbone at 1e-5
    _head_ids={id(p) for p in mal_model.pano_enc.head.parameters()}
    _backbone_p=[p for p in mal_model.pano_enc.parameters() if id(p) not in _head_ids]
    opt_m=torch.optim.AdamW([
        {"params": _backbone_p, "lr": 1e-5},
        {"params": list(mal_model.pano_enc.head.parameters()), "lr": 3e-4},
    ], weight_decay=1e-4)
    sch_m=torch.optim.lr_scheduler.CosineAnnealingLR(opt_m,T_max=MAL_EPOCHS,eta_min=1e-6)
    best_mf=0.0; hist_mal=[]

    def mixup_mal(imgs,lbls,alpha=0.3):
        lam=np.random.beta(alpha,alpha); idx=torch.randperm(imgs.size(0),device=imgs.device)
        return lam*imgs+(1-lam)*imgs[idx], lbls,lbls[idx],lam

    for ep in range(MAL_EPOCHS):
        mal_model.train(); tl=0.0
        for imgs,lm_feats,lbls in tqdm(mal_tr_ld,desc=f"[Mal] Ep {ep+1}/{MAL_EPOCHS}",leave=False):
            imgs,lbls=imgs.to(DEVICE),lbls.to(DEVICE)
            if np.random.rand()<0.4:
                im,la,lb,lam=mixup_mal(imgs,lbls)
                with autocast(dtype=torch.bfloat16):
                    _lg=mal_model(im); loss=lam*ce_m(_lg,la)+(1-lam)*ce_m(_lg,lb)
            else:
                with autocast(dtype=torch.bfloat16): loss=ce_m(mal_model(imgs),lbls)
            opt_m.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(mal_model.parameters(),3.0)
            opt_m.step(); tl+=loss.item()
        sch_m.step()

        mal_model.eval(); ap,al=[],[]
        with torch.no_grad():
            for imgs,lm_feats,lbls in mal_vl_ld:
                imgs=imgs.to(DEVICE)
                # TTA: base + hflip + scale jitter
                lg=mal_model(imgs)+mal_model(torch.flip(imgs,[3]))
                sc=F.interpolate(imgs,scale_factor=0.9,mode="bilinear",align_corners=False)
                sc=F.interpolate(sc,(MAL_SIZE,MAL_SIZE),mode="bilinear",align_corners=False)
                lg+=mal_model(sc); p=(lg/3).argmax(1).cpu()
                ap.extend(p.numpy()); al.extend(lbls.numpy())
        f1=f1_score(al,ap,average="macro",zero_division=0)
        hist_mal.append(dict(ep=ep+1,loss=tl/len(mal_tr_ld),f1=f1))
        print(f"  Mal {ep+1:02d}  loss={tl/len(mal_tr_ld):.4f} F1={f1:.4f}")
        if f1>best_mf: best_mf=f1; torch.save(mal_model.state_dict(),MAL_CKPT); print(f"    Best F1={f1:.4f}")
    print(f"Phase F done. Best F1={best_mf:.4f}")
    print(classification_report(al,ap,target_names=MAL_CLS,zero_division=0))


## Results Summary


In [ ]:
# Results summary — all metrics from final checkpoints
results = [
    ("A",  "Tooth Segmentation",    "IoU (clean)",  "0.9105"),
    ("A",  "Tooth Segmentation",    "IoU (mixed)",  "0.8573"),
    ("B",  "FDI Instance Seg",      "mAP50(M)",    "0.956"),
    ("B",  "FDI Instance Seg",      "Pixel mIoU",  "0.6868"),
    ("D",  "Disease Classification","Macro F1",    "0.8927"),
    ("D",  "Disease Classification","ROC-AUC",     "0.9911"),
    ("F",  "Malocclusion",          "Binary F1",   "0.5826"),
    ("G1", "Ceph Landmarks",        "MRE (mm)",    "3.59"),
    ("G1", "Ceph Landmarks",        "SDR @ 2mm",   "27.4%"),
    ("G1", "Ceph Landmarks",        "SDR @ 2.5mm", "36.9%"),
    ("G1", "Ceph Landmarks",        "SDR @ 4mm",   "61.4%"),
    ("G2", "CVM Staging",           "Accuracy",    "58.8%"),
]
print(f"{'Phase':<6} {'Task':<22} {'Metric':<14} {'Result'}")
print("-"*55)
for ph, task, met, res in results:
    print(f"{ph:<6} {task:<22} {met:<14} {res}")
print()
print("Phase A best: seg_v12_best.pth (batch=16, 200ep, SWA)")
print("Phase D best: cls_v14_best.pth (EfficientNetV2-M + CBAM + SSL)")
